In [1]:
!rm -rf gdn2-test
!git clone https://github.com/Akseleu-J/gdn2-test.git
import sys
sys.path.append("gdn2-test")
!pip install -U "jax[tpu]"
!pip install -U jax jaxlib 
!pip install flax=="0.12.9"

Cloning into 'gdn2-test'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 130 (delta 55), reused 24 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 81.82 KiB | 1.12 MiB/s, done.
Resolving deltas: 100% (55/55), done.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [24]:
"""
gdn2_causal_leak_diagnostics_deep.py

РАСШИРЕНИЕ gdn2_causal_leak_diagnostics.py -- для многочасового TPU-прогона.
Не заменяет первый скрипт, а идёт следующим шагом: если H1/H2/H3/H4 дали
сигнал (или не дали однозначного), здесь его закрепляют статистически и
локализуют физически (в каком слое, на каком расстоянии от границы chunk).

Пять независимых, самодостаточных блоков. Можно запускать по одному.

  D1. PER-LAYER LEAK LOCALIZATION.
      Захватываем промежуточные активации после КАЖДОГО блока (через
      flax sow/capture_intermediates) для x_a и x_b (возмущённый вход) и
      находим МИНИМАЛЬНЫЙ индекс сло
      
      
      
      я, на котором max|Δact[<T]| впервые
      превышает tol. Если утечка появляется резко на layer L (а не растёт
      плавно от layer 0) -- это сильный сигнал, что слой L (точнее, его
      GDN2Mixer) вносит НЕ-round-off эффект, а что-то структурное.

  D2. BOUNDARY-DISTANCE PROFILING.
      Для каждой границы chunk (bt и bc) сканируем T по МЕЛКОЙ сетке
      (каждая позиция, не только -1/0/+1) в окне +-32 вокруг границы, и
      строим профиль diff(distance_to_boundary). Если утечка -- round-off
      от WY-solve на границе, ожидаем резкий пик РОВНО на границе и
      быстрый спад по мере удаления (типичная сигнатура численного шума
      блочного solve). Если утечка "размазана" по всему chunk или растёт
      к концу chunk -- это больше похоже на логическую ошибку в
      cross-chunk state (Kernel D) или в дизайне clip/центрирования.

  D3. MULTI-SEED STATISTICAL ROBUSTNESS.
      Повторяем H2 (bare trainable causal leak) и H4 (centered A/B) на
      N_SEEDS независимых seed для init/inputs, чтобы исключить, что
      наблюдаемое -- артефакт одного "невезучего" seed. Сообщает
      fail-rate по позициям и разброс worst_diff.

  D4. DTYPE / WY_EPS ABLATION GRID.
      Полный grid: dtype in {fp32, bf16} x wy_eps in {0, 1e-3, 1e-2} x
      use_centering in {True, False}, на голом кернеле (H2-style). Цель:
      понять, требует ли утечка ИМЕННО bf16 (типичная численная причина)
      или воспроизводится и в чистом fp32 (тогда это логика, не dtype).

  D5. FULL-RESOLUTION SCAN (statistical safety net).
      На голом кернеле и на модели: скан ВСЕХ позиций T (не только
      границ bt/bc), с грубым шагом (每 8 или 16), чтобы убедиться, что
      утечка географически привязана именно к границам, а не появляется
      где-то ещё, что мы просто не смотрели.

Kaggle TPU v5e-8, notebook-only: без argparse, top-level константы,
main(). Рассчитан на многочасовой прогон -- каждый блок логирует
прогресс и промежуточные результаты в JSON (можно прервать и посмотреть
что накопилось).
"""
from __future__ import annotations

import gc
import json
import os
import sys
import time

import jax
import jax.numpy as jnp
import flax.linen as nn

from Atomic_ops.configs import KernelConfig, KAGGLE_MEDIUM, KAGGLE_MEDIUM_NOCENTER
from Atomic_ops.gdn2_pipeline import gdn2_pallas_forward_trainable

_RESULTS = {}
OUT_DIR = "./gdn2_causal_leak_deep_results"
os.makedirs(OUT_DIR, exist_ok=True)


def _dump(tag):
    path = os.path.join(OUT_DIR, f"{tag}.json")
    with open(path, "w") as f:
        json.dump(_RESULTS, f, indent=2, default=str)
    print(f"    [checkpoint written: {path}]")


def _max_abs_diff(a, b):
    return float(jnp.max(jnp.abs(jnp.asarray(a, jnp.float32) - jnp.asarray(b, jnp.float32))))


def _make_inputs(key, bsz, n_chunks, bt, H, D, decay_scale, dtype=jnp.float32, h0_nonzero=True):
    L = n_chunks * bt
    k1, k2, k3, k4, k5 = jax.random.split(key, 5)
    shape = (bsz, L, H, D)

    q = jax.random.normal(k1, shape)
    k = jax.random.normal(k2, shape)
    q = q / (jnp.linalg.norm(q, axis=-1, keepdims=True) + 1e-6)
    k = k / (jnp.linalg.norm(k, axis=-1, keepdims=True) + 1e-6)
    v = jax.random.normal(k3, shape) * 0.5
    w = jax.random.uniform(k4, shape, minval=0.2, maxval=1.0)
    b = jax.random.uniform(jax.random.fold_in(k4, 1), shape, minval=0.2, maxval=1.0)
    g = -jnp.abs(jax.random.normal(k5, shape)) * decay_scale

    h0 = None
    if h0_nonzero:
        h0 = jax.random.normal(jax.random.fold_in(key, 99), (bsz, H, D, D)) * 0.1

    q, k, v, w, b = (t.astype(dtype) for t in (q, k, v, w, b))
    return q, k, v, w, b, g.astype(jnp.float32), h0


def _boundary_positions(config: KernelConfig, seq_len: int):
    """Все уникальные границы bt/bc (без -1/0/+1 разброса -- используется
    как центр окна в D2)."""
    pts = set()
    for base in range(0, seq_len, config.bt):
        if 0 < base < seq_len:
            pts.add(base)
    for base in range(0, seq_len, config.bc):
        if 0 < base < seq_len:
            pts.add(base)
    return sorted(pts)


# ==========================================================================
# Модель с sow-инструментацией для D1 (per-layer localization)
# ==========================================================================
def _build_instrumented_model(kernel_config, num_layers, d_model=768, n_heads=6):
    d_head = d_model // n_heads
    assert d_head == 128

    def _safe_normalize(t, eps=1e-6):
        return t * jax.lax.rsqrt(jnp.sum(t * t, axis=-1, keepdims=True) + eps ** 2)

    def _sanitize(t):
        return jnp.nan_to_num(jnp.clip(t, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)

    class Mixer(nn.Module):
        @nn.compact
        def __call__(self, x):
            b, l, d = x.shape
            q_lin = nn.Dense(d, use_bias=False, name="q_proj", dtype=jnp.bfloat16)(x)
            k_lin = nn.Dense(d, use_bias=False, name="k_proj", dtype=jnp.bfloat16)(x)
            v_lin = nn.Dense(d, use_bias=False, name="v_proj", dtype=jnp.bfloat16)(x)
            q = jax.nn.silu(q_lin).reshape(b, l, n_heads, d_head).astype(jnp.float32)
            k = jax.nn.silu(k_lin).reshape(b, l, n_heads, d_head).astype(jnp.float32)
            v = jax.nn.silu(v_lin).reshape(b, l, n_heads, d_head).astype(jnp.float32)
            v = jnp.clip(v, -50.0, 50.0)
            q = _safe_normalize(q)
            k = _safe_normalize(k)

            b_gate = jax.nn.sigmoid(nn.Dense(d, name="erase_gate", dtype=jnp.bfloat16)(x)) \
                .reshape(b, l, n_heads, d_head).astype(jnp.float32)
            w_gate = jax.nn.sigmoid(nn.Dense(d, name="write_gate", dtype=jnp.bfloat16)(x)) \
                .reshape(b, l, n_heads, d_head).astype(jnp.float32)

            a_param = self.param("decay_a", nn.initializers.constant(-4.0), (n_heads,)).astype(jnp.float32)
            f_proj = nn.Dense(d, name="decay_proj", dtype=jnp.bfloat16)(x).reshape(b, l, n_heads, d_head)
            a_safe = jnp.clip(a_param, -20.0, 20.0)
            g = -jnp.exp(a_safe)[None, None, :, None] * jax.nn.softplus(f_proj.astype(jnp.float32))
            g = jnp.nan_to_num(g, nan=0.0, posinf=0.0, neginf=-20.0)

            out_gate = jnp.clip(nn.Dense(d, use_bias=False, name="out_gate", dtype=jnp.bfloat16)(x), -1e2, 1e2)

            q, k, v, w_gate, b_gate, g = map(_sanitize, (q, k, v, w_gate, b_gate, g))
            out, _hf = gdn2_pallas_forward_trainable(
                q, k, v, w_gate, b_gate, g, scale=1.0, config=kernel_config
            )
            out = out.reshape(b, l, d)
            out = nn.RMSNorm(epsilon=1e-6, name="mixer_out_norm")(out).astype(x.dtype)
            return nn.Dense(d, use_bias=False, name="out_proj", dtype=jnp.bfloat16)(out * jax.nn.silu(out_gate))

    class MLP(nn.Module):
        @nn.compact
        def __call__(self, x):
            d = x.shape[-1]
            h = 4 * d
            gate = nn.Dense(h, use_bias=False, name="gate_proj", dtype=jnp.bfloat16)(x)
            up = nn.Dense(h, use_bias=False, name="up_proj", dtype=jnp.bfloat16)(x)
            act = jax.nn.silu(gate) * up
            return nn.Dense(d, use_bias=False, name="down_proj", dtype=jnp.bfloat16)(act)

    class Block(nn.Module):
        layer_idx: int

        @nn.compact
        def __call__(self, x):
            h = Mixer(name="mixer")(nn.RMSNorm(epsilon=1e-6, name="mixer_norm")(x))
            x = jnp.nan_to_num(jnp.clip(x + h, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)
            m = MLP(name="mlp")(nn.RMSNorm(epsilon=1e-6, name="mlp_norm")(x))
            x = jnp.nan_to_num(jnp.clip(x + m, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)
            # sow -- captured via capture_intermediates, keyed by module path
            self.sow("intermediates", f"block_out_{self.layer_idx}", x)
            return x

    class InstrumentedLM(nn.Module):
        @nn.compact
        def __call__(self, input_ids):
            embed = nn.Embed(num_embeddings=256, features=d_model, name="embed", dtype=jnp.bfloat16)
            x = embed(input_ids)
            for i in range(num_layers):
                x = Block(layer_idx=i, name=f"block_{i}")(x)
            x = nn.RMSNorm(epsilon=1e-6, name="final_norm")(x).astype(x.dtype)
            return embed.attend(x)

    return InstrumentedLM()


# ==========================================================================
# D1: per-layer leak localization
# ==========================================================================
def test_d1_per_layer_localization(cfg):
    print("\n" + "=" * 78)
    print("D1: per-layer leak localization via intermediate capture")
    print("=" * 78)

    config = KAGGLE_MEDIUM
    L = cfg["seq_len"]
    B = cfg["bsz"]
    num_layers = cfg["d1_num_layers"]
    T_probe = cfg["d1_probe_T"]  # one boundary position to trace through depth

    model = _build_instrumented_model(config, num_layers=num_layers)
    init_rng = jax.random.PRNGKey(12345)
    dummy = jnp.zeros((B, L), dtype=jnp.int32)
    params = model.init(init_rng, dummy)["params"]

    x_a = jax.random.randint(jax.random.PRNGKey(777), (B, L), 0, 256, dtype=jnp.int32)
    x_b = x_a.at[:, T_probe].set((x_a[:, T_probe] + 1) % 256)

    @jax.jit
    def forward_with_intermediates(p, ids):
        _logits, mutated = model.apply({"params": p}, ids, mutable=["intermediates"])
        return mutated["intermediates"]

    inter_a = forward_with_intermediates(params, x_a)
    inter_b = forward_with_intermediates(params, x_b)
    jax.block_until_ready((inter_a, inter_b))

    # --- DEBUG: посмотреть, какие ключи реально вернул flax ---
    print("  captured intermediate keys:")
    for k in sorted(inter_a.keys()):
        print(f"    {k!r}")
    def _find_sow_key(d, layer_idx):
        """flax capture_intermediates ключует по имени модуля: 'block_N'.
        На случай других версий/конвенций — проверим несколько вариантов."""
        for cand in (
            f"block_{layer_idx}",          # <-- это и есть реальный ключ
            f"block_out_{layer_idx}",
            f"block_{layer_idx}_block_out_{layer_idx}",
            f"block_{layer_idx}/block_out_{layer_idx}",
        ):
            if cand in d:
                return cand
        for k in d:
            if f"block_out_{layer_idx}" in k:
                return k
        return None
    def _unwrap_to_array(v):
        """Продраться сквозь dict/list/tuple пока не упрёмся в jax array."""
        # максимум 6 уровней вложенности — этого хватит с запасом
        for _ in range(6):
            if isinstance(v, dict):
                if "intermediates" in v:
                    v = v["intermediates"]
                else:
                    v = next(iter(v.values()))
                continue
            if isinstance(v, (list, tuple)):
                v = v[0]
                continue
            break
        return v

    print(f"  probing T={T_probe} through {num_layers} layers (tol={cfg['causal_tol']:.1e})")
    first_leak_layer = None
    per_layer = {}
    for i in range(num_layers):
        key = _find_sow_key(inter_a, i)
        if key is None:
            print(f"    layer {i:2d}: KEY NOT FOUND")
            continue
        act_a = _unwrap_to_array(inter_a[key])
        act_b = _unwrap_to_array(inter_b[key])
        # отладочный принт один раз — убедиться, что достали массив
        if i == 0:
            print(f"    [debug] layer 0 unwrapped shape: {act_a.shape}, dtype={act_a.dtype}")
        diff = _max_abs_diff(act_a[:, :T_probe], act_b[:, :T_probe]) if T_probe > 0 else 0.0
        per_layer[i] = diff
        flag = ""
        if diff >= cfg["causal_tol"] and first_leak_layer is None:
            first_leak_layer = i
            flag = "  <-- FIRST LAYER ABOVE TOL"
        print(f"    layer {i:2d}: max|Δact[<{T_probe}]| = {diff:.3e}{flag}")

    _RESULTS["d1.T_probe"] = T_probe
    _RESULTS["d1.num_layers"] = num_layers
    _RESULTS["d1.per_layer_diff"] = per_layer
    _RESULTS["d1.first_leak_layer"] = first_leak_layer

    if first_leak_layer is None:
        print("  => No layer crossed tol at this T -- leak (if any) is below causal_tol "
              "through this depth; try a larger num_layers or a different T_probe.")
    elif first_leak_layer == 0:
        print("  => Leak present already at layer 0 -- NOT a depth-accumulation effect; "
              "points directly at GDN2Mixer/kernel behavior at this boundary.")
    else:
        growth = per_layer[first_leak_layer] / max(per_layer.get(first_leak_layer - 1, 1e-12), 1e-12)
        print(f"  => Leak first crosses tol at layer {first_leak_layer} "
              f"(jump factor vs previous layer: {growth:.2f}x). Consistent with progressive "
              "amplification through RMSNorm/normalize stack if growth factor is roughly "
              "constant per layer.")

    del model, params, inter_a, inter_b
    gc.collect()
    jax.clear_caches()
    _dump("d1_per_layer_localization")


# ==========================================================================
# D2: boundary-distance profiling (bare kernel, fine-grained)
# ==========================================================================
def test_d2_boundary_distance_profile(cfg):
    print("\n" + "=" * 78)
    print("D2: fine-grained boundary-distance leak profile (bare trainable kernel)")
    print("=" * 78)

    for label, config in (("centered", KAGGLE_MEDIUM), ("nocenter", KAGGLE_MEDIUM_NOCENTER)):
        bsz, H, D = cfg["bsz"], cfg["H"], cfg["D"]
        L = cfg["seq_len"]
        n_chunks = L // config.bt
        key = jax.random.PRNGKey(4242)
        q, k, v, w, b, g, h0 = _make_inputs(key, bsz, n_chunks, config.bt, H, D, decay_scale=0.1)

        @jax.jit
        def fwd(q_, k_, v_, w_, b_, g_):
            o, _hf = gdn2_pallas_forward_trainable(q_, k_, v_, w_, b_, g_, scale=1.0, h0=h0, config=config)
            return o

        o_a = fwd(q, k, v, w, b, g)
        jax.block_until_ready(o_a)

        boundaries = _boundary_positions(config, L)
        window = cfg["d2_window"]
        profile = {}  # boundary -> {offset: diff}

        for boundary in boundaries:
            offsets = [off for off in range(-window, window + 1)
                       if 0 < boundary + off < L]
            diffs_by_offset = {}
            for off in offsets:
                T = boundary + off
                q_b = q.at[:, T].add(0.3)
                k_b = k.at[:, T].add(0.3)
                v_b = v.at[:, T].add(0.3)
                w_b = jnp.clip(w.at[:, T].add(0.1), 0.01, 1.0)
                b_b = jnp.clip(b.at[:, T].add(0.1), 0.01, 1.0)
                g_b = g.at[:, T].add(-0.05)
                o_b = fwd(q_b, k_b, v_b, w_b, b_b, g_b)
                jax.block_until_ready(o_b)
                diff = _max_abs_diff(o_a[:, :T], o_b[:, :T]) if T > 0 else 0.0
                diffs_by_offset[off] = diff
            profile[boundary] = diffs_by_offset

            peak_off = max(diffs_by_offset, key=diffs_by_offset.get)
            peak_val = diffs_by_offset[peak_off]
            edge_val_pos = diffs_by_offset.get(window, 0.0)
            edge_val_neg = diffs_by_offset.get(-window, 0.0)
            decays = peak_val > 0 and max(edge_val_pos, edge_val_neg) < peak_val * 0.3
            print(f"  {label} boundary={boundary:4d}: peak@offset={peak_off:+d} "
                  f"diff={peak_val:.3e}  edges(+-{window})=({edge_val_neg:.2e},{edge_val_pos:.2e})  "
                  f"{'localized/decaying' if decays else 'NOT clearly localized -- check manually'}")

        _RESULTS[f"d2.{label}.profile"] = profile
        _dump("d2_boundary_distance_profile")


# ==========================================================================
# D3: multi-seed statistical robustness
# ==========================================================================
def test_d3_multiseed_robustness(cfg):
    print("\n" + "=" * 78)
    print(f"D3: multi-seed robustness ({cfg['d3_n_seeds']} seeds), bare trainable kernel")
    print("=" * 78)

    for label, config in (("centered", KAGGLE_MEDIUM), ("nocenter", KAGGLE_MEDIUM_NOCENTER)):
        bsz, H, D = cfg["bsz"], cfg["H"], cfg["D"]
        L = cfg["seq_len"]
        n_chunks = L // config.bt

        @jax.jit
        def fwd(q_, k_, v_, w_, b_, g_, h0_):
            o, _hf = gdn2_pallas_forward_trainable(q_, k_, v_, w_, b_, g_, scale=1.0, h0=h0_, config=config)
            return o

        boundaries = _boundary_positions(config, L)
        per_seed_worst = []
        per_seed_failcount = []

        for seed in range(cfg["d3_n_seeds"]):
            key = jax.random.PRNGKey(9000 + seed)
            q, k, v, w, b, g, h0 = _make_inputs(key, bsz, n_chunks, config.bt, H, D, decay_scale=0.1)
            o_a = fwd(q, k, v, w, b, g, h0)
            jax.block_until_ready(o_a)

            worst = 0.0
            n_fail = 0
            for T in boundaries:
                q_b = q.at[:, T].add(0.3)
                k_b = k.at[:, T].add(0.3)
                v_b = v.at[:, T].add(0.3)
                w_b = jnp.clip(w.at[:, T].add(0.1), 0.01, 1.0)
                b_b = jnp.clip(b.at[:, T].add(0.1), 0.01, 1.0)
                g_b = g.at[:, T].add(-0.05)
                o_b = fwd(q_b, k_b, v_b, w_b, b_b, g_b, h0)
                jax.block_until_ready(o_b)
                diff = _max_abs_diff(o_a[:, :T], o_b[:, :T])
                worst = max(worst, diff)
                if diff >= cfg["causal_tol"]:
                    n_fail += 1

            per_seed_worst.append(worst)
            per_seed_failcount.append(n_fail)
            print(f"  {label} seed={seed}: worst_diff={worst:.3e}  n_failed={n_fail}/{len(boundaries)}")

        _RESULTS[f"d3.{label}.per_seed_worst"] = per_seed_worst
        _RESULTS[f"d3.{label}.per_seed_failcount"] = per_seed_failcount
        _RESULTS[f"d3.{label}.mean_worst"] = float(sum(per_seed_worst) / len(per_seed_worst))
        _RESULTS[f"d3.{label}.max_worst"] = float(max(per_seed_worst))
        _RESULTS[f"d3.{label}.min_worst"] = float(min(per_seed_worst))
        print(f"  {label} summary: mean={_RESULTS[f'd3.{label}.mean_worst']:.3e} "
              f"min={_RESULTS[f'd3.{label}.min_worst']:.3e} "
              f"max={_RESULTS[f'd3.{label}.max_worst']:.3e}")
        _dump("d3_multiseed_robustness")


# ==========================================================================
# D4: dtype / wy_eps / centering ablation grid (bare kernel)
# ==========================================================================
def test_d4_ablation_grid(cfg):
    print("\n" + "=" * 78)
    print("D4: dtype x wy_eps x use_centering ablation grid (bare trainable kernel)")
    print("=" * 78)

    bsz, H, D = cfg["bsz"], cfg["H"], cfg["D"]
    L = cfg["seq_len"]
    bt, bc, mb = 256, 128, 16

    grid = []
    for dtype_label, dtype in (("fp32", jnp.float32), ("bf16", jnp.bfloat16)):
        for wy_eps in (0.0, 1e-3, 1e-2):
            for use_centering in (True, False):
                grid.append((dtype_label, dtype, wy_eps, use_centering))

    results = {}
    for dtype_label, dtype, wy_eps, use_centering in grid:
        config = KernelConfig(bt=bt, bc=bc, mb=mb, wy_eps=wy_eps, use_centering=use_centering)
        n_chunks = L // bt
        key = jax.random.PRNGKey(5555)
        q, k, v, w, b, g, h0 = _make_inputs(key, bsz, n_chunks, bt, H, D, decay_scale=0.1, dtype=dtype)

        @jax.jit
        def fwd(q_, k_, v_, w_, b_, g_):
            o, _hf = gdn2_pallas_forward_trainable(q_, k_, v_, w_, b_, g_, scale=1.0, h0=h0, config=config)
            return o

        o_a = fwd(q, k, v, w, b, g)
        jax.block_until_ready(o_a)

        boundaries = _boundary_positions(config, L)
        worst = 0.0
        n_fail = 0
        for T in boundaries:
            q_b = q.at[:, T].add(jnp.asarray(0.3, dtype))
            k_b = k.at[:, T].add(jnp.asarray(0.3, dtype))
            v_b = v.at[:, T].add(jnp.asarray(0.3, dtype))
            w_b = jnp.clip(w.at[:, T].add(jnp.asarray(0.1, dtype)), 0.01, 1.0)
            b_b = jnp.clip(b.at[:, T].add(jnp.asarray(0.1, dtype)), 0.01, 1.0)
            g_b = g.at[:, T].add(-0.05)
            o_b = fwd(q_b, k_b, v_b, w_b, b_b, g_b)
            jax.block_until_ready(o_b)
            diff = _max_abs_diff(o_a[:, :T].astype(jnp.float32), o_b[:, :T].astype(jnp.float32))
            worst = max(worst, diff)
            if diff >= cfg["causal_tol"]:
                n_fail += 1

        tag = f"{dtype_label}/wy_eps={wy_eps}/centering={use_centering}"
        results[tag] = dict(worst_diff=worst, n_failed=n_fail, n_total=len(boundaries))
        print(f"  {tag:45s} worst_diff={worst:.3e}  n_failed={n_fail}/{len(boundaries)}")

    _RESULTS["d4.grid"] = results
    _dump("d4_ablation_grid")

    print("\n  Interpretation:")
    fp32_leaks = any(r["n_failed"] > 0 for k, r in results.items() if k.startswith("fp32"))
    bf16_leaks = any(r["n_failed"] > 0 for k, r in results.items() if k.startswith("bf16"))
    if fp32_leaks:
        print("  => Leak reproduces even in PURE fp32 -- this is a LOGIC issue, not a bf16")
        print("     precision artifact. Focus on kernel math (centering/clip/boundary terms),")
        print("     not on dtype/precision tuning.")
    elif bf16_leaks and not fp32_leaks:
        print("  => Leak ONLY appears with bf16 inputs -- consistent with a precision/rounding")
        print("     sensitivity rather than a structural causality bug. Consider tightening")
        print("     Gate-2 tolerance for bf16 configs, or adding fp32 accumulation at the")
        print("     specific boundary op identified in D2.")
    else:
        print("  => No leak reproduced in this grid at all -- if D1-D3 showed a leak, it may")
        print("     require the full model stack (RMSNorm chain) to manifest; not a bare-kernel")
        print("     dtype/wy_eps/centering effect in isolation.")


# ==========================================================================
# D5: full-resolution scan (statistical safety net -- coarse stride over
#     ALL positions, not just known boundaries)
# ==========================================================================
def test_d5_full_resolution_scan(cfg):
    print("\n" + "=" * 78)
    print(f"D5: full-resolution scan, stride={cfg['d5_stride']} (bare trainable kernel)")
    print("=" * 78)

    for label, config in (("centered", KAGGLE_MEDIUM), ("nocenter", KAGGLE_MEDIUM_NOCENTER)):
        bsz, H, D = cfg["bsz"], cfg["H"], cfg["D"]
        L = cfg["seq_len"]
        n_chunks = L // config.bt
        key = jax.random.PRNGKey(6161)
        q, k, v, w, b, g, h0 = _make_inputs(key, bsz, n_chunks, config.bt, H, D, decay_scale=0.1)

        @jax.jit
        def fwd(q_, k_, v_, w_, b_, g_):
            o, _hf = gdn2_pallas_forward_trainable(q_, k_, v_, w_, b_, g_, scale=1.0, h0=h0, config=config)
            return o

        o_a = fwd(q, k, v, w, b, g)
        jax.block_until_ready(o_a)

        stride = cfg["d5_stride"]
        all_positions = list(range(stride, L, stride))
        known_boundaries = set(_boundary_positions(config, L))

        unexpected_leaks = []
        for T in all_positions:
            q_b = q.at[:, T].add(0.3)
            k_b = k.at[:, T].add(0.3)
            v_b = v.at[:, T].add(0.3)
            w_b = jnp.clip(w.at[:, T].add(0.1), 0.01, 1.0)
            b_b = jnp.clip(b.at[:, T].add(0.1), 0.01, 1.0)
            g_b = g.at[:, T].add(-0.05)
            o_b = fwd(q_b, k_b, v_b, w_b, b_b, g_b)
            jax.block_until_ready(o_b)
            diff = _max_abs_diff(o_a[:, :T], o_b[:, :T])
            near_known_boundary = any(abs(T - kb) <= 2 for kb in known_boundaries)
            if diff >= cfg["causal_tol"] and not near_known_boundary:
                unexpected_leaks.append((T, diff))
                print(f"    [UNEXPECTED] {label} T={T}: diff={diff:.3e} "
                      f"(NOT near a known bt/bc boundary!)")

        _RESULTS[f"d5.{label}.n_scanned"] = len(all_positions)
        _RESULTS[f"d5.{label}.n_unexpected_leaks"] = len(unexpected_leaks)
        _RESULTS[f"d5.{label}.unexpected_leaks"] = unexpected_leaks
        print(f"  {label}: scanned {len(all_positions)} positions (stride={stride}), "
              f"{len(unexpected_leaks)} unexpected leaks away from known boundaries.")
        _dump("d5_full_resolution_scan")


# ==========================================================================
# Entrypoint
# ==========================================================================
RUN_CONFIG = dict(
    bsz=2,
    H=6,
    D=128,
    seq_len=1024,
    causal_tol=1e-5,
    d1_num_layers=13,
    d1_probe_T=127,          # a known leaking boundary from Gate 2 logs
    d2_window=32,
    d3_n_seeds=8,
    d5_stride=8,
)


def main(cfg=RUN_CONFIG):
    print(f"JAX version: {jax.__version__}")
    print(f"Devices: {jax.devices()}")
    t0 = time.time()

    test_d1_per_layer_localization(cfg)
    test_d2_boundary_distance_profile(cfg)
    test_d3_multiseed_robustness(cfg)
    test_d4_ablation_grid(cfg)
    test_d5_full_resolution_scan(cfg)

    elapsed = time.time() - t0
    _dump("FINAL_all_results")
    print("\n" + "=" * 78)
    print(f"DEEP DIAGNOSTICS COMPLETE. Elapsed: {elapsed/3600:.2f}h ({elapsed:.0f}s)")
    print(f"All results in {OUT_DIR}/")
    print("=" * 78)


if __name__ == "__main__":
    main()

JAX version: 0.11.1
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]

D1: per-layer leak localization via intermediate capture
  captured intermediate keys:
    'block_0'
    'block_1'
    'block_10'
    'block_11'
    'block_12'
    'block_2'
    'block_3'
    'block_4'
    'block_5'
    'block_6'
    'block_7'
    'block_8'
    'block_9'
  probing T=127 through 13 layers (tol=1.0e-05)
    [debug] layer 0 unwrapped shape: (2, 1024, 768), dtype=bfloat16
    layer  0: max|Δact[<127]| = 1.562e-02  <-- FIRST 

In [25]:
"""
gdn2_causal_leak_diagnostics.py

Долгий, но РЕШАЮЩИЙ TPU-диагностический прогон для локализации Gate-2
causal-leak регрессии, обнаруженной в gdn2_150m_honest_train_v3.py.

Контекст (см. centering-c8-test.ipynb):
  - test_centering_boundary (Kernel A/B4 изолированно, interpret=True,
    jax.vjp cross-check) -- PASS, rel_err~1e-6, даже под насыщением
    clip[-20,20] при decay_scale=0.5.
  - Gate 2 (causal leak, полная 13-слойная модель, gdn2_pallas_forward_trainable)
    -- FAIL на границах bc/bt (T=1,127,129,255,257,...), diff до 0.24.
  - Ручная бисекция в ноутбуке: gdn2_pallas_forward (НЕ _trainable) на
    голых тензорах с теми же decay/shape -- diff=0.0 на всех T.
  - Изолированная проверка одной A (WY-solve) до Kernel D -- diff~1e-7
    ИМЕННО на границах bc (127,255,383) -- float32 roundoff, не логика.

Вопрос, на который отвечает этот скрипт: где именно возникает утечка --
  (H1) в custom_vjp обёртке gdn2_pallas_forward_trainable
       (несовпадение gdn2_pallas_forward_with_residuals bit-exact с
       gdn2_pallas_forward, как заявлено в комментариях gdn2_fwd.py)?
  (H2) в самом голом кернеле gdn2_pallas_forward_trainable (без модели)?
  (H3) усиление ~1e-7 WY-solve-граничного roundoff через глубину модели
       (RMSNorm / L2-normalize чувствительны к малым возмущениям)?
  (H4) специфично для use_centering=True (per-pair gn_i/gn_j на границах
       bc), т.е. исчезает/уменьшается на *_NOCENTER пресетах?

Каждая секция -- независимый, самодостаточный гейт с PASS/FAIL и точным
местом отказа. Секции НЕ шарят состояние -- можно бежать по одной.

Kaggle TPU v5e-8, notebook-only: без argparse, все параметры -- top-level
константы, вход через main().
"""
from __future__ import annotations

import gc
import json
import os
import sys
import time

import jax
import jax.numpy as jnp

from Atomic_ops.configs import (
    KernelConfig, KAGGLE_MEDIUM, KAGGLE_MEDIUM_NOCENTER,
)
from Atomic_ops.gdn2_fwd import (
    gdn2_pallas_forward,
    gdn2_pallas_forward_with_residuals,
)
from Atomic_ops.gdn2_pipeline import gdn2_pallas_forward_trainable

_FAILURES = []
_RESULTS = {}

OUT_DIR = "./gdn2_causal_leak_diag_results"
os.makedirs(OUT_DIR, exist_ok=True)


def _check(name, cond_ok, extra=""):
    status = "PASS" if cond_ok else "FAIL"
    print(f"[{status}] {name}  {extra}")
    if not cond_ok:
        _FAILURES.append(name)
    return cond_ok


def _rel_err(a, b):
    a = jnp.asarray(a, dtype=jnp.float32)
    b = jnp.asarray(b, dtype=jnp.float32)
    num = jnp.max(jnp.abs(a - b))
    den = jnp.maximum(jnp.max(jnp.abs(b)), 1e-8)
    return float(num / den)


def _max_abs_diff(a, b):
    return float(jnp.max(jnp.abs(jnp.asarray(a, jnp.float32) - jnp.asarray(b, jnp.float32))))


def _make_inputs(key, bsz, n_chunks, bt, H, D, decay_scale, h0_nonzero=True):
    L = n_chunks * bt
    k1, k2, k3, k4, k5 = jax.random.split(key, 5)
    shape = (bsz, L, H, D)

    q = jax.random.normal(k1, shape)
    k = jax.random.normal(k2, shape)
    q = q / (jnp.linalg.norm(q, axis=-1, keepdims=True) + 1e-6)
    k = k / (jnp.linalg.norm(k, axis=-1, keepdims=True) + 1e-6)
    v = jax.random.normal(k3, shape) * 0.5
    w = jax.random.uniform(k4, shape, minval=0.2, maxval=1.0)
    b = jax.random.uniform(jax.random.fold_in(k4, 1), shape, minval=0.2, maxval=1.0)
    g = -jnp.abs(jax.random.normal(k5, shape)) * decay_scale

    h0 = None
    if h0_nonzero:
        h0 = jax.random.normal(jax.random.fold_in(key, 99), (bsz, H, D, D)) * 0.1

    return q, k, v, w, b, g.astype(jnp.float32), h0


def _causal_test_positions(config: KernelConfig, seq_len: int):
    positions = set()
    for base in range(0, seq_len, config.bt):
        for off in (-1, 0, 1):
            p = base + off
            if 0 <= p < seq_len:
                positions.add(p)
    for base in range(0, seq_len, config.bc):
        for off in (-1, 0, 1):
            p = base + off
            if 0 <= p < seq_len:
                positions.add(p)
    positions.update([1, seq_len // 4, seq_len // 2, 3 * seq_len // 4, seq_len - 1])
    return sorted(positions)


# ==========================================================================
# H1: bit-exact primal check -- gdn2_pallas_forward vs
#     gdn2_pallas_forward_with_residuals, на РЕАЛЬНОМ shape модели
#     (bsz=2, H=6, n_chunks=4 => seq_len=1024, тот же decay=0.1 что в
#     run_correctness_gate/run_causal_leak_gate).
# ==========================================================================
def test_h1_primal_bit_exact(cfg):
    print("\n" + "=" * 78)
    print("H1: primal bit-exactness -- forward vs forward_with_residuals")
    print("    (model-shape: bsz=2, H=6, D=128, seq_len=1024, decay=0.1)")
    print("=" * 78)

    for label, config in (("centered", KAGGLE_MEDIUM), ("nocenter", KAGGLE_MEDIUM_NOCENTER)):
        bsz, H, D = 2, 6, 128
        n_chunks = 1024 // config.bt
        key = jax.random.PRNGKey(777)
        q, k, v, w, b, g, h0 = _make_inputs(key, bsz, n_chunks, config.bt, H, D, decay_scale=0.1)

        o1, h1 = gdn2_pallas_forward(q, k, v, w, b, g, scale=1.0, h0=h0, config=config)
        o2, h2, _res = gdn2_pallas_forward_with_residuals(q, k, v, w, b, g, scale=1.0, h0=h0, config=config)
        jax.block_until_ready((o1, h1, o2, h2))

        o_diff = _max_abs_diff(o1, o2)
        h_diff = _max_abs_diff(h1, h2)
        o_bitexact = o_diff == 0.0
        h_bitexact = h_diff == 0.0

        _RESULTS[f"h1.{label}.o_max_abs_diff"] = o_diff
        _RESULTS[f"h1.{label}.h_max_abs_diff"] = h_diff

        _check(f"H1.{label}.o_bit_exact", o_bitexact, f"max_abs_diff={o_diff:.3e}")
        _check(f"H1.{label}.h_final_bit_exact", h_bitexact, f"max_abs_diff={h_diff:.3e}")

        if not o_bitexact or not h_bitexact:
            print(f"    !!! {label}: primal NOT bit-exact as claimed in gdn2_fwd.py comments.")
            print(f"        This alone can explain Gate-2 divergence if H1 fails and H2 (below) passes.")


# ==========================================================================
# H2: causal leak gate on the BARE trainable kernel path (no model, no
#     RMSNorm/L2-normalize) -- isolates custom_vjp / residuals plumbing
#     from any amplification through model layers.
# ==========================================================================
def test_h2_bare_trainable_causal_leak(cfg):
    print("\n" + "=" * 78)
    print("H2: causal leak on bare gdn2_pallas_forward_trainable (no model layers)")
    print("=" * 78)

    for label, config in (("centered", KAGGLE_MEDIUM), ("nocenter", KAGGLE_MEDIUM_NOCENTER)):
        bsz, H, D = cfg["bsz"], cfg["H"], cfg["D"]
        L = cfg["seq_len"]
        n_chunks = L // config.bt
        key = jax.random.PRNGKey(777)
        q, k, v, w, b, g, h0 = _make_inputs(key, bsz, n_chunks, config.bt, H, D, decay_scale=0.1)

        @jax.jit
        def fwd(q_, k_, v_, w_, b_, g_):
            o, _hf = gdn2_pallas_forward_trainable(q_, k_, v_, w_, b_, g_, scale=1.0, h0=h0, config=config)
            return o

        o_a = fwd(q, k, v, w, b, g)
        jax.block_until_ready(o_a)

        positions = _causal_test_positions(config, L)
        n_fail = 0
        worst = (None, 0.0)
        for T in positions:
            q_b = q.at[:, T].add(0.3)
            k_b = k.at[:, T].add(0.3)
            v_b = v.at[:, T].add(0.3)
            w_b = jnp.clip(w.at[:, T].add(0.1), 0.01, 1.0)
            b_b = jnp.clip(b.at[:, T].add(0.1), 0.01, 1.0)
            g_b = g.at[:, T].add(-0.05)
            o_b = fwd(q_b, k_b, v_b, w_b, b_b, g_b)
            jax.block_until_ready(o_b)

            diff_before = _max_abs_diff(o_a[:, :T], o_b[:, :T]) if T > 0 else 0.0
            causal_ok = diff_before < cfg["causal_tol"]
            if not causal_ok:
                n_fail += 1
                if diff_before > worst[1]:
                    worst = (T, diff_before)
                print(f"    [FAIL] {label} T={T:4d}: max|Δo[<{T}]|={diff_before:.3e} "
                      f"(tol={cfg['causal_tol']:.1e})")

        _RESULTS[f"h2.{label}.n_positions"] = len(positions)
        _RESULTS[f"h2.{label}.n_failed"] = n_fail
        _RESULTS[f"h2.{label}.worst_T"] = worst[0]
        _RESULTS[f"h2.{label}.worst_diff"] = worst[1]

        _check(f"H2.{label}.bare_trainable_causal", n_fail == 0,
               f"{n_fail}/{len(positions)} positions failed"
               + (f", worst T={worst[0]} diff={worst[1]:.3e}" if n_fail else ""))


# ==========================================================================
# H3: depth sweep on the FULL model path -- does leak amplitude grow with
#     num_layers? This distinguishes "kernel bug" (leak present and
#     roughly constant even at num_layers=1) from "roundoff amplified by
#     RMSNorm/L2-normalize stack" (leak grows with depth, near-zero at
#     num_layers=1).
#
#     NOTE: uses a trimmed-down version of ByteGDN2LM/GDN2Mixer inlined
#     here (not imported from the training script) so this file has no
#     external dependency beyond Atomic_ops itself.
# ==========================================================================
def _build_mini_model(kernel_config, num_layers, d_model=768, n_heads=6):
    import flax.linen as nn

    d_head = d_model // n_heads
    assert d_head == 128

    def _safe_normalize(t, eps=1e-6):
        return t * jax.lax.rsqrt(jnp.sum(t * t, axis=-1, keepdims=True) + eps ** 2)

    def _sanitize(t):
        return jnp.nan_to_num(jnp.clip(t, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)

    class Mixer(nn.Module):
        @nn.compact
        def __call__(self, x):
            b, l, d = x.shape
            q_lin = nn.Dense(d, use_bias=False, name="q_proj", dtype=jnp.bfloat16)(x)
            k_lin = nn.Dense(d, use_bias=False, name="k_proj", dtype=jnp.bfloat16)(x)
            v_lin = nn.Dense(d, use_bias=False, name="v_proj", dtype=jnp.bfloat16)(x)
            q = jax.nn.silu(q_lin).reshape(b, l, n_heads, d_head).astype(jnp.float32)
            k = jax.nn.silu(k_lin).reshape(b, l, n_heads, d_head).astype(jnp.float32)
            v = jax.nn.silu(v_lin).reshape(b, l, n_heads, d_head).astype(jnp.float32)
            v = jnp.clip(v, -50.0, 50.0)
            q = _safe_normalize(q)
            k = _safe_normalize(k)

            b_gate = jax.nn.sigmoid(nn.Dense(d, name="erase_gate", dtype=jnp.bfloat16)(x)) \
                .reshape(b, l, n_heads, d_head).astype(jnp.float32)
            w_gate = jax.nn.sigmoid(nn.Dense(d, name="write_gate", dtype=jnp.bfloat16)(x)) \
                .reshape(b, l, n_heads, d_head).astype(jnp.float32)

            a_param = self.param("decay_a", nn.initializers.constant(-4.0), (n_heads,)).astype(jnp.float32)
            f_proj = nn.Dense(d, name="decay_proj", dtype=jnp.bfloat16)(x).reshape(b, l, n_heads, d_head)
            a_safe = jnp.clip(a_param, -20.0, 20.0)
            g = -jnp.exp(a_safe)[None, None, :, None] * jax.nn.softplus(f_proj.astype(jnp.float32))
            g = jnp.nan_to_num(g, nan=0.0, posinf=0.0, neginf=-20.0)

            out_gate = jnp.clip(nn.Dense(d, use_bias=False, name="out_gate", dtype=jnp.bfloat16)(x), -1e2, 1e2)

            q, k, v, w_gate, b_gate, g = map(_sanitize, (q, k, v, w_gate, b_gate, g))
            out, _hf = gdn2_pallas_forward_trainable(
                q, k, v, w_gate, b_gate, g, scale=1.0, config=kernel_config
            )
            out = out.reshape(b, l, d)
            out = nn.RMSNorm(epsilon=1e-6, name="mixer_out_norm")(out).astype(x.dtype)
            return nn.Dense(d, use_bias=False, name="out_proj", dtype=jnp.bfloat16)(out * jax.nn.silu(out_gate))

    class MLP(nn.Module):
        @nn.compact
        def __call__(self, x):
            d = x.shape[-1]
            h = 4 * d
            gate = nn.Dense(h, use_bias=False, name="gate_proj", dtype=jnp.bfloat16)(x)
            up = nn.Dense(h, use_bias=False, name="up_proj", dtype=jnp.bfloat16)(x)
            act = jax.nn.silu(gate) * up
            return nn.Dense(d, use_bias=False, name="down_proj", dtype=jnp.bfloat16)(act)

    class Block(nn.Module):
        @nn.compact
        def __call__(self, x):
            h = Mixer(name="mixer")(nn.RMSNorm(epsilon=1e-6, name="mixer_norm")(x))
            x = jnp.nan_to_num(jnp.clip(x + h, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)
            m = MLP(name="mlp")(nn.RMSNorm(epsilon=1e-6, name="mlp_norm")(x))
            x = jnp.nan_to_num(jnp.clip(x + m, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)
            return x

    class MiniLM(nn.Module):
        @nn.compact
        def __call__(self, input_ids):
            embed = nn.Embed(num_embeddings=256, features=d_model, name="embed", dtype=jnp.bfloat16)
            x = embed(input_ids)
            for i in range(num_layers):
                x = Block(name=f"block_{i}")(x)
            x = nn.RMSNorm(epsilon=1e-6, name="final_norm")(x).astype(x.dtype)
            return embed.attend(x)

    return MiniLM()


def test_h3_depth_sweep(cfg):
    print("\n" + "=" * 78)
    print("H3: causal-leak amplitude vs num_layers (isolates roundoff amplification)")
    print("=" * 78)

    config = KAGGLE_MEDIUM
    L = cfg["seq_len"]
    B = cfg["depth_sweep_bsz"]
    positions = _causal_test_positions(config, L)
    depth_results = {}

    for num_layers in cfg["depth_sweep_layers"]:
        print(f"\n  -- num_layers={num_layers} --")
        model = _build_mini_model(config, num_layers=num_layers)
        init_rng = jax.random.PRNGKey(12345)
        dummy = jnp.zeros((B, L), dtype=jnp.int32)
        params = model.init(init_rng, dummy)["params"]

        x_a = jax.random.randint(jax.random.PRNGKey(777), (B, L), 0, 256, dtype=jnp.int32)

        @jax.jit
        def forward(p, ids):
            return model.apply({"params": p}, ids).astype(jnp.float32)

        logits_a = forward(params, x_a)
        jax.block_until_ready(logits_a)

        worst = (None, 0.0)
        n_fail = 0
        for T in positions:
            x_b = x_a.at[:, T].set((x_a[:, T] + 1) % 256)
            logits_b = forward(params, x_b)
            jax.block_until_ready(logits_b)
            diff_before = _max_abs_diff(logits_a[:, :T, :], logits_b[:, :T, :]) if T > 0 else 0.0
            if diff_before >= cfg["causal_tol"]:
                n_fail += 1
            if diff_before > worst[1]:
                worst = (T, diff_before)

        depth_results[num_layers] = dict(n_failed=n_fail, n_total=len(positions),
                                          worst_T=worst[0], worst_diff=worst[1])
        print(f"    n_failed={n_fail}/{len(positions)}  worst: T={worst[0]} diff={worst[1]:.3e}")

        del model, params
        gc.collect()
        jax.clear_caches()

    _RESULTS["h3.depth_sweep"] = depth_results

    diffs = [depth_results[nl]["worst_diff"] for nl in cfg["depth_sweep_layers"]]
    monotonic_growth = all(diffs[i] <= diffs[i + 1] * 3 + 1e-9 for i in range(len(diffs) - 1))
    # not a strict pass/fail gate -- diagnostic signal only, printed for interpretation
    print(f"\n  worst_diff by depth: {dict(zip(cfg['depth_sweep_layers'], diffs))}")
    if diffs[0] < cfg["causal_tol"] and diffs[-1] >= cfg["causal_tol"]:
        print("  => SIGNAL: leak appears only at higher depth -- consistent with roundoff "
              "amplification through RMSNorm/L2-normalize stack (H3 supported).")
    elif diffs[0] >= cfg["causal_tol"]:
        print("  => SIGNAL: leak already present at num_layers=1 -- NOT a depth-amplification "
              "story; points back to the bare kernel/model-integration path (H1/H2/H4).")


# ==========================================================================
# H4: centered vs nocenter comparison at fixed depth (uses the same mini
#     model as H3, one fixed num_layers value from cfg).
# ==========================================================================
def test_h4_centering_ab(cfg):
    print("\n" + "=" * 78)
    print("H4: use_centering=True vs NOCENTER -- causal leak A/B at fixed depth")
    print("=" * 78)

    L = cfg["seq_len"]
    B = cfg["depth_sweep_bsz"]
    num_layers = cfg["h4_num_layers"]

    for label, config in (("centered", KAGGLE_MEDIUM), ("nocenter", KAGGLE_MEDIUM_NOCENTER)):
        positions = _causal_test_positions(config, L)
        model = _build_mini_model(config, num_layers=num_layers)
        init_rng = jax.random.PRNGKey(12345)
        dummy = jnp.zeros((B, L), dtype=jnp.int32)
        params = model.init(init_rng, dummy)["params"]

        x_a = jax.random.randint(jax.random.PRNGKey(777), (B, L), 0, 256, dtype=jnp.int32)

        @jax.jit
        def forward(p, ids):
            return model.apply({"params": p}, ids).astype(jnp.float32)

        logits_a = forward(params, x_a)
        jax.block_until_ready(logits_a)

        worst = (None, 0.0)
        n_fail = 0
        for T in positions:
            x_b = x_a.at[:, T].set((x_a[:, T] + 1) % 256)
            logits_b = forward(params, x_b)
            jax.block_until_ready(logits_b)
            diff_before = _max_abs_diff(logits_a[:, :T, :], logits_b[:, :T, :]) if T > 0 else 0.0
            if diff_before >= cfg["causal_tol"]:
                n_fail += 1
            if diff_before > worst[1]:
                worst = (T, diff_before)

        _RESULTS[f"h4.{label}.num_layers"] = num_layers
        _RESULTS[f"h4.{label}.n_failed"] = n_fail
        _RESULTS[f"h4.{label}.worst_diff"] = worst[1]
        print(f"  {label}: n_failed={n_fail}/{len(positions)}  worst: T={worst[0]} diff={worst[1]:.3e}")

        del model, params
        gc.collect()
        jax.clear_caches()

    c_diff = _RESULTS["h4.centered.worst_diff"]
    n_diff = _RESULTS["h4.nocenter.worst_diff"]
    print(f"\n  centered worst_diff={c_diff:.3e}  nocenter worst_diff={n_diff:.3e}")
    if n_diff < cfg["causal_tol"] <= c_diff:
        print("  => SIGNAL: leak is specific to use_centering=True (H4 supported).")
    elif c_diff < cfg["causal_tol"] and n_diff < cfg["causal_tol"]:
        print("  => SIGNAL: neither leaks at this depth -- leak may require more layers/bsz "
              "to manifest, or was already explained by H1/H2/H3.")
    else:
        print("  => SIGNAL: both leak comparably -- NOT specific to centering.")


# ==========================================================================
# Entrypoint
# ==========================================================================
RUN_CONFIG = dict(
    bsz=2,
    H=6,
    D=128,
    seq_len=1024,
    causal_tol=1e-5,
    depth_sweep_layers=[1, 2, 4, 8, 13],
    depth_sweep_bsz=2,
    h4_num_layers=8,
)


def main(cfg=RUN_CONFIG):
    print(f"JAX version: {jax.__version__}")
    print(f"Devices: {jax.devices()}")
    t0 = time.time()

    test_h1_primal_bit_exact(cfg)
    test_h2_bare_trainable_causal_leak(cfg)
    test_h3_depth_sweep(cfg)
    test_h4_centering_ab(cfg)

    elapsed = time.time() - t0
    out_path = os.path.join(OUT_DIR, "causal_leak_diagnostics.json")
    with open(out_path, "w") as f:
        json.dump(dict(results=_RESULTS, failures=_FAILURES, elapsed_s=elapsed), f, indent=2, default=str)

    print("\n" + "=" * 78)
    print(f"Elapsed: {elapsed:.1f}s. Full results written to {out_path}")
    print("=" * 78)
    if _FAILURES:
        print(f"FAILED gates ({len(_FAILURES)}): {_FAILURES}")
        print("\nInterpretation guide:")
        print("  - H1 fails, H2 passes -> bug in gdn2_pallas_forward_with_residuals / custom_vjp fwd,")
        print("    NOT in the bare kernel's math. Fix residual capture in gdn2_fwd.py.")
        print("  - H1 passes, H2 fails -> bug is in the trainable path itself (custom_vjp bwd/fwd")
        print("    registration in gdn2_pipeline.py), reproducible without any model.")
        print("  - H2 passes, H3 shows growth with depth -> not a kernel bug; roundoff amplified")
        print("    by RMSNorm/L2-normalize stack. Consider tightening Gate-2 tolerance per-depth")
        print("    or stabilizing _safe_normalize/RMSNorm epsilon.")
        print("  - H4 shows centered-only leak -> revisit per-pair gn_i/gn_j boundary terms in")
        print("    _kernel_a_body/_kernel_b4_body (gdn2_fwd.py / gdn2_bwd.py).")
        
    else:
        print("ALL bit-exactness / bare-kernel causal gates PASSED.")
        print("Check H3/H4 diagnostic signals above (not pass/fail gates) to interpret Gate-2.")
        

if __name__ == "__main__":
    main()

JAX version: 0.11.1
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]

H1: primal bit-exactness -- forward vs forward_with_residuals
    (model-shape: bsz=2, H=6, D=128, seq_len=1024, decay=0.1)
[PASS] H1.centered.o_bit_exact  max_abs_diff=0.000e+00
[PASS] H1.centered.h_final_bit_exact  max_abs_diff=0.000e+00
[PASS] H1.nocenter.o_bit_exact  max_abs_diff=0.000e+00
[PASS] H1.nocenter.h_final_bit_exact  max_abs_diff=0.000e+00

H2: causal leak on bare gdn2_pallas_forward_trainable (no model layers)
[PASS] H2.ce

In [45]:
"""
gdn2_bs5_nocenter_and_amplification.py

Два теста, закрывающих главный открытый вопрос после BS1-BS4/Test D/E:

  BS5a. TEST D/E ПОВТОРЁННЫЙ НА NOCENTER.
        Test D (centered) показал: kernel out[:T] diff ~1.7e-6 (шум),
        logits[:T] diff ~1.6e-2 (усиление). Test H4 показал nocenter
        даёт РОВНО 0.0 на уровне logits через 8 bf16-слоёв. Но H4 не
        смотрел на kernel out напрямую -- только на logits. Этот блок
        повторяет ТОЧНО Test D (jit-capture kernel input/output,
        [:T] vs [T]) но с KAGGLE_MEDIUM_NOCENTER, чтобы подтвердить
        симметрично: kernel out[:T] diff == 0.0 (не просто "меньше
        tol", а ровно ноль -- как и logits в H4).

        Если out[:T]==0.0 И logits[:T]==0.0 для nocenter -- это прямо
        подтверждает: источник неточности живёт ТОЛЬКО в
        use_centering=True кернеле (даже там, где он microscopic,
        ~1e-6), downstream-стек сам по себе причинен.

  BS5b. BISECTION УСИЛЕНИЯ БЕЗ КЕРНЕЛА.
        Берём kernel out ИЗ NOCENTER forward (заведомо причинный,
        значит [:T] у него бит-идентичен между x_a/x_b) и добавляем
        ИСКУССТВЕННЫЙ шум ~1e-6 в позицию T (той же магнитуды, что
        centered-кернель реально вносит по Test D). Прогоняем ТОЛЬКО
        downstream часть Mixer/Block (RMSNorm -> out_proj -> residual
        -> MLP -> final_norm -> embed.attend) на "чистом" и
        "зашумлённом на T" out, смотрим diff[:T] на выходе.

        Если ~1e-6 шум на входе даёt ~1e-2 diff на logits -- это
        доказывает, что усиление 1e-6 -> 1e-2 есть СВОЙСТВО
        downstream-архитектуры (RMSNorm/bf16/residual/Dense stack),
        а не что-то специфичное для centered-математики. Тогда чинить
        (если вообще нужно) имеет смысл maybe amplification, а не
        centering formula -- или просто документировать как
        приемлемый уровень (1e-6 << шум Adam/градиентов).

Не заменяет предыдущие скрипты -- независимый, самодостаточный блок.
Kaggle TPU v5e-8, notebook-only: без argparse, top-level константы, main().
"""
from __future__ import annotations

import gc
import json
import os
import time

import jax
import jax.numpy as jnp
import flax.linen as nn

from Atomic_ops.configs import KAGGLE_MEDIUM, KAGGLE_MEDIUM_NOCENTER
from Atomic_ops.gdn2_pipeline import gdn2_pallas_forward_trainable

_RESULTS = {}
OUT_DIR = "./gdn2_bs5_results"
os.makedirs(OUT_DIR, exist_ok=True)


def _dump(tag):
    path = os.path.join(OUT_DIR, f"{tag}.json")
    with open(path, "w") as f:
        json.dump(_RESULTS, f, indent=2, default=str)
    print(f"    [checkpoint written: {path}]")


def _max_abs_diff(a, b):
    return float(jnp.max(jnp.abs(jnp.asarray(a, jnp.float32) - jnp.asarray(b, jnp.float32))))


# ==========================================================================
# Модель -- идентична H3/BS-серии _build_mini_model, но с явным
# kernel_config аргументом (чтобы переключать centered/nocenter) и с
# разложением Mixer на "upstream" (до кернела) и "downstream" (после
# кернела) частями -- downstream вызывается отдельно в BS5b.
# ==========================================================================
def _build_model(kernel_config, num_layers, d_model=768, n_heads=6):
    d_head = d_model // n_heads
    assert d_head == 128

    def _safe_normalize(t, eps=1e-6):
        return t * jax.lax.rsqrt(jnp.sum(t * t, axis=-1, keepdims=True) + eps ** 2)

    def _sanitize(t):
        return jnp.nan_to_num(jnp.clip(t, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)

    class Mixer(nn.Module):
        layer_idx: int

        @nn.compact
        def __call__(self, x):
            b, l, d = x.shape
            q_lin = nn.Dense(d, use_bias=False, name="q_proj", dtype=jnp.bfloat16)(x)
            k_lin = nn.Dense(d, use_bias=False, name="k_proj", dtype=jnp.bfloat16)(x)
            v_lin = nn.Dense(d, use_bias=False, name="v_proj", dtype=jnp.bfloat16)(x)
            q = jax.nn.silu(q_lin).reshape(b, l, n_heads, d_head).astype(jnp.float32)
            k = jax.nn.silu(k_lin).reshape(b, l, n_heads, d_head).astype(jnp.float32)
            v = jax.nn.silu(v_lin).reshape(b, l, n_heads, d_head).astype(jnp.float32)
            v = jnp.clip(v, -50.0, 50.0)
            q = _safe_normalize(q)
            k = _safe_normalize(k)

            b_gate = jax.nn.sigmoid(nn.Dense(d, name="erase_gate", dtype=jnp.bfloat16)(x)) \
                .reshape(b, l, n_heads, d_head).astype(jnp.float32)
            w_gate = jax.nn.sigmoid(nn.Dense(d, name="write_gate", dtype=jnp.bfloat16)(x)) \
                .reshape(b, l, n_heads, d_head).astype(jnp.float32)

            a_param = self.param("decay_a", nn.initializers.constant(-4.0), (n_heads,)).astype(jnp.float32)
            f_proj = nn.Dense(d, name="decay_proj", dtype=jnp.bfloat16)(x).reshape(b, l, n_heads, d_head)
            a_safe = jnp.clip(a_param, -20.0, 20.0)
            g = -jnp.exp(a_safe)[None, None, :, None] * jax.nn.softplus(f_proj.astype(jnp.float32))
            g = jnp.nan_to_num(g, nan=0.0, posinf=0.0, neginf=-20.0)

            out_gate = jnp.clip(nn.Dense(d, use_bias=False, name="out_gate", dtype=jnp.bfloat16)(x), -1e2, 1e2)

            q, k, v, w_gate, b_gate, g = map(_sanitize, (q, k, v, w_gate, b_gate, g))

            if self.layer_idx == 0:
                self.sow("intermediates", "kin_q_0", q)
                self.sow("intermediates", "kin_k_0", k)
                self.sow("intermediates", "kin_v_0", v)
                self.sow("intermediates", "kin_w_0", w_gate)
                self.sow("intermediates", "kin_b_0", b_gate)
                self.sow("intermediates", "kin_g_0", g)
                self.sow("intermediates", "out_gate_0", out_gate)

            out, _hf = gdn2_pallas_forward_trainable(
                q, k, v, w_gate, b_gate, g, scale=1.0, config=kernel_config
            )

            if self.layer_idx == 0:
                self.sow("intermediates", "kout_raw_0", out)

            out = out.reshape(b, l, d)
            out = nn.RMSNorm(epsilon=1e-6, name="mixer_out_norm")(out).astype(x.dtype)
            return nn.Dense(d, use_bias=False, name="out_proj", dtype=jnp.bfloat16)(out * jax.nn.silu(out_gate))

    class MLP(nn.Module):
        @nn.compact
        def __call__(self, x):
            d = x.shape[-1]
            h = 4 * d
            gate = nn.Dense(h, use_bias=False, name="gate_proj", dtype=jnp.bfloat16)(x)
            up = nn.Dense(h, use_bias=False, name="up_proj", dtype=jnp.bfloat16)(x)
            act = jax.nn.silu(gate) * up
            return nn.Dense(d, use_bias=False, name="down_proj", dtype=jnp.bfloat16)(act)

    class Block(nn.Module):
        layer_idx: int

        @nn.compact
        def __call__(self, x):
            h = Mixer(layer_idx=self.layer_idx, name="mixer")(nn.RMSNorm(epsilon=1e-6, name="mixer_norm")(x))
            x = jnp.nan_to_num(jnp.clip(x + h, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)
            m = MLP(name="mlp")(nn.RMSNorm(epsilon=1e-6, name="mlp_norm")(x))
            x = jnp.nan_to_num(jnp.clip(x + m, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)
            return x

    class LM(nn.Module):
        @nn.compact
        def __call__(self, input_ids):
            embed = nn.Embed(num_embeddings=256, features=d_model, name="embed", dtype=jnp.bfloat16)
            x = embed(input_ids)
            for i in range(num_layers):
                x = Block(layer_idx=i, name=f"block_{i}")(x)
            x = nn.RMSNorm(epsilon=1e-6, name="final_norm")(x).astype(x.dtype)
            return embed.attend(x)

    return LM()


def _find_and_unwrap(intermediates, target_key):
    def _walk(node):
        if isinstance(node, dict):
            if target_key in node:
                return node[target_key]
            for v in node.values():
                found = _walk(v)
                if found is not None:
                    return found
        return None

    found = _walk(intermediates)
    if found is None:
        raise KeyError(f"sow key {target_key!r} не найден (top-level keys: {list(intermediates.keys())})")
    while isinstance(found, (list, tuple)):
        found = found[0]
    return found


# ==========================================================================
# BS5a: Test D повторённый один-в-один, но параметризованный по конфигу
# (centered / nocenter), с явным сравнением обоих в одной таблице.
# ==========================================================================
def test_bs5a_test_d_both_configs(cfg):
    print("\n" + "=" * 78)
    print("BS5a: Test D (kernel out[:T]/logits[:T] diff) для centered И nocenter")
    print("=" * 78)

    L, B = cfg["seq_len"], cfg["bsz"]
    num_layers = cfg["num_layers"]
    T = cfg["T"]

    results = {}
    for label, config in (("centered", KAGGLE_MEDIUM), ("nocenter", KAGGLE_MEDIUM_NOCENTER)):
        model = _build_model(config, num_layers=num_layers)
        params = model.init(jax.random.PRNGKey(12345), jnp.zeros((B, L), jnp.int32))["params"]

        x_a = jax.random.randint(jax.random.PRNGKey(777), (B, L), 0, 256, dtype=jnp.int32)
        x_b = x_a.at[:, T].set((x_a[:, T] + 1) % 256)

        @jax.jit
        def fwd_capture(p, ids):
            logits, mutated = model.apply({"params": p}, ids, mutable=["intermediates"])
            return logits, mutated["intermediates"]

        logits_a, inter_a = fwd_capture(params, x_a)
        jax.block_until_ready((logits_a, inter_a))
        logits_b, inter_b = fwd_capture(params, x_b)
        jax.block_until_ready((logits_b, inter_b))

        names = ("q", "k", "v", "w", "b", "g")
        input_diff = max(
            _max_abs_diff(_find_and_unwrap(inter_a, f"kin_{n}_0")[:, :T],
                          _find_and_unwrap(inter_b, f"kin_{n}_0")[:, :T])
            for n in names
        ) if T > 0 else 0.0

        kout_a = _find_and_unwrap(inter_a, "kout_raw_0")
        kout_b = _find_and_unwrap(inter_b, "kout_raw_0")
        kout_diff_lessT = _max_abs_diff(kout_a[:, :T], kout_b[:, :T]) if T > 0 else 0.0
        kout_diff_atT = _max_abs_diff(kout_a[:, T], kout_b[:, T])

        logits_diff_lessT = _max_abs_diff(
            jnp.asarray(logits_a, jnp.float32)[:, :T],
            jnp.asarray(logits_b, jnp.float32)[:, :T],
        ) if T > 0 else 0.0

        results[label] = dict(
            input_diff_lessT=input_diff,
            kernel_out_diff_lessT=kout_diff_lessT,
            kernel_out_diff_atT=kout_diff_atT,
            logits_diff_lessT=logits_diff_lessT,
        )
        print(f"\n  --- {label} (T={T}) ---")
        print(f"    input[:T] diff       = {input_diff:.3e}   (должно быть 0.0)")
        print(f"    kernel out[:T] diff  = {kout_diff_lessT:.3e}")
        print(f"    kernel out[T]  diff  = {kout_diff_atT:.3e}   (positive control, должно быть >0)")
        print(f"    logits[:T] diff      = {logits_diff_lessT:.3e}")

        del model, params, inter_a, inter_b
        gc.collect()
        jax.clear_caches()

    _RESULTS["bs5a"] = results
    _dump("bs5a_test_d_both_configs")

    c = results["centered"]
    n = results["nocenter"]
    print("\n  Сводка:")
    print(f"    centered: kernel_out[:T]={c['kernel_out_diff_lessT']:.3e}  logits[:T]={c['logits_diff_lessT']:.3e}")
    print(f"    nocenter: kernel_out[:T]={n['kernel_out_diff_lessT']:.3e}  logits[:T]={n['logits_diff_lessT']:.3e}")

    if n["kernel_out_diff_lessT"] == 0.0 and n["logits_diff_lessT"] == 0.0:
        print("\n  => ПОДТВЕРЖДЕНО: nocenter причинен НА ОБОИХ уровнях (kernel out И logits),")
        print("     бит-в-бит, через bf16-стек. Источник неточности -- ТОЛЬКО centered-путь.")
        print("     Усиление 1e-6 -> 1e-2 -- свойство downstream-архитектуры поверх")
        print("     микроскопической неточности centered-кернела. См. BS5b.")
    elif n["kernel_out_diff_lessT"] > 0.0:
        print("\n  => НЕОЖИДАННО: nocenter kernel out тоже не бит-точен. Пересмотреть")
        print("     заключение H4 (там смотрели только logits, не kernel out напрямую).")
    else:
        print("\n  => nocenter logits течёт, хотя kernel out чист -- утечка не в кернеле")
        print("     вообще ни в одном режиме; искать в Dense/RMSNorm/attend отдельно.")

    return results


# ==========================================================================
# BS5b: bisection усиления БЕЗ кернела -- искусственный шум на "чистом"
# (nocenter, гарантированно причинном) kernel out, прогнанный через
# downstream Mixer/Block стек.
# ==========================================================================
def _build_downstream_only(d_model=768, n_heads=6):
    """Downstream-часть Mixer + MLP + final_norm + attend, БЕЗ кернела.
    Принимает готовый kernel out (b, l, n_heads, d_head) и out_gate
    (b, l, d) на входе -- ровно то, что Mixer передаёт дальше после
    вызова gdn2_pallas_forward_trainable."""
    d_head = d_model // n_heads

    class MixerTail(nn.Module):
        @nn.compact
        def __call__(self, kernel_out, out_gate, x_dtype):
            b, l, h, dh = kernel_out.shape
            out = kernel_out.reshape(b, l, h * dh)
            out = nn.RMSNorm(epsilon=1e-6, name="mixer_out_norm")(out).astype(x_dtype)
            return nn.Dense(d_model, use_bias=False, name="out_proj", dtype=jnp.bfloat16)(
                out * jax.nn.silu(out_gate)
            )

    class MLP(nn.Module):
        @nn.compact
        def __call__(self, x):
            d = x.shape[-1]
            hdim = 4 * d
            gate = nn.Dense(hdim, use_bias=False, name="gate_proj", dtype=jnp.bfloat16)(x)
            up = nn.Dense(hdim, use_bias=False, name="up_proj", dtype=jnp.bfloat16)(x)
            act = jax.nn.silu(gate) * up
            return nn.Dense(d, use_bias=False, name="down_proj", dtype=jnp.bfloat16)(act)

    class DownstreamOnly(nn.Module):
        """Emulates: x (post-embed residual stream) + Mixer-tail(kernel_out) -> MLP -> final_norm -> attend."""
        num_extra_layers: int  # additional plain Blocks after layer 0's tail, to emulate depth amplification

        @nn.compact
        def __call__(self, x_pre_mixer, kernel_out, out_gate, embed_table):
            x_dtype = x_pre_mixer.dtype
            h = MixerTail(name="tail_0")(kernel_out, out_gate, x_dtype)
            x = jnp.nan_to_num(jnp.clip(x_pre_mixer + h, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)
            m = MLP(name="mlp_0")(nn.RMSNorm(epsilon=1e-6, name="mlp_norm_0")(x))
            x = jnp.nan_to_num(jnp.clip(x + m, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)

            for i in range(self.num_extra_layers):
                # plain self-contained block, no external kernel -- just to see if
                # depth alone keeps amplifying an already-injected difference
                hh = nn.Dense(x.shape[-1], use_bias=False, name=f"extra_mix_{i}", dtype=jnp.bfloat16)(
                    nn.RMSNorm(epsilon=1e-6, name=f"extra_norm1_{i}")(x)
                )
                x = jnp.nan_to_num(jnp.clip(x + hh.astype(x_dtype), -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)
                mm = MLP(name=f"extra_mlp_{i}")(nn.RMSNorm(epsilon=1e-6, name=f"extra_norm2_{i}")(x))
                x = jnp.nan_to_num(jnp.clip(x + mm, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)

            x = nn.RMSNorm(epsilon=1e-6, name="final_norm")(x).astype(x_dtype)
            return x @ embed_table.T

    return DownstreamOnly


def test_bs5b_amplification_without_kernel(cfg):
    print("\n" + "=" * 78)
    print("BS5b: усиление искусственного шума 1e-6 через downstream-стек, БЕЗ кернела")
    print("=" * 78)

    L, B = cfg["seq_len"], cfg["bsz"]
    d_model, n_heads, d_head = 768, 6, 128
    T = cfg["T"]
    noise_scale = cfg["injected_noise_scale"]

    # --- Собираем "чистый" (nocenter) kernel out через реальный forward,
    #     чтобы иметь реалистичные значения, а не случайный шум ---
    model_nocenter = _build_model(KAGGLE_MEDIUM_NOCENTER, num_layers=1)
    params_nc = model_nocenter.init(jax.random.PRNGKey(12345), jnp.zeros((B, L), jnp.int32))["params"]
    x_a = jax.random.randint(jax.random.PRNGKey(777), (B, L), 0, 256, dtype=jnp.int32)

    @jax.jit
    def fwd_capture(p, ids):
        _logits, mutated = model_nocenter.apply({"params": p}, ids, mutable=["intermediates"])
        return mutated["intermediates"]

    inter_a = fwd_capture(params_nc, x_a)
    jax.block_until_ready(inter_a)

    kout_clean = _find_and_unwrap(inter_a, "kout_raw_0")          # (B, L, H, Dh), causal-clean
    out_gate = _find_and_unwrap(inter_a, "out_gate_0")             # (B, L, d_model)
    embed_table = params_nc["embed"]["embedding"]                  # (256, d_model)

    # x_pre_mixer = вход в Block до Mixer, т.е. embed(x_a) (num_layers=1,
    # так что residual stream перед Mixer -- ровно embedding output)
    x_pre_mixer = jnp.take(embed_table, x_a, axis=0)

    print(f"  kernel_out shape: {kout_clean.shape}  dtype: {kout_clean.dtype}")
    print(f"  injected noise scale (matches observed centered-kernel out[:T] diff): {noise_scale:.1e}")

    # --- Искусственный шум: добавляем noise_scale к kernel_out ТОЛЬКО в
    #     позиции T (эмулируя, что centered-кернель на позиции T выдаёт
    #     микроскопически другое значение) ---
    key_noise = jax.random.PRNGKey(999)
    noise = jax.random.normal(key_noise, kout_clean[:, T].shape) * noise_scale
    kout_noisy = kout_clean.at[:, T].add(noise)

    for n_extra in cfg["extra_layers_grid"]:
        DownstreamOnly = _build_downstream_only(d_model, n_heads)
        ds_model = DownstreamOnly(num_extra_layers=n_extra)
        dummy_kout = jnp.zeros((B, L, n_heads, d_head), jnp.float32)
        dummy_gate = jnp.zeros((B, L, d_model), jnp.float32)
        ds_params = ds_model.init(
            jax.random.PRNGKey(555), x_pre_mixer, dummy_kout, dummy_gate, embed_table
        )["params"]

        @jax.jit
        def ds_fwd(p, kout):
            return ds_model.apply({"params": p}, x_pre_mixer, kout, out_gate, embed_table)

        logits_clean = ds_fwd(ds_params, kout_clean)
        logits_noisy = ds_fwd(ds_params, kout_noisy)
        jax.block_until_ready((logits_clean, logits_noisy))

        diff_lessT = _max_abs_diff(logits_clean[:, :T], logits_noisy[:, :T]) if T > 0 else 0.0
        diff_atT = _max_abs_diff(logits_clean[:, T], logits_noisy[:, T])

        amplification = diff_lessT / max(noise_scale, 1e-30)

        _RESULTS[f"bs5b.extra_layers={n_extra}"] = dict(
            noise_scale=noise_scale,
            diff_lessT=diff_lessT,
            diff_atT=diff_atT,
            amplification_factor=amplification,
        )
        print(f"\n  extra_layers={n_extra}: logits[:T] diff={diff_lessT:.3e}  "
              f"(input noise={noise_scale:.1e}, amplification={amplification:.1f}x)  "
              f"logits[T] diff={diff_atT:.3e}")

        del ds_model, ds_params
        gc.collect()
        jax.clear_caches()

    _dump("bs5b_amplification_without_kernel")

    print("\n  Интерпретация:")
    print("  Если diff[:T] здесь того же порядка (~1e-2), что наблюдалось в Test D для")
    print("  centered kernel (kernel_out~1.7e-6 -> logits~1.6e-2), это ПРЯМО доказывает,")
    print("  что усиление -- свойство RMSNorm/bf16/residual/Dense downstream-стека,")
    print("  а НЕ специфика centered-кернела как такового. Кернель лишь предоставляет")
    print("  затравку ~1e-6, дальше её усиливает архитектура.")
    print("  Если diff[:T] здесь << 1e-2 (например, остаётся ~1e-5) -- усиление, видимое")
    print("  в Test D, требует ЧЕГО-ТО ЕЩЁ специфичного для реального forward пути")
    print("  (например, взаимодействия шума С РЕАЛЬНЫМИ q/k/v на позициях >T через")
    print("  несколько слоёв, чего этот изолированный тест не воспроизводит).")

    del model_nocenter, params_nc, inter_a
    gc.collect()
    jax.clear_caches()


# ==========================================================================
# Entrypoint
# ==========================================================================
RUN_CONFIG = dict(
    bsz=2,
    seq_len=1024,
    num_layers=1,
    T=1023,                        # worst-case T из Test D
    injected_noise_scale=1.7e-6,   # ровно то, что Test D увидел на kernel out[:T] для centered
    extra_layers_grid=[0, 1, 3, 7, 12],  # эмулирует глубину 1,2,4,8,13 слоёв из H3
)


def main(cfg=RUN_CONFIG):
    print(f"JAX version: {jax.__version__}")
    print(f"Devices: {jax.devices()}")
    t0 = time.time()

    test_bs5a_test_d_both_configs(cfg)
    test_bs5b_amplification_without_kernel(cfg)

    elapsed = time.time() - t0
    _dump("FINAL_bs5_results")
    print("\n" + "=" * 78)
    print(f"BS5 COMPLETE. Elapsed: {elapsed:.1f}s")
    print(f"Все результаты: {OUT_DIR}/")
    print("=" * 78)


if __name__ == "__main__":
    main()

JAX version: 0.11.1
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]

BS5a: Test D (kernel out[:T]/logits[:T] diff) для centered И nocenter

  --- centered (T=1023) ---
    input[:T] diff       = 0.000e+00   (должно быть 0.0)
    kernel out[:T] diff  = 1.669e-06
    kernel out[T]  diff  = 9.351e-01   (positive control, должно быть >0)
    logits[:T] diff      = 1.562e-02

  --- nocenter (T=1023) ---
    input[:T] diff       = 0.000e+00   (должно быть 0.0)
    kernel out[:T] diff  = 0.000e+00
    kernel out

In [27]:
# gdn2_reconcile.py
import jax, jax.numpy as jnp
import sys
sys.path.append("gdn2-test")  # где лежат оба старых скрипта

# 1. Импортируем H3's builder НАПРЯМУЮ (не переписываем)
from Atomic_ops.configs import KAGGLE_MEDIUM

config = KAGGLE_MEDIUM
L, B = 1024, 2
num_layers = 1

model = _build_mini_model(config, num_layers=num_layers)  # ТА САМАЯ функция из H3
init_rng = jax.random.PRNGKey(12345)
dummy = jnp.zeros((B, L), dtype=jnp.int32)
params = model.init(init_rng, dummy)["params"]

x_a = jax.random.randint(jax.random.PRNGKey(777), (B, L), 0, 256, dtype=jnp.int32)

@jax.jit
def forward(p, ids):
    return model.apply({"params": p}, ids).astype(jnp.float32)

logits_a = forward(params, x_a)
jax.block_until_ready(logits_a)

for T in [1, 127, 255, 383, 511, 639, 767, 895, 1023]:
    x_b = x_a.at[:, T].set((x_a[:, T] + 1) % 256)
    logits_b = forward(params, x_b)
    diff = float(jnp.max(jnp.abs(logits_a[:, :T] - logits_b[:, :T]))) if T > 0 else 0.0
    print(f"T={T:4d} diff={diff:.3e}")

T=   1 diff=1.069e-02
T= 127 diff=1.412e-02
T= 255 diff=1.564e-02
T= 383 diff=1.260e-02
T= 511 diff=1.168e-02
T= 639 diff=1.392e-02
T= 767 diff=1.262e-02
T= 895 diff=1.374e-02
T=1023 diff=1.629e-02


In [28]:
"""
gdn2_bs1_reconcile_h3_builder.py

BS1-методология поверх ЕДИНСТВЕННОГО источника правды -- _build_mini_model
из gdn2_causal_leak_diagnostics.py (H3). Импортируется ТОЛЬКО функция, не
модуль: сборка модели идёт ровно тем кодом, что дал H3-leak.

Перехват входов кернела делается через sys.modules[функция.__module__]:
берём модуль-родитель и на время подменяем в его глобалах
gdn2_pallas_forward_trainable -- это ровно тот глобал, который видит
_build_mini_model при выполнении.
"""
from __future__ import annotations

import gc
import json
import os
import sys
import time

import jax
import jax.numpy as jnp

# --- Импорт ТОЛЬКО функции. Подправьте путь, если скрипт лежит иначе ---
_H3_DIR = "gdn2-test"
if _H3_DIR not in sys.path:
    sys.path.append(_H3_DIR)


from Atomic_ops.configs import KernelConfig, KAGGLE_MEDIUM, KAGGLE_MEDIUM_NOCENTER  # noqa: E402
from Atomic_ops.gdn2_pipeline import gdn2_pallas_forward_trainable as _bare_kernel  # noqa: E402

# Модуль-родитель функции -- чтобы можно было monkey-patch'ить его глобал
_H3_MODULE = sys.modules[_build_mini_model.__module__]

_RESULTS = {}
OUT_DIR = "./gdn2_bs1_reconcile_h3_results"
os.makedirs(OUT_DIR, exist_ok=True)


def _dump(tag):
    path = os.path.join(OUT_DIR, f"{tag}.json")
    with open(path, "w") as f:
        json.dump(_RESULTS, f, indent=2, default=str)
    print(f"    [checkpoint written: {path}]")


def _max_abs_diff(a, b):
    return float(jnp.max(jnp.abs(jnp.asarray(a, jnp.float32) - jnp.asarray(b, jnp.float32))))


def _boundary_positions(config: KernelConfig, seq_len: int):
    pts = set()
    for base in range(0, seq_len, config.bt):
        if 0 < base < seq_len:
            pts.add(base)
    for base in range(0, seq_len, config.bc):
        if 0 < base < seq_len:
            pts.add(base)
    return sorted(pts)


# ==========================================================================
# Monkey-patch: перехват РЕАЛЬНЫХ входов кернела в namespace H3-модуля
# ==========================================================================
class _KernelCapture:
    """Подменяет gdn2_pallas_forward_trainable в ГЛОБАЛАХ модуля-родителя
    _build_mini_model. При num_layers=1 кернел вызывается ровно один раз
    на forward -- перехват однозначен."""

    def __init__(self):
        self.orig = None
        self.captured = None

    def __enter__(self):
        if not hasattr(_H3_MODULE, "gdn2_pallas_forward_trainable"):
            raise RuntimeError(
                f"в модуле {_H3_MODULE.__name__!r} нет глобала "
                f"gdn2_pallas_forward_trainable -- проверьте, как H3-файл "
                f"импортирует кернель"
            )
        self.orig = _H3_MODULE.gdn2_pallas_forward_trainable
        self.captured = {}

        def _wrapper(q, k, v, w, b, g, **kwargs):
            out, hf = self.orig(q, k, v, w, b, g, **kwargs)
            if not isinstance(q, jax.core.Tracer):
                self.captured.update(q=q, k=k, v=v, w=w, b=b, g=g, out=out)
            return out, hf

        _H3_MODULE.gdn2_pallas_forward_trainable = _wrapper
        return self

    def __exit__(self, *exc):
        _H3_MODULE.gdn2_pallas_forward_trainable = self.orig
        return False


# ==========================================================================
# BS1: replay реальных layer-0 входов H3-модели в ИЗОЛИРОВАННОМ @jax.jit
# ==========================================================================
def test_bs1_reconcile(cfg):
    print("\n" + "=" * 78)
    print("BS1: H3-builder (_build_mini_model импортирована как есть) + replay")
    print("=" * 78)

    config = KAGGLE_MEDIUM
    L, B = cfg["seq_len"], cfg["bsz"]
    num_layers = cfg["bisect_num_layers"]
    T_list = cfg["probe_T_list"]
    tol = cfg["causal_tol"]

    # --- 1. Собираем модель ТОЙ ЖЕ функцией из H3 ---
    model = _build_mini_model(config, num_layers=num_layers)
    params = model.init(jax.random.PRNGKey(12345), jnp.zeros((B, L), jnp.int32))["params"]
    print(f"  model type: {type(model).__name__}  num_layers={num_layers}")

    x_a = jax.random.randint(jax.random.PRNGKey(777), (B, L), 0, 256, dtype=jnp.int32)

    @jax.jit
    def forward_jit(p, ids):
        return model.apply({"params": p}, ids).astype(jnp.float32)

    def forward_eager_capture(p, ids):
        with _KernelCapture() as cap:
            logits = model.apply({"params": p}, ids).astype(jnp.float32)
        if not cap.captured:
            raise RuntimeError("monkey-patch не поймал вызов кернела")
        return logits, dict(cap.captured)

    @jax.jit
    def bare_fwd(q_, k_, v_, w_, b_, g_):
        o, _ = _bare_kernel(q_, k_, v_, w_, b_, g_, scale=1.0, config=config)
        return o

    # --- прогрев A ---
    logits_a_jit = forward_jit(params, x_a)
    jax.block_until_ready(logits_a_jit)
    logits_a_eager, cap_a = forward_eager_capture(params, x_a)
    jax.block_until_ready(logits_a_eager)

    print(f"  [debug] captured: q={cap_a['q'].shape}  "
          f"w in ({float(cap_a['w'].min()):.3f},{float(cap_a['w'].max()):.3f})  "
          f"g in ({float(cap_a['g'].min()):.4f},{float(cap_a['g'].max()):.4f})")

    results = {}
    print(f"\n  {'T':>5}  {'inmodel(jit)':>12}  {'inmodel(eager)':>14}  "
          f"{'input<T':>10}  {'isolated':>10}  verdict")
    print("  " + "-" * 78)

    for T in T_list:
        if T <= 0 or T >= L:
            continue
        x_b = x_a.at[:, T].set((x_a[:, T] + 1) % 256)

        logits_b_jit = forward_jit(params, x_b)
        jax.block_until_ready(logits_b_jit)
        logits_b_eager, cap_b = forward_eager_capture(params, x_b)
        jax.block_until_ready(logits_b_eager)

        diff_inmodel_jit   = _max_abs_diff(logits_a_jit[:, :T],   logits_b_jit[:, :T])
        diff_inmodel_eager = _max_abs_diff(logits_a_eager[:, :T], logits_b_eager[:, :T])

        input_diff = max(
            _max_abs_diff(cap_a[n][:, :T], cap_b[n][:, :T])
            for n in ("q", "k", "v", "w", "b", "g")
        )

        oa = bare_fwd(cap_a["q"], cap_a["k"], cap_a["v"],
                      cap_a["w"], cap_a["b"], cap_a["g"])
        ob = bare_fwd(cap_b["q"], cap_b["k"], cap_b["v"],
                      cap_b["w"], cap_b["b"], cap_b["g"])
        jax.block_until_ready((oa, ob))
        diff_isolated = _max_abs_diff(oa[:, :T], ob[:, :T])

        if input_diff >= tol:
            verdict = "ВХОДЫ РАЗНЫЕ ДО T -- leak ВЫШЕ кернела"
        elif diff_isolated >= tol:
            verdict = "MATH/VALUES: leak в изолированном replay"
        elif diff_inmodel_jit >= tol:
            verdict = "INFRA: leak ТОЛЬКО внутри H3-модели"
        else:
            verdict = "чисто"

        results[T] = dict(
            diff_inmodel_jit=diff_inmodel_jit,
            diff_inmodel_eager=diff_inmodel_eager,
            input_diff_before_T=input_diff,
            diff_isolated=diff_isolated,
            verdict=verdict,
        )
        print(f"  {T:>5}  {diff_inmodel_jit:>12.3e}  {diff_inmodel_eager:>14.3e}  "
              f"{input_diff:>10.3e}  {diff_isolated:>10.3e}  {verdict}")

    _RESULTS["bs1_reconcile"] = results
    _dump("bs1_reconcile_h3_builder")

    del model, params
    gc.collect()
    jax.clear_caches()
    return results


# ==========================================================================
# Entrypoint
# ==========================================================================
RUN_CONFIG = dict(
    bsz=2,
    H=6,
    D=128,
    seq_len=1024,
    causal_tol=1e-5,
    bisect_num_layers=1,
    probe_T_list=[1, 127, 255, 383, 511, 639, 767, 895, 1023],
)


def main(cfg=RUN_CONFIG):
    print(f"JAX version: {jax.__version__}")
    print(f"Devices: {jax.devices()}")
    t0 = time.time()

    bs1 = test_bs1_reconcile(cfg)

    elapsed = time.time() - t0
    _dump("FINAL_bs1_reconcile")

    print("\n" + "=" * 78)
    print(f"BS1-RECONCILE COMPLETE. Elapsed: {elapsed:.1f}s")
    print("=" * 78)
    print("\nСводный вердикт:")
    for T, r in sorted(bs1.items()):
        print(f"  T={T:4d}  in-model(jit)={r['diff_inmodel_jit']:.3e}  "
              f"isolated={r['diff_isolated']:.3e}  "
              f"input<T={r['input_diff_before_T']:.3e}  -> {r['verdict']}")

    print("\nКак читать:")
    print("  in-model(jit)~1e-2 & isolated~0 & input<T~0  -> (A) INFRA/компиляция:")
    print("     тот же кернель течёт ВНУТРИ H3-модели и чист в изоляции на тех же")
    print("     значениях. Дальше -- построчный diff двух builder'ов.")
    print("  isolated ~ in-model ~ 1e-2                    -> (B) MATH/VALUES:")
    print("     кернель течёт и в изоляции -- искать конкретные числа в q/k/v/w/b/g.")
    print("  input<T >= tol                                -> leak ВЫШЕ кернела:")
    print("     баг в Dense/embed/silu/sigmoid, кернель вообще ни при чём.")
    print(f"\nВсе результаты: {OUT_DIR}/")


if __name__ == "__main__":
    main()

JAX version: 0.11.1
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]

BS1: H3-builder (_build_mini_model импортирована как есть) + replay
  model type: MiniLM  num_layers=1
  [debug] captured: q=(2, 1024, 6, 128)  w in (0.006,0.992)  g in (-0.0781,-0.0003)

      T  inmodel(jit)  inmodel(eager)     input<T    isolated  verdict
  ------------------------------------------------------------------------------
      1     1.069e-02       9.766e-03   0.000e+00   7.413e-07  INFRA: leak ТОЛЬКО внутри H3-модели
  

In [44]:
"""
gdn2_bs1_hlo_diff.py

Step 1 из плана после BS1-reconcile. Результат предыдущего прогона
показал: тот же кернель на ТЕХ ЖЕ captured входных тензорах
  - внутри H3-модели (MiniLM) даёт diff ~1e-2 на [:T]
  - в изолированном @jax.jit даёт diff ~1e-6 (round-off)

input<T = 0.000e+00 на всех T -- входы кернела побитово совпадают,
leak НЕ выше кернела. Изолированный replay на captured значениях --
чистый, leak НЕ в математике. Значит pallas_call получает разный
lowering в двух контекстах.

Этот скрипт:
  1. Собирает H3-модель через _build_mini_model (импорт как есть).
  2. Инициализирует params на PRNGKey(12345).
  3. Через monkey-patch перехватывает РЕАЛЬНЫЕ q/k/v/w/b/g на входе
     кернела layer 0 для x_a.
  4. Lower'ит в StableHLO:
       (a) изолированный @jax.jit с captured q/k/v/w/b/g -> HLO_A
       (b) @jax.jit(model.apply) -> HLO_B (полный граф модели)
  5. Сохраняет оба HLO в файлы.
  6. Извлекает секции, относящиеся к custom-call / pallas / mosaic,
     и печатает их (с контекстом) из обоих HLO.
  7. Делает keyword-only diff: только строки, содержащие
     custom-call / layout / backend_config / block_shape / mosaic --
     там и живёт расхождение.

Что смотреть в выводе:
  - Секция custom-call в HLO_A (изолированный) vs HLO_B (в модели).
    Ключевое: backend_config=, layout={...}, operand_layouts=,
    result_layouts=, block_shape=.
  - Если backend_config отличается по num_warps/num_stages/block_shape
    -- причина в lowering choice (Pallas autotuner выбрал разное).
  - Если operand_layouts/result_layouts отличаются -- причина в layout
    входных тензоров, назначенных XLA для двух контекстов.
  - Если HLO_A вообще не содержит pallas/custom-call -- значит
    изолированный jit откомпилировал kernel в чистый XLA (не Pallas);
    это отдельная находка, скажет о том, что в изоляции идёт другой
    code path.

Kaggle TPU v5e-8, notebook-only, без argparse, top-level константы.
"""
from __future__ import annotations

import difflib
import gc
import json
import os
import sys
import time

import jax
import jax.numpy as jnp

# --- Импорт ТОЛЬКО функции из H3 ---
_H3_DIR = "gdn2-test"
if _H3_DIR not in sys.path:
    sys.path.append(_H3_DIR)


from Atomic_ops.configs import KAGGLE_MEDIUM  # noqa: E402
from Atomic_ops.gdn2_pipeline import gdn2_pallas_forward_trainable as _bare_kernel  # noqa: E402

_H3_MODULE = sys.modules[_build_mini_model.__module__]

OUT_DIR = "./gdn2_bs1_hlo_diff_results"
os.makedirs(OUT_DIR, exist_ok=True)

_RESULTS = {}


def _dump(tag):
    path = os.path.join(OUT_DIR, f"{tag}.json")
    with open(path, "w") as f:
        json.dump(_RESULTS, f, indent=2, default=str)
    print(f"    [checkpoint written: {path}]")


def _dump_text(name, text):
    path = os.path.join(OUT_DIR, f"{name}.txt")
    with open(path, "w") as f:
        f.write(text)
    print(f"    [saved: {path}  ({len(text)} chars, {text.count(chr(10))} lines)]")


# ==========================================================================
# Monkey-patch: перехват входов kernel'а в namespace H3-модуля
# ==========================================================================
class _KernelCapture:
    def __init__(self):
        self.orig = None
        self.captured = None

    def __enter__(self):
        if not hasattr(_H3_MODULE, "gdn2_pallas_forward_trainable"):
            raise RuntimeError(
                f"модуль {_H3_MODULE.__name__!r} не содержит глобала "
                f"gdn2_pallas_forward_trainable -- проверьте, как H3-файл "
                f"импортирует kernel"
            )
        self.orig = _H3_MODULE.gdn2_pallas_forward_trainable
        self.captured = {}

        def _wrapper(q, k, v, w, b, g, **kwargs):
            out, hf = self.orig(q, k, v, w, b, g, **kwargs)
            if not isinstance(q, jax.core.Tracer):
                self.captured.update(q=q, k=k, v=v, w=w, b=b, g=g,
                                     out=out, kwargs=dict(kwargs))
            return out, hf

        _H3_MODULE.gdn2_pallas_forward_trainable = _wrapper
        return self

    def __exit__(self, *exc):
        _H3_MODULE.gdn2_pallas_forward_trainable = self.orig
        return False


# ==========================================================================
# Lowering
# ==========================================================================
def _lower_to_hlo(fn, args):
    """Возвращает StableHLO-текст для jit-функции fn на аргументах args.
    Пробуем несколько API -- в зависимости от версии JAX/бэкенда может
    работать то или другое."""
    errors = []
    # 1. jit.lower(...).as_text()
    try:
        lowered = jax.jit(fn).lower(*args)
        return lowered.as_text(), "jit.lower().as_text()"
    except Exception as e:
        errors.append(f"jit.lower(): {type(e).__name__}: {e}")
    # 2. jax.xla_computation (старый API)
    try:
        comp = jax.xla_computation(fn)(*args)
        return comp.as_hlo_text(), "jax.xla_computation().as_hlo_text()"
    except Exception as e:
        errors.append(f"xla_computation(): {type(e).__name__}: {e}")
    # 3. jit.lower().compile().as_text() -- иногда lower без compile не
    #    даёт текст на TPU
    try:
        lowered = jax.jit(fn).lower(*args)
        compiled = lowered.compile()
        return compiled.as_text(), "jit.lower().compile().as_text()"
    except Exception as e:
        errors.append(f"lower().compile(): {type(e).__name__}: {e}")
    raise RuntimeError("не удалось получить HLO ни одним способом:\n  "
                       + "\n  ".join(errors))


# ==========================================================================
# Извлечение и сравнение секций
# ==========================================================================
_PALLAS_KEYWORDS = (
    "custom-call", "custom_call",
    "pallas", "mosaic", "tpu_custom",
    "backend_config",
    "block_shape",
    "operand_layouts", "result_layouts",
)


def _extract_sections_with_context(hlo_text, keywords, context_before=1, context_after=8):
    """Короткие блоки вокруг каждого вхождения. Long строки обрезаются."""
    lines = hlo_text.splitlines()

    def _trim(ln, max_len=220):
        return ln if len(ln) <= max_len else ln[:max_len] + f"...[{len(ln)-max_len} more]"

    hits = []
    last_hi = -1
    for i, ln in enumerate(lines):
        if not any(kw in ln for kw in keywords):
            continue
        if i < last_hi:  # уже попал в предыдущий блок
            continue
        lo = max(0, i - context_before)
        hi = min(len(lines), i + context_after)
        last_hi = hi
        block = "\n".join(
            f"  {j:5d}: {_trim(lines[j])}" for j in range(lo, hi)
        )
        hits.append((i, block))
    return hits


import hashlib
import re

def _summarize_pallas_lines(hlo_text):
    """Возвращает список СТРУКТУРНЫХ строк (без base64 тел),
    пригодных для diff'а. Каждое вхождение custom-call сжимается до
    одной строки с метаданными."""
    out = []
    # Разбиваем HLO по custom_call-строкам
    custom_call_re = re.compile(r"custom_call\s+@(\w+)\(")
    backend_re = re.compile(r'backend_config = "(.+?)", kernel_name')
    kernel_name_re = re.compile(r'kernel_name = "([^"]+)"')
    layouts_re = re.compile(r'(operand_layouts|result_layouts) = (\[[^\]]*\])')
    shapes_re = re.compile(r': (\([^)]*\)) -> (\([^)]*\))')

    lines = hlo_text.splitlines()
    for i, ln in enumerate(lines):
        m = custom_call_re.search(ln)
        if not m:
            continue
        target = m.group(1)
        kn = kernel_name_re.search(ln)
        kn = kn.group(1) if kn else "?"
        layouts = layouts_re.findall(ln)
        shapes = shapes_re.search(ln)
        bc = backend_re.search(ln)
        bc_len = len(bc.group(1)) if bc else 0
        bc_hash = hashlib.md5(bc.group(1).encode()).hexdigest()[:12] if bc else "?"
        out.append(
            f"line {i:5d}  target={target}  kernel={kn}  "
            f"bc_len={bc_len} bc_md5={bc_hash}  "
            f"operand_layouts={[l[1] for l in layouts if l[0]=='operand_layouts']}  "
            f"result_layouts={[l[1] for l in layouts if l[0]=='result_layouts']}  "
            f"shapes={shapes.group(0) if shapes else '?'}"
        )
    return out

# ==========================================================================
# Main test
# ==========================================================================
def test_hlo_diff(cfg):
    print("\n" + "=" * 78)
    print("HLO diff: isolated-jit vs in-model (H3 builder as single source of truth)")
    print("=" * 78)

    config = KAGGLE_MEDIUM
    L, B = cfg["seq_len"], cfg["bsz"]
    num_layers = cfg["bisect_num_layers"]

    # --- 1. Собираем H3-модель ---
    model = _build_mini_model(config, num_layers=num_layers)
    params = model.init(jax.random.PRNGKey(12345), jnp.zeros((B, L), jnp.int32))["params"]
    print(f"  model type: {type(model).__name__}  num_layers={num_layers}")

    x_a = jax.random.randint(jax.random.PRNGKey(777), (B, L), 0, 256, dtype=jnp.int32)

    # --- 2. Перехватываем РЕАЛЬНЫЕ входы kernel'а layer 0 (eager) ---
    def forward_eager_capture(p, ids):
        with _KernelCapture() as cap:
            logits = model.apply({"params": p}, ids).astype(jnp.float32)
        if not cap.captured:
            raise RuntimeError("monkey-patch не поймал вызов kernel'а")
        return logits, dict(cap.captured)

    print("  running eager forward to capture kernel inputs...")
    _logits, cap = forward_eager_capture(params, x_a)
    jax.block_until_ready(_logits)
    q = cap["q"]; k = cap["k"]; v = cap["v"]
    w = cap["w"]; b = cap["b"]; g = cap["g"]
    print(f"  captured: q={q.shape} dtype={q.dtype}  "
          f"w in ({float(w.min()):.3f},{float(w.max()):.3f})  "
          f"g in ({float(g.min()):.4f},{float(g.max()):.4f})")

    # --- 3. Изолированный jit: ровно та же call-site, что H3-модель ---
    def isolated(q_, k_, v_, w_, b_, g_):
        out, _hf = _bare_kernel(q_, k_, v_, w_, b_, g_, scale=1.0, config=config)
        return out

    print("  lowering isolated jit...")
    hlo_iso, how_iso = _lower_to_hlo(isolated, (q, k, v, w, b, g))
    print(f"    via {how_iso}")

    # --- 4. In-model jit: полный граф MiniLM ---
    def inmodel(ids):
        return model.apply({"params": params}, ids)

    print("  lowering in-model jit (full MiniLM graph)...")
    hlo_model, how_model = _lower_to_hlo(inmodel, (x_a,))
    print(f"    via {how_model}")

    # --- 5. Сохраняем в файлы ---
    _dump_text("HLO_isolated", hlo_iso)
    _dump_text("HLO_inmodel", hlo_model)

    # --- 6. Краткая сводка ---
    _RESULTS["hlo_isolated_how"] = how_iso
    _RESULTS["hlo_inmodel_how"] = how_model
    _RESULTS["hlo_isolated_n_lines"] = hlo_iso.count("\n")
    _RESULTS["hlo_inmodel_n_lines"] = hlo_model.count("\n")

    n_iso_hits = sum(hlo_iso.count(kw) for kw in _PALLAS_KEYWORDS)
    n_mod_hits = sum(hlo_model.count(kw) for kw in _PALLAS_KEYWORDS)
    print(f"\n  HLO sizes: isolated={hlo_iso.count(chr(10))} lines, "
          f"in-model={hlo_model.count(chr(10))} lines")
    print(f"  pallas-related keyword hits: isolated={n_iso_hits}, "
          f"in-model={n_mod_hits}")

    # --- 7. Печатаем секции вокруг pallas/custom-call ---
    print("\n" + "-" * 78)
    print("ISOLATED HLO -- pallas/custom-call sections:")
    print("-" * 78)
    for line_no, block in _extract_sections_with_context(hlo_iso, _PALLAS_KEYWORDS):
        print(f"\n  [hit at line {line_no}]")
        print(block)

    print("\n" + "-" * 78)
    print("IN-MODEL HLO -- pallas/custom-call sections:")
    print("-" * 78)
    for line_no, block in _extract_sections_with_context(hlo_model, _PALLAS_KEYWORDS):
        print(f"\n  [hit at line {line_no}]")
        print(block)

    # --- 8. Keyword-only diff ---
    print("\n" + "=" * 78)
    print("KEYWORD-ONLY DIFF (только строки, релевантные pallas/layout/config)")
    print("=" * 78)
    print("\n" + "=" * 78)
    print("STRUCTURED PALLAS SUMMARY (без base64 — только метаданные)")
    print("=" * 78)

    summary_iso = _summarize_pallas_lines(hlo_iso)
    summary_mod = _summarize_pallas_lines(hlo_model)

    print("\n--- isolated ---")
    for s in summary_iso:
        print(f"  {s}")
    print("\n--- in-model ---")
    for s in summary_mod:
        print(f"  {s}")

    print("\n--- diff ---")
    diff = list(difflib.unified_diff(
        summary_iso, summary_mod,
        fromfile="isolated", tofile="in-model", lineterm="",
    ))
    if not diff:
        print("  (идентичны -- pallas lowering не отличается)")
    else:
        for ln in diff:
            print(f"  {ln}")    
    if not diff:
        print("  (нет различий в keyword-строках -- странно, ожидали хотя бы layout)")
    else:
        for ln in diff:
            print(f"  {ln}")

    _RESULTS["n_iso_pallas_entries"] = len(summary_iso)
    _RESULTS["n_mod_pallas_entries"] = len(summary_mod)
    _RESULTS["pallas_identical"] = (summary_iso == summary_mod)
    _dump("hlo_diff_summary")
    del model, params
    gc.collect()
    jax.clear_caches()


# ==========================================================================
# Entrypoint
# ==========================================================================
RUN_CONFIG = dict(
    bsz=2,
    seq_len=1024,
    bisect_num_layers=1,
)


def main(cfg=RUN_CONFIG):
    print(f"JAX version: {jax.__version__}")
    print(f"Devices: {jax.devices()}")
    t0 = time.time()

    test_hlo_diff(cfg)

    elapsed = time.time() - t0
    print("\n" + "=" * 78)
    print(f"HLO DIFF COMPLETE. Elapsed: {elapsed:.1f}s")
    print(f"Full HLOs saved to: {OUT_DIR}/HLO_isolated.txt and HLO_inmodel.txt")
    print("=" * 78)
    print("\nКак читать результат:")
    print("  1. Смотри на строки 'custom-call' в каждом HLO. Обычно это")
    print("     отдельная HLO-операция, у которой есть:")
    print("         custom_call_target=\"...\"")
    print("         backend_config=\"...\"       <-- параметры lowering'а")
    print("         operand_layouts={...}      <-- layout входов")
    print("         result_layouts={...}       <-- layout выходов")
    print("         api_version=...")
    print("  2. Сравни backend_config между isolated и in-model:")
    print("     - если есть различия в num_warps / num_stages / block_shape --")
    print("       Pallas autotuner выбрал разный lowering, и один из них багованный.")
    print("     - если operand_layouts отличаются -- XLA поставил другой layout")
    print("       входным тензорам в модели vs в изоляции, и kernel на одном")
    print("       из layout'ов содержит bug на границах chunk'ов.")
    print("  3. Если HLO_isolated вообще НЕ содержит pallas/custom-call --")
    print("     изолированный jit откомпилил kernel в чистый XLA. Это означает,")
    print("     что в модели и в изоляции идут РАЗНЫЕ code paths, и leak -- в")
    print("     Pallas-версии. Тогда причина -- в Mosaic lowering, не в XLA.")
    print("  4. Если diff keyword-only пуст, но leak есть в одном контексте --")
    print("     иди в полные файлы HLO_*.txt и делай полноценный diff вручную:")
    print("         diff HLO_isolated.txt HLO_inmodel.txt | head -200")
    print("     ...но фильтруй шум от нерелевантных операций модели.")


if __name__ == "__main__":
    main()

JAX version: 0.11.1
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]

HLO diff: isolated-jit vs in-model (H3 builder as single source of truth)
  model type: MiniLM  num_layers=1
  running eager forward to capture kernel inputs...
  captured: q=(2, 1024, 6, 128) dtype=float32  w in (0.006,0.992)  g in (-0.0781,-0.0003)
  lowering isolated jit...
    via jit.lower().as_text()
  lowering in-model jit (full MiniLM graph)...
    via jit.lower().as_text()
    [saved: ./gdn2_bs1_hlo_diff_results/HLO_isolated.txt

In [32]:
@jax.jit
def fwd(p, ids):
    return model.apply({"params": p}, ids)

o1 = fwd(params, x_a); jax.block_until_ready(o1)
o2 = fwd(params, x_a); jax.block_until_ready(o2)
print("self-diff:", float(jnp.max(jnp.abs(o1 - o2))))

self-diff: 0.0


In [36]:
# ============================================================================
# Test B — полный, самодостаточный
# ============================================================================
import sys
import jax
import jax.numpy as jnp

# --- 1) Захват внутри jit ---
_h3_mod = sys.modules[_build_mini_model.__module__]

@jax.jit
def fwd_jit_capture(p, ids):
    captured = []
    orig = _h3_mod.gdn2_pallas_forward_trainable

    def _cap(q, k, v, w, b, g, **kwargs):
        out, hf = orig(q, k, v, w, b, g, **kwargs)
        captured.append((q, k, v, w, b, g))
        return out, hf

    _h3_mod.gdn2_pallas_forward_trainable = _cap
    try:
        logits = model.apply({"params": p}, ids)
    finally:
        _h3_mod.gdn2_pallas_forward_trainable = orig

    q, k, v, w, b, g = captured[0]
    return logits, q, k, v, w, b, g

print("running jit-capture...")
logits_jit, q_jit, k_jit, v_jit, w_jit, b_jit, g_jit = fwd_jit_capture(params, x_a)
jax.block_until_ready((logits_jit, q_jit, k_jit, v_jit, w_jit, b_jit, g_jit))
print("  jit logits dtype:", logits_jit.dtype, "shape:", logits_jit.shape)

# --- 2) Eager-capture через monkey-patch (определяем здесь же) ---
class _KernelCapture:
    def __init__(self):
        self.orig = None
        self.captured = None

    def __enter__(self):
        self.orig = _h3_mod.gdn2_pallas_forward_trainable
        self.captured = {}

        def _wrapper(q, k, v, w, b, g, **kwargs):
            out, hf = self.orig(q, k, v, w, b, g, **kwargs)
            if not isinstance(q, jax.core.Tracer):
                self.captured.update(q=q, k=k, v=v, w=w, b=b, g=g, out=out)
            return out, hf

        _h3_mod.gdn2_pallas_forward_trainable = _wrapper
        return self

    def __exit__(self, *exc):
        _h3_mod.gdn2_pallas_forward_trainable = self.orig
        return False


def forward_eager_capture(p, ids):
    with _KernelCapture() as cap:
        logits = model.apply({"params": p}, ids)
    if not cap.captured:
        raise RuntimeError("monkey-patch не поймал вызов kernel'а в eager")
    return logits, dict(cap.captured)


print("running eager-capture...")
logits_eager, cap_eager = forward_eager_capture(params, x_a)
jax.block_until_ready(logits_eager)
print("  eager logits dtype:", logits_eager.dtype, "shape:", logits_eager.shape)

# --- 3) Сравнение входов kernel'а ---
q_eager = cap_eager["q"]; k_eager = cap_eager["k"]; v_eager = cap_eager["v"]
w_eager = cap_eager["w"]; b_eager = cap_eager["b"]; g_eager = cap_eager["g"]

def _maxdiff(a, b, name):
    a32 = jnp.asarray(a, jnp.float32)
    b32 = jnp.asarray(b, jnp.float32)
    d = float(jnp.max(jnp.abs(a32 - b32)))
    print(f"  {name:4s}  jit-vs-eager max|Δ| = {d:.3e}  "
          f"(range jit=({float(a32.min()):.3e},{float(a32.max()):.3e}))")
    return d

print("\nTest B: jit-internal vs eager-capture, входы kernel'а layer 0")
dq = _maxdiff(q_jit, q_eager, "q")
dk = _maxdiff(k_jit, k_eager, "k")
dv = _maxdiff(v_jit, v_eager, "v")
dw = _maxdiff(w_jit, w_eager, "w")
db = _maxdiff(b_jit, b_eager, "b")
dg = _maxdiff(g_jit, g_eager, "g")

worst = max(dq, dk, dv, dw, db, dg)
print(f"\n  WORST input diff: {worst:.3e}")

# --- 4) Сравнение logits ---
d_logits = float(jnp.max(jnp.abs(
    jnp.asarray(logits_jit, jnp.float32) - jnp.asarray(logits_eager, jnp.float32)
)))
print(f"  logits: jit-vs-eager max|Δ| = {d_logits:.3e}")

# --- 5) Вердикт ---
print("\n--- Интерпретация ---")
if worst > 1e-6:
    print("  H1 ПОДТВЕРЖДЁН: jit-internal входы kernel'а ОТЛИЧАЮТСЯ от eager.")
    print("  Твой isolated replay в BS1 подавал НЕ те числа. Leak = усиление")
    print("  от XLA-fusion через conditioning kernel'а.")
elif worst > 0.0:
    print(f"  Разница малая ({worst:.3e}) -- вероятно fp32-шум.")
else:
    print("  jit и eager дают БИТ-ИДЕНТИЧНЫЕ входы. H1 не подтверждён.")

if d_logits > 1e-5 and worst < 1e-7:
    print("  НО logits jit-vs-eager > 1e-5 -- leak ПОСЛЕ kernel'а (RMSNorm/out_proj/MLP).")
elif d_logits > 1e-5 and worst > 1e-6:
    print("  И входы, и logits расходятся -- H1 основная причина.")

running jit-capture...
  jit logits dtype: bfloat16 shape: (2, 1024, 256)
running eager-capture...
  eager logits dtype: bfloat16 shape: (2, 1024, 256)

Test B: jit-internal vs eager-capture, входы kernel'а layer 0
  q     jit-vs-eager max|Δ| = 2.535e-03  (range jit=(-6.773e-02,6.985e-01))
  k     jit-vs-eager max|Δ| = 2.323e-03  (range jit=(-5.999e-02,6.428e-01))
  v     jit-vs-eager max|Δ| = 2.598e-02  (range jit=(-2.785e-01,4.605e+00))
  w     jit-vs-eager max|Δ| = 2.686e-03  (range jit=(6.011e-03,9.924e-01))
  b     jit-vs-eager max|Δ| = 2.705e-03  (range jit=(1.281e-02,9.862e-01))
  g     jit-vs-eager max|Δ| = 1.383e-04  (range jit=(-7.821e-02,-2.620e-04))

  WORST input diff: 2.598e-02
  logits: jit-vs-eager max|Δ| = 4.688e-02

--- Интерпретация ---
  H1 ПОДТВЕРЖДЁН: jit-internal входы kernel'а ОТЛИЧАЮТСЯ от eager.
  Твой isolated replay в BS1 подавал НЕ те числа. Leak = усиление
  от XLA-fusion через conditioning kernel'а.
  И входы, и logits расходятся -- H1 основная причина.


In [37]:
# ============================================================================
# Test C: isolated replay на JIT-captured значениях, а не на eager-captured.
# Если течёт -- conditioning kernel'а. Если чист -- XLA fusion переносит
# информацию назад вне kernel'а.
# ============================================================================

@jax.jit
def fwd_jit_capture(p, ids):
    captured = []
    orig = _h3_mod.gdn2_pallas_forward_trainable
    def _cap(q, k, v, w, b, g, **kwargs):
        out, hf = orig(q, k, v, w, b, g, **kwargs)
        captured.append((q, k, v, w, b, g))
        return out, hf
    _h3_mod.gdn2_pallas_forward_trainable = _cap
    try:
        logits = model.apply({"params": p}, ids)
    finally:
        _h3_mod.gdn2_pallas_forward_trainable = orig
    q, k, v, w, b, g = captured[0]
    return logits, q, k, v, w, b, g

# --- Прогон для x_a ---
print("jit-capture x_a...")
logits_a, qj_a, kj_a, vj_a, wj_a, bj_a, gj_a = fwd_jit_capture(params, x_a)
jax.block_until_ready((logits_a, qj_a, kj_a, vj_a, wj_a, bj_a, gj_a))

# --- Прогон для x_b (T=1023, worst-case из BS1) ---
T = 1023
x_b = x_a.at[:, T].set((x_a[:, T] + 1) % 256)
print(f"jit-capture x_b (perturbed at T={T})...")
logits_b, qj_b, kj_b, vj_b, wj_b, bj_b, gj_b = fwd_jit_capture(params, x_b)
jax.block_until_ready((logits_b, qj_b, kj_b, vj_b, wj_b, bj_b, gj_b))

# --- 1) input<T для JIT-captured значений ---
print(f"\n--- input<T (JIT-captured) при T={T} ---")
inputs_a = (qj_a, kj_a, vj_a, wj_a, bj_a, gj_a)
inputs_b = (qj_b, kj_b, vj_b, wj_b, bj_b, gj_b)
names = ("q", "k", "v", "w", "b", "g")
for n, a, b in zip(names, inputs_a, inputs_b):
    d = float(jnp.max(jnp.abs(jnp.asarray(a[:, :T], jnp.float32)
                              - jnp.asarray(b[:, :T], jnp.float32))))
    print(f"  {n:4s}  jit_a vs jit_b, [:{T}]  max|Δ| = {d:.3e}")

# --- 2) in-model logits diff на [:T] ---
diff_inmodel = float(jnp.max(jnp.abs(
    jnp.asarray(logits_a[:, :T], jnp.float32)
    - jnp.asarray(logits_b[:, :T], jnp.float32)
)))
print(f"\n--- in-model logits diff [: {T}] ---")
print(f"  diff_inmodel = {diff_inmodel:.3e}")

# --- 3) isolated replay на JIT-captured значениях ---
@jax.jit
def bare_fwd(q_, k_, v_, w_, b_, g_):
    o, _ = _bare_kernel(q_, k_, v_, w_, b_, g_, scale=1.0, config=config)
    return o

print("\n--- isolated replay на JIT-captured ---")
o_jit_a = bare_fwd(qj_a, kj_a, vj_a, wj_a, bj_a, gj_a)
o_jit_b = bare_fwd(qj_b, kj_b, vj_b, wj_b, bj_b, gj_b)
jax.block_until_ready((o_jit_a, o_jit_b))
diff_isolated_jit = float(jnp.max(jnp.abs(
    jnp.asarray(o_jit_a[:, :T], jnp.float32)
    - jnp.asarray(o_jit_b[:, :T], jnp.float32)
)))
print(f"  diff_isolated (на jit-captured) = {diff_isolated_jit:.3e}")

# --- 4) isolated replay на EAGER-captured (как в BS1) ---
print("\n--- isolated replay на EAGER-captured (контроль, как в BS1) ---")
_, cap_eager_a = forward_eager_capture(params, x_a)
_, cap_eager_b = forward_eager_capture(params, x_b)
o_eager_a = bare_fwd(cap_eager_a["q"], cap_eager_a["k"], cap_eager_a["v"],
                     cap_eager_a["w"], cap_eager_a["b"], cap_eager_a["g"])
o_eager_b = bare_fwd(cap_eager_b["q"], cap_eager_b["k"], cap_eager_b["v"],
                     cap_eager_b["w"], cap_eager_b["b"], cap_eager_b["g"])
jax.block_until_ready((o_eager_a, o_eager_b))
diff_isolated_eager = float(jnp.max(jnp.abs(
    jnp.asarray(o_eager_a[:, :T], jnp.float32)
    - jnp.asarray(o_eager_b[:, :T], jnp.float32)
)))
print(f"  diff_isolated (на eager-captured) = {diff_isolated_eager:.3e}")

# --- 5) Вердикт ---
print("\n--- ИНТЕРПРЕТАЦИЯ ---")
print(f"  in-model:              {diff_inmodel:.3e}")
print(f"  isolated (jit-captured): {diff_isolated_jit:.3e}")
print(f"  isolated (eager-captured): {diff_isolated_eager:.3e}")

if diff_isolated_jit > 1e-5:
    print("\n  => ПРИЧИНА A: conditioning kernel'а.")
    print("     Isolated replay на JIT-captured ЗНАЧЕНИЯХ тоже течёт. Значит,")
    print("     kernel численно нестабилен на этих числах. Eager-capture давал")
    print("     ДРУГИЕ числа (на 1e-3..1e-2) и просто не попадал в этот режим.")
    print("     Leak = нестабильность WY-solve/centering, а не утечка из будущего.")
elif diff_isolated_jit <= 1e-5 and diff_inmodel > 1e-5:
    print("\n  => ПРИЧИНА B: XLA fusion вокруг pallas_call.")
    print("     Isolated replay на JIT-значениях ЧИСТ, но in-model течёт.")
    print("     Значит проблема не в численных значениях, а в том, что XLA")
    print("     как-то организует граф вокруг pallas_call так, что информация")
    print("     из T влияет на <T. Это инфраструктурный баг, не математика.")
else:
    print("\n  => Неожиданный результат, разбираться отдельно.")

jit-capture x_a...
jit-capture x_b (perturbed at T=1023)...

--- input<T (JIT-captured) при T=1023 ---
  q     jit_a vs jit_b, [:1023]  max|Δ| = 0.000e+00
  k     jit_a vs jit_b, [:1023]  max|Δ| = 0.000e+00
  v     jit_a vs jit_b, [:1023]  max|Δ| = 0.000e+00
  w     jit_a vs jit_b, [:1023]  max|Δ| = 0.000e+00
  b     jit_a vs jit_b, [:1023]  max|Δ| = 0.000e+00
  g     jit_a vs jit_b, [:1023]  max|Δ| = 0.000e+00

--- in-model logits diff [: 1023] ---
  diff_inmodel = 1.562e-02

--- isolated replay на JIT-captured ---
  diff_isolated (на jit-captured) = 1.669e-06

--- isolated replay на EAGER-captured (контроль, как в BS1) ---
  diff_isolated (на eager-captured) = 1.788e-06

--- ИНТЕРПРЕТАЦИЯ ---
  in-model:              1.562e-02
  isolated (jit-captured): 1.669e-06
  isolated (eager-captured): 1.788e-06

  => ПРИЧИНА B: XLA fusion вокруг pallas_call.
     Isolated replay на JIT-значениях ЧИСТ, но in-model течёт.
     Значит проблема не в численных значениях, а в том, что XLA
     как-т

In [38]:
# ============================================================================
# Test D: захватить РАВЫЙ выход kernel'а (out) внутри jit и сравнить [:T]
# между x_a и x_b. Плюс self-determinism kernel'а внутри модели.
# ============================================================================

@jax.jit
def fwd_jit_capture_full(p, ids):
    captured = []
    orig = _h3_mod.gdn2_pallas_forward_trainable
    def _cap(q, k, v, w, b, g, **kwargs):
        out, hf = orig(q, k, v, w, b, g, **kwargs)
        captured.append({"q": q, "k": k, "v": v, "w": w, "b": b, "g": g,
                         "out": out, "hf": hf})
        return out, hf
    _h3_mod.gdn2_pallas_forward_trainable = _cap
    try:
        logits = model.apply({"params": p}, ids)
    finally:
        _h3_mod.gdn2_pallas_forward_trainable = orig
    return logits, captured[0]

T = 1023
x_b = x_a.at[:, T].set((x_a[:, T] + 1) % 256)

print("Test D: захват kernel output внутри jit")
logits_a, cap_a = fwd_jit_capture_full(params, x_a)
jax.block_until_ready((logits_a, cap_a))
logits_b, cap_b = fwd_jit_capture_full(params, x_b)
jax.block_until_ready((logits_b, cap_b))

print(f"\n{'tensor':>6}  {'[:T] diff':>12}  {'[T] diff':>12}")
print("-" * 36)
for name in ("q", "k", "v", "w", "b", "g", "out", "hf"):
    va = jnp.asarray(cap_a[name], jnp.float32)
    vb = jnp.asarray(cap_b[name], jnp.float32)
    if va.ndim >= 3:  # позиционная размерность есть
        d_lessT = float(jnp.max(jnp.abs(va[:, :T] - vb[:, :T])))
        d_T     = float(jnp.max(jnp.abs(va[:, T]   - vb[:, T])))
    else:
        d_lessT = float(jnp.max(jnp.abs(va - vb)))
        d_T     = d_lessT
    print(f"{name:>6}  {d_lessT:>12.3e}  {d_T:>12.3e}")

# --- Self-determinism kernel'а ВНУТРИ модели ---
print("\nSelf-determinism kernel'а внутри модели (2 прогона x_a):")
_, cap_a2 = fwd_jit_capture_full(params, x_a)
jax.block_until_ready(cap_a2)
for name in ("out", "hf"):
    d = float(jnp.max(jnp.abs(
        jnp.asarray(cap_a[name], jnp.float32)
        - jnp.asarray(cap_a2[name], jnp.float32))))
    print(f"  {name}: self-diff = {d:.3e}")

# --- Downstream: применим те же downstream-операции к out_a и out_b ---
# Если out_a[:T] == out_b[:T] точно, а logits отличаются -- leak в downstream.
# Если out_a[:T] != out_b[:T] -- kernel течёт даже с одинаковыми входами [:T].

out_diff_lessT = float(jnp.max(jnp.abs(
    jnp.asarray(cap_a["out"], jnp.float32)[:, :T]
    - jnp.asarray(cap_b["out"], jnp.float32)[:, :T])))
logits_diff_lessT = float(jnp.max(jnp.abs(
    jnp.asarray(logits_a, jnp.float32)[:, :T]
    - jnp.asarray(logits_b, jnp.float32)[:, :T])))

print(f"\nout[:T] diff    = {out_diff_lessT:.3e}")
print(f"logits[:T] diff = {logits_diff_lessT:.3e}")

if out_diff_lessT > 1e-6:
    print("\n  => KERNEL ТЕЧЁТ ВНУТРИ МОДЕЛИ при БИТ-ИДЕНТИЧНЫХ входах [:T].")
    print("     Это невозможно для чистой функции. Значит: scratch-память,")
    print("     async-race, или aliasing буфера kernel'а с чем-то в графе.")
elif logits_diff_lessT > 1e-5:
    print("\n  => KERNEL output [:T] идентичен, но logits течёт.")
    print("     Leak в RMSNorm / out_proj / MLP / final_norm ПОСЛЕ kernel'а.")
    print("     Проверить их отдельно.")
else:
    print("\n  => Всё чисто на [:T]. Странно с учётом предыдущих наблюдений.")

Test D: захват kernel output внутри jit

tensor     [:T] diff      [T] diff
------------------------------------
     q     0.000e+00     5.086e-01
     k     0.000e+00     5.019e-01
     v     0.000e+00     3.547e+00
     w     0.000e+00     8.505e-01
     b     0.000e+00     9.028e-01
     g     0.000e+00     6.447e-02
   out     1.669e-06     9.351e-01
    hf     7.824e-01     5.284e-01

Self-determinism kernel'а внутри модели (2 прогона x_a):
  out: self-diff = 0.000e+00
  hf: self-diff = 0.000e+00

out[:T] diff    = 1.669e-06
logits[:T] diff = 1.562e-02

  => KERNEL ТЕЧЁТ ВНУТРИ МОДЕЛИ при БИТ-ИДЕНТИЧНЫХ входах [:T].
     Это невозможно для чистой функции. Значит: scratch-память,
     async-race, или aliasing буфера kernel'а с чем-то в графе.


In [41]:
# ============================================================================
# Test E (правильный): fp32-модель через модификацию исходного кода builder'а.
# ============================================================================

import inspect, jax, jax.numpy as jnp

# --- 1) Проверить, что bf16 baseline на самом деле bf16 ---
print("=== Sanity check: dtype параметров bf16-модели ===")
dtypes_bf16 = set()
for leaf in jax.tree_util.tree_leaves(params):
    if hasattr(leaf, "dtype"):
        dtypes_bf16.add(leaf.dtype)
print("  dtypes в params (bf16-модель):", dtypes_bf16)

# --- 2) Построить fp32-модель через source-модификацию ---
src = inspect.getsource(_build_mini_model)
src_fp32 = src.replace("jnp.bfloat16", "jnp.float32")
assert "bfloat16" not in src_fp32, "замена не сработала"

# exec в namespace с нужными замыканиями
ns = dict(globals())
exec(src_fp32, ns)
_build_mini_model_fp32 = ns["_build_mini_model"]

model_fp32 = _build_mini_model_fp32(config, num_layers=1)
params_fp32 = model_fp32.init(
    jax.random.PRNGKey(12345),
    jnp.zeros((B, L), jnp.int32)
)["params"]

# --- 3) Проверить, что fp32-модель реально fp32 ---
print("\n=== Sanity check: dtype параметров fp32-модели ===")
dtypes_fp32 = set()
for leaf in jax.tree_util.tree_leaves(params_fp32):
    if hasattr(leaf, "dtype"):
        dtypes_fp32.add(leaf.dtype)
print("  dtypes в params (fp32-модель):", dtypes_fp32)

if jnp.bfloat16 in dtypes_fp32:
    print("  !!! ВНИМАНИЕ: bf16 всё ещё присутствует в fp32-модели !!!")
    print("  source-replace не покрыл все места. Возможно, dtype=bf16 есть")
    print("  в другом файле (configs.py, gdn2_pipeline.py).")
else:
    print("  OK: fp32-модель действительно во fp32")

# --- 4) Прогон ---
T = 1023
x_b = x_a.at[:, T].set((x_a[:, T] + 1) % 256)

@jax.jit
def fwd_fp32(p, ids):
    return model_fp32.apply({"params": p}, ids)

logits_fp32_a = fwd_fp32(params_fp32, x_a)
logits_fp32_b = fwd_fp32(params_fp32, x_b)
jax.block_until_ready((logits_fp32_a, logits_fp32_b))

print(f"\n  fp32 logits dtype: {logits_fp32_a.dtype}")
diff_fp32 = float(jnp.max(jnp.abs(
    jnp.asarray(logits_fp32_a[:, :T], jnp.float32)
    - jnp.asarray(logits_fp32_b[:, :T], jnp.float32))))
print(f"\n  fp32 in-model diff[:T] = {diff_fp32:.3e}")

# --- 5) Тот же тест для bf16 для сравнения ---
@jax.jit
def fwd_bf16(p, ids):
    return model.apply({"params": p}, ids)

logits_bf16_a = fwd_bf16(params, x_a)
logits_bf16_b = fwd_bf16(params, x_b)
jax.block_until_ready((logits_bf16_a, logits_bf16_b))
diff_bf16 = float(jnp.max(jnp.abs(
    jnp.asarray(logits_bf16_a[:, :T], jnp.float32)
    - jnp.asarray(logits_bf16_b[:, :T], jnp.float32))))
print(f"  bf16 in-model diff[:T] = {diff_bf16:.3e}")

# --- 6) Вердикт ---
print("\n--- Интерпретация ---")
print(f"  bf16: {diff_bf16:.3e}")
print(f"  fp32: {diff_fp32:.3e}")

if abs(diff_bf16 - diff_fp32) < 1e-8:
    print("""
  => diff ОДИНАКОВЫЙ. Это невозможно для реального fp32.
     Либо fp32-модель не построилась (см. sanity check выше),
     либо leak идёт из ОДНОГО И ТОГО ЖЕ источника, не зависящего
     от dtype. Скорее первое.
""")
elif diff_fp32 < 1e-4 and diff_bf16 > 1e-3:
    print("""
  => ПОДТВЕРЖДЕНО: leak -- это bf16 amplification.
     В fp32 сигнал падает до round-off уровня. Kernel причинно корректен,
     RMSNorm+Dense+residual в bf16 амплифицируют шум до 1e-2.
""")
elif diff_fp32 > 1e-3:
    print("""
  => fp32 тоже течёт на том же уровне. Это НЕ precision-артефакт.
     Настоящий структурный bug. Нужен пошаговый test F.
""")

=== Sanity check: dtype параметров bf16-модели ===
  dtypes в params (bf16-модель): {dtype('float32')}

=== Sanity check: dtype параметров fp32-модели ===
  dtypes в params (fp32-модель): {dtype('float32')}
  OK: fp32-модель действительно во fp32

  fp32 logits dtype: float32

  fp32 in-model diff[:T] = 8.618e-03
  bf16 in-model diff[:T] = 1.562e-02

--- Интерпретация ---
  bf16: 1.562e-02
  fp32: 8.618e-03

  => fp32 тоже течёт на том же уровне. Это НЕ precision-артефакт.
     Настоящий структурный bug. Нужен пошаговый test F.



In [26]:
"""
gdn2_causal_leak_diagnostics_deep.py

РАСШИРЕНИЕ gdn2_causal_leak_diagnostics.py -- для многочасового TPU-прогона.
Не заменяет первый скрипт, а идёт следующим шагом: если H1/H2/H3/H4 дали
сигнал (или не дали однозначного), здесь его закрепляют статистически и
локализуют физически (в каком слое, на каком расстоянии от границы chunk).

Пять независимых, самодостаточных блоков. Можно запускать по одному.

  D1. PER-LAYER LEAK LOCALIZATION.
      Захватываем промежуточные активации после КАЖДОГО блока (через
      flax sow/capture_intermediates) для x_a и x_b (возмущённый вход) и
      находим МИНИМАЛЬНЫЙ индекс сло
      
      
      
      я, на котором max|Δact[<T]| впервые
      превышает tol. Если утечка появляется резко на layer L (а не растёт
      плавно от layer 0) -- это сильный сигнал, что слой L (точнее, его
      GDN2Mixer) вносит НЕ-round-off эффект, а что-то структурное.

  D2. BOUNDARY-DISTANCE PROFILING.
      Для каждой границы chunk (bt и bc) сканируем T по МЕЛКОЙ сетке
      (каждая позиция, не только -1/0/+1) в окне +-32 вокруг границы, и
      строим профиль diff(distance_to_boundary). Если утечка -- round-off
      от WY-solve на границе, ожидаем резкий пик РОВНО на границе и
      быстрый спад по мере удаления (типичная сигнатура численного шума
      блочного solve). Если утечка "размазана" по всему chunk или растёт
      к концу chunk -- это больше похоже на логическую ошибку в
      cross-chunk state (Kernel D) или в дизайне clip/центрирования.

  D3. MULTI-SEED STATISTICAL ROBUSTNESS.
      Повторяем H2 (bare trainable causal leak) и H4 (centered A/B) на
      N_SEEDS независимых seed для init/inputs, чтобы исключить, что
      наблюдаемое -- артефакт одного "невезучего" seed. Сообщает
      fail-rate по позициям и разброс worst_diff.

  D4. DTYPE / WY_EPS ABLATION GRID.
      Полный grid: dtype in {fp32, bf16} x wy_eps in {0, 1e-3, 1e-2} x
      use_centering in {True, False}, на голом кернеле (H2-style). Цель:
      понять, требует ли утечка ИМЕННО bf16 (типичная численная причина)
      или воспроизводится и в чистом fp32 (тогда это логика, не dtype).

  D5. FULL-RESOLUTION SCAN (statistical safety net).
      На голом кернеле и на модели: скан ВСЕХ позиций T (не только
      границ bt/bc), с грубым шагом (每 8 или 16), чтобы убедиться, что
      утечка географически привязана именно к границам, а не появляется
      где-то ещё, что мы просто не смотрели.

Kaggle TPU v5e-8, notebook-only: без argparse, top-level константы,
main(). Рассчитан на многочасовой прогон -- каждый блок логирует
прогресс и промежуточные результаты в JSON (можно прервать и посмотреть
что накопилось).
"""
from __future__ import annotations

import gc
import json
import os
import sys
import time

import jax
import jax.numpy as jnp
import flax.linen as nn

from Atomic_ops.configs import KernelConfig, KAGGLE_MEDIUM, KAGGLE_MEDIUM_NOCENTER
from Atomic_ops.gdn2_pipeline import gdn2_pallas_forward_trainable

_RESULTS = {}
OUT_DIR = "./gdn2_causal_leak_deep_results"
os.makedirs(OUT_DIR, exist_ok=True)


def _dump(tag):
    path = os.path.join(OUT_DIR, f"{tag}.json")
    with open(path, "w") as f:
        json.dump(_RESULTS, f, indent=2, default=str)
    print(f"    [checkpoint written: {path}]")


def _max_abs_diff(a, b):
    return float(jnp.max(jnp.abs(jnp.asarray(a, jnp.float32) - jnp.asarray(b, jnp.float32))))


def _make_inputs(key, bsz, n_chunks, bt, H, D, decay_scale, dtype=jnp.float32, h0_nonzero=True):
    L = n_chunks * bt
    k1, k2, k3, k4, k5 = jax.random.split(key, 5)
    shape = (bsz, L, H, D)

    q = jax.random.normal(k1, shape)
    k = jax.random.normal(k2, shape)
    q = q / (jnp.linalg.norm(q, axis=-1, keepdims=True) + 1e-6)
    k = k / (jnp.linalg.norm(k, axis=-1, keepdims=True) + 1e-6)
    v = jax.random.normal(k3, shape) * 0.5
    w = jax.random.uniform(k4, shape, minval=0.2, maxval=1.0)
    b = jax.random.uniform(jax.random.fold_in(k4, 1), shape, minval=0.2, maxval=1.0)
    g = -jnp.abs(jax.random.normal(k5, shape)) * decay_scale

    h0 = None
    if h0_nonzero:
        h0 = jax.random.normal(jax.random.fold_in(key, 99), (bsz, H, D, D)) * 0.1

    q, k, v, w, b = (t.astype(dtype) for t in (q, k, v, w, b))
    return q, k, v, w, b, g.astype(jnp.float32), h0


def _boundary_positions(config: KernelConfig, seq_len: int):
    """Все уникальные границы bt/bc (без -1/0/+1 разброса -- используется
    как центр окна в D2)."""
    pts = set()
    for base in range(0, seq_len, config.bt):
        if 0 < base < seq_len:
            pts.add(base)
    for base in range(0, seq_len, config.bc):
        if 0 < base < seq_len:
            pts.add(base)
    return sorted(pts)


# ==========================================================================
# Модель с sow-инструментацией для D1 (per-layer localization)
# ==========================================================================
def _build_instrumented_model(kernel_config, num_layers, d_model=768, n_heads=6):
    d_head = d_model // n_heads
    assert d_head == 128

    def _safe_normalize(t, eps=1e-6):
        return t * jax.lax.rsqrt(jnp.sum(t * t, axis=-1, keepdims=True) + eps ** 2)

    def _sanitize(t):
        return jnp.nan_to_num(jnp.clip(t, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)

    class Mixer(nn.Module):
        @nn.compact
        def __call__(self, x):
            b, l, d = x.shape
            q_lin = nn.Dense(d, use_bias=False, name="q_proj", dtype=jnp.bfloat16)(x)
            k_lin = nn.Dense(d, use_bias=False, name="k_proj", dtype=jnp.bfloat16)(x)
            v_lin = nn.Dense(d, use_bias=False, name="v_proj", dtype=jnp.bfloat16)(x)
            q = jax.nn.silu(q_lin).reshape(b, l, n_heads, d_head).astype(jnp.float32)
            k = jax.nn.silu(k_lin).reshape(b, l, n_heads, d_head).astype(jnp.float32)
            v = jax.nn.silu(v_lin).reshape(b, l, n_heads, d_head).astype(jnp.float32)
            v = jnp.clip(v, -50.0, 50.0)
            q = _safe_normalize(q)
            k = _safe_normalize(k)

            b_gate = jax.nn.sigmoid(nn.Dense(d, name="erase_gate", dtype=jnp.bfloat16)(x)) \
                .reshape(b, l, n_heads, d_head).astype(jnp.float32)
            w_gate = jax.nn.sigmoid(nn.Dense(d, name="write_gate", dtype=jnp.bfloat16)(x)) \
                .reshape(b, l, n_heads, d_head).astype(jnp.float32)

            a_param = self.param("decay_a", nn.initializers.constant(-4.0), (n_heads,)).astype(jnp.float32)
            f_proj = nn.Dense(d, name="decay_proj", dtype=jnp.bfloat16)(x).reshape(b, l, n_heads, d_head)
            a_safe = jnp.clip(a_param, -20.0, 20.0)
            g = -jnp.exp(a_safe)[None, None, :, None] * jax.nn.softplus(f_proj.astype(jnp.float32))
            g = jnp.nan_to_num(g, nan=0.0, posinf=0.0, neginf=-20.0)

            out_gate = jnp.clip(nn.Dense(d, use_bias=False, name="out_gate", dtype=jnp.bfloat16)(x), -1e2, 1e2)

            q, k, v, w_gate, b_gate, g = map(_sanitize, (q, k, v, w_gate, b_gate, g))
            out, _hf = gdn2_pallas_forward_trainable(
                q, k, v, w_gate, b_gate, g, scale=1.0, config=kernel_config
            )
            out = out.reshape(b, l, d)
            out = nn.RMSNorm(epsilon=1e-6, name="mixer_out_norm")(out).astype(x.dtype)
            return nn.Dense(d, use_bias=False, name="out_proj", dtype=jnp.bfloat16)(out * jax.nn.silu(out_gate))

    class MLP(nn.Module):
        @nn.compact
        def __call__(self, x):
            d = x.shape[-1]
            h = 4 * d
            gate = nn.Dense(h, use_bias=False, name="gate_proj", dtype=jnp.bfloat16)(x)
            up = nn.Dense(h, use_bias=False, name="up_proj", dtype=jnp.bfloat16)(x)
            act = jax.nn.silu(gate) * up
            return nn.Dense(d, use_bias=False, name="down_proj", dtype=jnp.bfloat16)(act)

    class Block(nn.Module):
        layer_idx: int

        @nn.compact
        def __call__(self, x):
            h = Mixer(name="mixer")(nn.RMSNorm(epsilon=1e-6, name="mixer_norm")(x))
            x = jnp.nan_to_num(jnp.clip(x + h, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)
            m = MLP(name="mlp")(nn.RMSNorm(epsilon=1e-6, name="mlp_norm")(x))
            x = jnp.nan_to_num(jnp.clip(x + m, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)
            # sow -- captured via capture_intermediates, keyed by module path
            self.sow("intermediates", f"block_out_{self.layer_idx}", x)
            return x

    class InstrumentedLM(nn.Module):
        @nn.compact
        def __call__(self, input_ids):
            embed = nn.Embed(num_embeddings=256, features=d_model, name="embed", dtype=jnp.bfloat16)
            x = embed(input_ids)
            for i in range(num_layers):
                x = Block(layer_idx=i, name=f"block_{i}")(x)
            x = nn.RMSNorm(epsilon=1e-6, name="final_norm")(x).astype(x.dtype)
            return embed.attend(x)

    return InstrumentedLM()


# ==========================================================================
# D1: per-layer leak localization
# ==========================================================================
def test_d1_per_layer_localization(cfg):
    print("\n" + "=" * 78)
    print("D1: per-layer leak localization via intermediate capture")
    print("=" * 78)

    config = KAGGLE_MEDIUM
    L = cfg["seq_len"]
    B = cfg["bsz"]
    num_layers = cfg["d1_num_layers"]
    T_probe = cfg["d1_probe_T"]  # one boundary position to trace through depth

    model = _build_instrumented_model(config, num_layers=num_layers)
    init_rng = jax.random.PRNGKey(12345)
    dummy = jnp.zeros((B, L), dtype=jnp.int32)
    params = model.init(init_rng, dummy)["params"]

    x_a = jax.random.randint(jax.random.PRNGKey(777), (B, L), 0, 256, dtype=jnp.int32)
    x_b = x_a.at[:, T_probe].set((x_a[:, T_probe] + 1) % 256)

    @jax.jit
    def forward_with_intermediates(p, ids):
        _logits, mutated = model.apply({"params": p}, ids, mutable=["intermediates"])
        return mutated["intermediates"]

    inter_a = forward_with_intermediates(params, x_a)
    inter_b = forward_with_intermediates(params, x_b)
    jax.block_until_ready((inter_a, inter_b))

    # --- DEBUG: посмотреть, какие ключи реально вернул flax ---
    print("  captured intermediate keys:")
    for k in sorted(inter_a.keys()):
        print(f"    {k!r}")
    def _find_sow_key(d, layer_idx):
        """flax capture_intermediates ключует по имени модуля: 'block_N'.
        На случай других версий/конвенций — проверим несколько вариантов."""
        for cand in (
            f"block_{layer_idx}",          # <-- это и есть реальный ключ
            f"block_out_{layer_idx}",
            f"block_{layer_idx}_block_out_{layer_idx}",
            f"block_{layer_idx}/block_out_{layer_idx}",
        ):
            if cand in d:
                return cand
        for k in d:
            if f"block_out_{layer_idx}" in k:
                return k
        return None
    def _unwrap_to_array(v):
        """Продраться сквозь dict/list/tuple пока не упрёмся в jax array."""
        # максимум 6 уровней вложенности — этого хватит с запасом
        for _ in range(6):
            if isinstance(v, dict):
                if "intermediates" in v:
                    v = v["intermediates"]
                else:
                    v = next(iter(v.values()))
                continue
            if isinstance(v, (list, tuple)):
                v = v[0]
                continue
            break
        return v

    print(f"  probing T={T_probe} through {num_layers} layers (tol={cfg['causal_tol']:.1e})")
    first_leak_layer = None
    per_layer = {}
    for i in range(num_layers):
        key = _find_sow_key(inter_a, i)
        if key is None:
            print(f"    layer {i:2d}: KEY NOT FOUND")
            continue
        act_a = _unwrap_to_array(inter_a[key])
        act_b = _unwrap_to_array(inter_b[key])
        # отладочный принт один раз — убедиться, что достали массив
        if i == 0:
            print(f"    [debug] layer 0 unwrapped shape: {act_a.shape}, dtype={act_a.dtype}")
        diff = _max_abs_diff(act_a[:, :T_probe], act_b[:, :T_probe]) if T_probe > 0 else 0.0
        per_layer[i] = diff
        flag = ""
        if diff >= cfg["causal_tol"] and first_leak_layer is None:
            first_leak_layer = i
            flag = "  <-- FIRST LAYER ABOVE TOL"
        print(f"    layer {i:2d}: max|Δact[<{T_probe}]| = {diff:.3e}{flag}")

    _RESULTS["d1.T_probe"] = T_probe
    _RESULTS["d1.num_layers"] = num_layers
    _RESULTS["d1.per_layer_diff"] = per_layer
    _RESULTS["d1.first_leak_layer"] = first_leak_layer

    if first_leak_layer is None:
        print("  => No layer crossed tol at this T -- leak (if any) is below causal_tol "
              "through this depth; try a larger num_layers or a different T_probe.")
    elif first_leak_layer == 0:
        print("  => Leak present already at layer 0 -- NOT a depth-accumulation effect; "
              "points directly at GDN2Mixer/kernel behavior at this boundary.")
    else:
        growth = per_layer[first_leak_layer] / max(per_layer.get(first_leak_layer - 1, 1e-12), 1e-12)
        print(f"  => Leak first crosses tol at layer {first_leak_layer} "
              f"(jump factor vs previous layer: {growth:.2f}x). Consistent with progressive "
              "amplification through RMSNorm/normalize stack if growth factor is roughly "
              "constant per layer.")

    del model, params, inter_a, inter_b
    gc.collect()
    jax.clear_caches()
    _dump("d1_per_layer_localization")


# ==========================================================================
# D2: boundary-distance profiling (bare kernel, fine-grained)
# ==========================================================================
def test_d2_boundary_distance_profile(cfg):
    print("\n" + "=" * 78)
    print("D2: fine-grained boundary-distance leak profile (bare trainable kernel)")
    print("=" * 78)

    for label, config in (("centered", KAGGLE_MEDIUM), ("nocenter", KAGGLE_MEDIUM_NOCENTER)):
        bsz, H, D = cfg["bsz"], cfg["H"], cfg["D"]
        L = cfg["seq_len"]
        n_chunks = L // config.bt
        key = jax.random.PRNGKey(4242)
        q, k, v, w, b, g, h0 = _make_inputs(key, bsz, n_chunks, config.bt, H, D, decay_scale=0.1)

        @jax.jit
        def fwd(q_, k_, v_, w_, b_, g_):
            o, _hf = gdn2_pallas_forward_trainable(q_, k_, v_, w_, b_, g_, scale=1.0, h0=h0, config=config)
            return o

        o_a = fwd(q, k, v, w, b, g)
        jax.block_until_ready(o_a)

        boundaries = _boundary_positions(config, L)
        window = cfg["d2_window"]
        profile = {}  # boundary -> {offset: diff}

        for boundary in boundaries:
            offsets = [off for off in range(-window, window + 1)
                       if 0 < boundary + off < L]
            diffs_by_offset = {}
            for off in offsets:
                T = boundary + off
                q_b = q.at[:, T].add(0.3)
                k_b = k.at[:, T].add(0.3)
                v_b = v.at[:, T].add(0.3)
                w_b = jnp.clip(w.at[:, T].add(0.1), 0.01, 1.0)
                b_b = jnp.clip(b.at[:, T].add(0.1), 0.01, 1.0)
                g_b = g.at[:, T].add(-0.05)
                o_b = fwd(q_b, k_b, v_b, w_b, b_b, g_b)
                jax.block_until_ready(o_b)
                diff = _max_abs_diff(o_a[:, :T], o_b[:, :T]) if T > 0 else 0.0
                diffs_by_offset[off] = diff
            profile[boundary] = diffs_by_offset

            peak_off = max(diffs_by_offset, key=diffs_by_offset.get)
            peak_val = diffs_by_offset[peak_off]
            edge_val_pos = diffs_by_offset.get(window, 0.0)
            edge_val_neg = diffs_by_offset.get(-window, 0.0)
            decays = peak_val > 0 and max(edge_val_pos, edge_val_neg) < peak_val * 0.3
            print(f"  {label} boundary={boundary:4d}: peak@offset={peak_off:+d} "
                  f"diff={peak_val:.3e}  edges(+-{window})=({edge_val_neg:.2e},{edge_val_pos:.2e})  "
                  f"{'localized/decaying' if decays else 'NOT clearly localized -- check manually'}")

        _RESULTS[f"d2.{label}.profile"] = profile
        _dump("d2_boundary_distance_profile")


# ==========================================================================
# D3: multi-seed statistical robustness
# ==========================================================================
def test_d3_multiseed_robustness(cfg):
    print("\n" + "=" * 78)
    print(f"D3: multi-seed robustness ({cfg['d3_n_seeds']} seeds), bare trainable kernel")
    print("=" * 78)

    for label, config in (("centered", KAGGLE_MEDIUM), ("nocenter", KAGGLE_MEDIUM_NOCENTER)):
        bsz, H, D = cfg["bsz"], cfg["H"], cfg["D"]
        L = cfg["seq_len"]
        n_chunks = L // config.bt

        @jax.jit
        def fwd(q_, k_, v_, w_, b_, g_, h0_):
            o, _hf = gdn2_pallas_forward_trainable(q_, k_, v_, w_, b_, g_, scale=1.0, h0=h0_, config=config)
            return o

        boundaries = _boundary_positions(config, L)
        per_seed_worst = []
        per_seed_failcount = []

        for seed in range(cfg["d3_n_seeds"]):
            key = jax.random.PRNGKey(9000 + seed)
            q, k, v, w, b, g, h0 = _make_inputs(key, bsz, n_chunks, config.bt, H, D, decay_scale=0.1)
            o_a = fwd(q, k, v, w, b, g, h0)
            jax.block_until_ready(o_a)

            worst = 0.0
            n_fail = 0
            for T in boundaries:
                q_b = q.at[:, T].add(0.3)
                k_b = k.at[:, T].add(0.3)
                v_b = v.at[:, T].add(0.3)
                w_b = jnp.clip(w.at[:, T].add(0.1), 0.01, 1.0)
                b_b = jnp.clip(b.at[:, T].add(0.1), 0.01, 1.0)
                g_b = g.at[:, T].add(-0.05)
                o_b = fwd(q_b, k_b, v_b, w_b, b_b, g_b, h0)
                jax.block_until_ready(o_b)
                diff = _max_abs_diff(o_a[:, :T], o_b[:, :T])
                worst = max(worst, diff)
                if diff >= cfg["causal_tol"]:
                    n_fail += 1

            per_seed_worst.append(worst)
            per_seed_failcount.append(n_fail)
            print(f"  {label} seed={seed}: worst_diff={worst:.3e}  n_failed={n_fail}/{len(boundaries)}")

        _RESULTS[f"d3.{label}.per_seed_worst"] = per_seed_worst
        _RESULTS[f"d3.{label}.per_seed_failcount"] = per_seed_failcount
        _RESULTS[f"d3.{label}.mean_worst"] = float(sum(per_seed_worst) / len(per_seed_worst))
        _RESULTS[f"d3.{label}.max_worst"] = float(max(per_seed_worst))
        _RESULTS[f"d3.{label}.min_worst"] = float(min(per_seed_worst))
        print(f"  {label} summary: mean={_RESULTS[f'd3.{label}.mean_worst']:.3e} "
              f"min={_RESULTS[f'd3.{label}.min_worst']:.3e} "
              f"max={_RESULTS[f'd3.{label}.max_worst']:.3e}")
        _dump("d3_multiseed_robustness")


# ==========================================================================
# D4: dtype / wy_eps / centering ablation grid (bare kernel)
# ==========================================================================
def test_d4_ablation_grid(cfg):
    print("\n" + "=" * 78)
    print("D4: dtype x wy_eps x use_centering ablation grid (bare trainable kernel)")
    print("=" * 78)

    bsz, H, D = cfg["bsz"], cfg["H"], cfg["D"]
    L = cfg["seq_len"]
    bt, bc, mb = 256, 128, 16

    grid = []
    for dtype_label, dtype in (("fp32", jnp.float32), ("bf16", jnp.bfloat16)):
        for wy_eps in (0.0, 1e-3, 1e-2):
            for use_centering in (True, False):
                grid.append((dtype_label, dtype, wy_eps, use_centering))

    results = {}
    for dtype_label, dtype, wy_eps, use_centering in grid:
        config = KernelConfig(bt=bt, bc=bc, mb=mb, wy_eps=wy_eps, use_centering=use_centering)
        n_chunks = L // bt
        key = jax.random.PRNGKey(5555)
        q, k, v, w, b, g, h0 = _make_inputs(key, bsz, n_chunks, bt, H, D, decay_scale=0.1, dtype=dtype)

        @jax.jit
        def fwd(q_, k_, v_, w_, b_, g_):
            o, _hf = gdn2_pallas_forward_trainable(q_, k_, v_, w_, b_, g_, scale=1.0, h0=h0, config=config)
            return o

        o_a = fwd(q, k, v, w, b, g)
        jax.block_until_ready(o_a)

        boundaries = _boundary_positions(config, L)
        worst = 0.0
        n_fail = 0
        for T in boundaries:
            q_b = q.at[:, T].add(jnp.asarray(0.3, dtype))
            k_b = k.at[:, T].add(jnp.asarray(0.3, dtype))
            v_b = v.at[:, T].add(jnp.asarray(0.3, dtype))
            w_b = jnp.clip(w.at[:, T].add(jnp.asarray(0.1, dtype)), 0.01, 1.0)
            b_b = jnp.clip(b.at[:, T].add(jnp.asarray(0.1, dtype)), 0.01, 1.0)
            g_b = g.at[:, T].add(-0.05)
            o_b = fwd(q_b, k_b, v_b, w_b, b_b, g_b)
            jax.block_until_ready(o_b)
            diff = _max_abs_diff(o_a[:, :T].astype(jnp.float32), o_b[:, :T].astype(jnp.float32))
            worst = max(worst, diff)
            if diff >= cfg["causal_tol"]:
                n_fail += 1

        tag = f"{dtype_label}/wy_eps={wy_eps}/centering={use_centering}"
        results[tag] = dict(worst_diff=worst, n_failed=n_fail, n_total=len(boundaries))
        print(f"  {tag:45s} worst_diff={worst:.3e}  n_failed={n_fail}/{len(boundaries)}")

    _RESULTS["d4.grid"] = results
    _dump("d4_ablation_grid")

    print("\n  Interpretation:")
    fp32_leaks = any(r["n_failed"] > 0 for k, r in results.items() if k.startswith("fp32"))
    bf16_leaks = any(r["n_failed"] > 0 for k, r in results.items() if k.startswith("bf16"))
    if fp32_leaks:
        print("  => Leak reproduces even in PURE fp32 -- this is a LOGIC issue, not a bf16")
        print("     precision artifact. Focus on kernel math (centering/clip/boundary terms),")
        print("     not on dtype/precision tuning.")
    elif bf16_leaks and not fp32_leaks:
        print("  => Leak ONLY appears with bf16 inputs -- consistent with a precision/rounding")
        print("     sensitivity rather than a structural causality bug. Consider tightening")
        print("     Gate-2 tolerance for bf16 configs, or adding fp32 accumulation at the")
        print("     specific boundary op identified in D2.")
    else:
        print("  => No leak reproduced in this grid at all -- if D1-D3 showed a leak, it may")
        print("     require the full model stack (RMSNorm chain) to manifest; not a bare-kernel")
        print("     dtype/wy_eps/centering effect in isolation.")


# ==========================================================================
# D5: full-resolution scan (statistical safety net -- coarse stride over
#     ALL positions, not just known boundaries)
# ==========================================================================
def test_d5_full_resolution_scan(cfg):
    print("\n" + "=" * 78)
    print(f"D5: full-resolution scan, stride={cfg['d5_stride']} (bare trainable kernel)")
    print("=" * 78)

    for label, config in (("centered", KAGGLE_MEDIUM), ("nocenter", KAGGLE_MEDIUM_NOCENTER)):
        bsz, H, D = cfg["bsz"], cfg["H"], cfg["D"]
        L = cfg["seq_len"]
        n_chunks = L // config.bt
        key = jax.random.PRNGKey(6161)
        q, k, v, w, b, g, h0 = _make_inputs(key, bsz, n_chunks, config.bt, H, D, decay_scale=0.1)

        @jax.jit
        def fwd(q_, k_, v_, w_, b_, g_):
            o, _hf = gdn2_pallas_forward_trainable(q_, k_, v_, w_, b_, g_, scale=1.0, h0=h0, config=config)
            return o

        o_a = fwd(q, k, v, w, b, g)
        jax.block_until_ready(o_a)

        stride = cfg["d5_stride"]
        all_positions = list(range(stride, L, stride))
        known_boundaries = set(_boundary_positions(config, L))

        unexpected_leaks = []
        for T in all_positions:
            q_b = q.at[:, T].add(0.3)
            k_b = k.at[:, T].add(0.3)
            v_b = v.at[:, T].add(0.3)
            w_b = jnp.clip(w.at[:, T].add(0.1), 0.01, 1.0)
            b_b = jnp.clip(b.at[:, T].add(0.1), 0.01, 1.0)
            g_b = g.at[:, T].add(-0.05)
            o_b = fwd(q_b, k_b, v_b, w_b, b_b, g_b)
            jax.block_until_ready(o_b)
            diff = _max_abs_diff(o_a[:, :T], o_b[:, :T])
            near_known_boundary = any(abs(T - kb) <= 2 for kb in known_boundaries)
            if diff >= cfg["causal_tol"] and not near_known_boundary:
                unexpected_leaks.append((T, diff))
                print(f"    [UNEXPECTED] {label} T={T}: diff={diff:.3e} "
                      f"(NOT near a known bt/bc boundary!)")

        _RESULTS[f"d5.{label}.n_scanned"] = len(all_positions)
        _RESULTS[f"d5.{label}.n_unexpected_leaks"] = len(unexpected_leaks)
        _RESULTS[f"d5.{label}.unexpected_leaks"] = unexpected_leaks
        print(f"  {label}: scanned {len(all_positions)} positions (stride={stride}), "
              f"{len(unexpected_leaks)} unexpected leaks away from known boundaries.")
        _dump("d5_full_resolution_scan")


# ==========================================================================
# Entrypoint
# ==========================================================================
RUN_CONFIG = dict(
    bsz=2,
    H=6,
    D=128,
    seq_len=1024,
    causal_tol=1e-5,
    d1_num_layers=13,
    d1_probe_T=127,          # a known leaking boundary from Gate 2 logs
    d2_window=32,
    d3_n_seeds=8,
    d5_stride=8,
)


def main(cfg=RUN_CONFIG):
    print(f"JAX version: {jax.__version__}")
    print(f"Devices: {jax.devices()}")
    t0 = time.time()

    test_d1_per_layer_localization(cfg)
    test_d2_boundary_distance_profile(cfg)
    test_d3_multiseed_robustness(cfg)
    test_d4_ablation_grid(cfg)
    test_d5_full_resolution_scan(cfg)

    elapsed = time.time() - t0
    _dump("FINAL_all_results")
    print("\n" + "=" * 78)
    print(f"DEEP DIAGNOSTICS COMPLETE. Elapsed: {elapsed/3600:.2f}h ({elapsed:.0f}s)")
    print(f"All results in {OUT_DIR}/")
    print("=" * 78)


if __name__ == "__main__":
    main()

JAX version: 0.11.1
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]

D1: per-layer leak localization via intermediate capture
  captured intermediate keys:
    'block_0'
    'block_1'
    'block_10'
    'block_11'
    'block_12'
    'block_2'
    'block_3'
    'block_4'
    'block_5'
    'block_6'
    'block_7'
    'block_8'
    'block_9'
  probing T=127 through 13 layers (tol=1.0e-05)
    [debug] layer 0 unwrapped shape: (2, 1024, 768), dtype=bfloat16
    layer  0: max|Δact[<127]| = 1.562e-02  <-- FIRST 

In [10]:
"""
gdn2_causal_leak_bisection_real_values.py

Решающий bisection-тест, продолжающий gdn2_causal_leak_diagnostics.py и
gdn2_causal_leak_diagnostics_deep.py.

К этому моменту установлено (H1-H4, D1-D5):
  - Голый gdn2_pallas_forward_trainable в ИЗОЛИРОВАННОМ jit НЕ течёт --
    ни на одном seed, ни в одном dtype (fp32/bf16), ни при одном wy_eps,
    ни при use_centering=True/False (H2, D2, D3, D4, D5 -- все чисто).
  - Тот же кернель, встроенный в полную 13-слойную модель, ТЕЧЁТ уже на
    layer 0, СТРОГО СПЕЦИФИЧНО к use_centering=True (H4: centered
    14/24 fail worst=8.8e-2; nocenter 0/24, worst РОВНО 0.0 -- то есть
    nocenter даёт bit-exact причинность даже через 8 bf16-слоёв, значит
    это НЕ фоновый bf16-шум, а настоящая утечка информации из будущего).

Раз голый кернель чист при ЛЮБЫХ синтетических условиях, а встроенный в
модель -- течёт, остаются два объяснения:

  (A) КОНТЕКСТ КОМПИЛЯЦИИ. pallas_call внутри большого XLA-графа модели
      (embed -> 6 Dense-слоёв на mixer -> RMSNorm -> sigmoid/softplus ->
      sanitize -> kernel -> RMSNorm -> Dense -> MLP -> ...) может
      получить другой layout/buffer-aliasing/fusion вокруг BlockSpec-grid,
      чем тот же pallas_call как единственная операция в jit. Особенно
      уязвимы срезы gc[i0]/gc[j1-1] (per-pair local centering) -- это
      статически известные, но "похожие на dynamic" scalar-slice внутри
      кернела.

  (B) ЗНАЧЕНИЯ ВХОДОВ. Модель порождает decay g = -exp(-4.0)*softplus(f)
      величиной ~-0.018 при инициализации -- НА ПОРЯДОК меньше, чем
      decay_scale=0.1, которым был зафиксирован ВЕСЬ grid в D4. D4 закрыл
      dtype x wy_eps x centering, но НЕ закрыл magnitude самого decay --
      это дыра в покрытии, а не доказанное отсутствие эффекта.

Этот скрипт разделяет (A) и (B) напрямую:

  BS1. REPLAY В ИЗОЛИРОВАННОМ JIT. Перехватываем (через self.sow, ПОСЛЕ
       _sanitize, ПРЯМО ПЕРЕД вызовом кернела) реальные q/k/v/w/b/g,
       которые модель фактически вычисляет на layer 0 -- отдельно для
       невозмущённого x_a и возмущённого в позиции T x_b. Кормим эти
       РЕАЛЬНЫЕ тензоры в НОВЫЙ, полностью изолированный @jax.jit,
       вызывающий gdn2_pallas_forward_trainable напрямую (структура
       идентична H2). Сравниваем:
         diff_inmodel   = |kernel_out_a[<T] - kernel_out_b[<T]|  (внутри модели)
         diff_isolated  = |bare_out_a[<T]   - bare_out_b[<T]|    (изолированный replay)
       diff_isolated ~ 0, diff_inmodel > tol  => (A) контекст компиляции.
       diff_isolated ~ diff_inmodel           => (B) значения/математика.
       Дополнительно проверяем input_diff_before_T -- что сами
       q/k/v/w/b/g ДЕЙСТВИТЕЛЬНО совпадают до позиции T (иначе утечка
       уже ДО кернела, в Dense/embed -- совсем другой баг).

  BS2. То же самое, но БЕЗ внешнего @jax.jit (eager per-op dispatch) --
       ещё более резкое сужение компиляционного контекста, чем "один
       jit из одной операции" в BS1.

  BS3. DECAY-SCALE SWEEP на голом кернеле (H2-style), закрывает дыру
       покрытия D4: decay_scale in {0.1, 0.05, 0.018, 0.005, 0.001}
       (0.018 -- реалистичное значение из модели при init), centered vs
       nocenter. Если утечка появляется на голом кернеле при малом
       decay_scale -- гипотеза (B) подтверждена НАПРЯМУЮ, без всякого
       replay.

Kaggle TPU v5e-8, notebook-only: без argparse, top-level константы,
main(). BS1/BS2 быстрые (секунды-минуты); BS3 по времени сопоставим с D4.
"""
from __future__ import annotations

import gc
import json
import os
import time

import jax
import jax.numpy as jnp
import flax.linen as nn

from Atomic_ops.configs import KernelConfig, KAGGLE_MEDIUM, KAGGLE_MEDIUM_NOCENTER
from Atomic_ops.gdn2_pipeline import gdn2_pallas_forward_trainable

_RESULTS = {}
OUT_DIR = "./gdn2_causal_leak_bisection_results"
os.makedirs(OUT_DIR, exist_ok=True)


def _dump(tag):
    path = os.path.join(OUT_DIR, f"{tag}.json")
    with open(path, "w") as f:
        json.dump(_RESULTS, f, indent=2, default=str)
    print(f"    [checkpoint written: {path}]")


def _max_abs_diff(a, b):
    return float(jnp.max(jnp.abs(jnp.asarray(a, jnp.float32) - jnp.asarray(b, jnp.float32))))


def _boundary_positions(config: KernelConfig, seq_len: int):
    pts = set()
    for base in range(0, seq_len, config.bt):
        if 0 < base < seq_len:
            pts.add(base)
    for base in range(0, seq_len, config.bc):
        if 0 < base < seq_len:
            pts.add(base)
    return sorted(pts)


def _make_bare_inputs(key, bsz, n_chunks, bt, H, D, decay_scale, h0_nonzero=True):
    L = n_chunks * bt
    k1, k2, k3, k4, k5 = jax.random.split(key, 5)
    shape = (bsz, L, H, D)

    q = jax.random.normal(k1, shape)
    k = jax.random.normal(k2, shape)
    q = q / (jnp.linalg.norm(q, axis=-1, keepdims=True) + 1e-6)
    k = k / (jnp.linalg.norm(k, axis=-1, keepdims=True) + 1e-6)
    v = jax.random.normal(k3, shape) * 0.5
    w = jax.random.uniform(k4, shape, minval=0.2, maxval=1.0)
    b = jax.random.uniform(jax.random.fold_in(k4, 1), shape, minval=0.2, maxval=1.0)
    g = -jnp.abs(jax.random.normal(k5, shape)) * decay_scale

    h0 = None
    if h0_nonzero:
        h0 = jax.random.normal(jax.random.fold_in(key, 99), (bsz, H, D, D)) * 0.1

    return q, k, v, w, b, g.astype(jnp.float32), h0


# ==========================================================================
# Модель с перехватом РЕАЛЬНЫХ входов/выхода кернела на layer 0
# ==========================================================================
def _build_bisection_model(kernel_config, num_layers, d_model=768, n_heads=6):
    d_head = d_model // n_heads
    assert d_head == 128

    def _safe_normalize(t, eps=1e-6):
        return t * jax.lax.rsqrt(jnp.sum(t * t, axis=-1, keepdims=True) + eps ** 2)

    def _sanitize(t):
        return jnp.nan_to_num(jnp.clip(t, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)

    class Mixer(nn.Module):
        layer_idx: int

        @nn.compact
        def __call__(self, x):
            b, l, d = x.shape
            q_lin = nn.Dense(d, use_bias=False, name="q_proj", dtype=jnp.bfloat16)(x)
            k_lin = nn.Dense(d, use_bias=False, name="k_proj", dtype=jnp.bfloat16)(x)
            v_lin = nn.Dense(d, use_bias=False, name="v_proj", dtype=jnp.bfloat16)(x)
            q = jax.nn.silu(q_lin).reshape(b, l, n_heads, d_head).astype(jnp.float32)
            k = jax.nn.silu(k_lin).reshape(b, l, n_heads, d_head).astype(jnp.float32)
            v = jax.nn.silu(v_lin).reshape(b, l, n_heads, d_head).astype(jnp.float32)
            v = jnp.clip(v, -50.0, 50.0)
            q = _safe_normalize(q)
            k = _safe_normalize(k)

            b_gate = jax.nn.sigmoid(nn.Dense(d, name="erase_gate", dtype=jnp.bfloat16)(x)) \
                .reshape(b, l, n_heads, d_head).astype(jnp.float32)
            w_gate = jax.nn.sigmoid(nn.Dense(d, name="write_gate", dtype=jnp.bfloat16)(x)) \
                .reshape(b, l, n_heads, d_head).astype(jnp.float32)

            a_param = self.param("decay_a", nn.initializers.constant(-4.0), (n_heads,)).astype(jnp.float32)
            f_proj = nn.Dense(d, name="decay_proj", dtype=jnp.bfloat16)(x).reshape(b, l, n_heads, d_head)
            a_safe = jnp.clip(a_param, -20.0, 20.0)
            g = -jnp.exp(a_safe)[None, None, :, None] * jax.nn.softplus(f_proj.astype(jnp.float32))
            g = jnp.nan_to_num(g, nan=0.0, posinf=0.0, neginf=-20.0)

            out_gate = jnp.clip(nn.Dense(d, use_bias=False, name="out_gate", dtype=jnp.bfloat16)(x), -1e2, 1e2)

            q, k, v, w_gate, b_gate, g = map(_sanitize, (q, k, v, w_gate, b_gate, g))

            # --- ПЕРЕХВАТ: ровно те тензоры, что реально уходят в кернел ---
            if self.layer_idx == 0:
                self.sow("intermediates", "kin_q_0", q)
                self.sow("intermediates", "kin_k_0", k)
                self.sow("intermediates", "kin_v_0", v)
                self.sow("intermediates", "kin_w_0", w_gate)
                self.sow("intermediates", "kin_b_0", b_gate)
                self.sow("intermediates", "kin_g_0", g)

            out, _hf = gdn2_pallas_forward_trainable(
                q, k, v, w_gate, b_gate, g, scale=1.0, config=kernel_config
            )

            if self.layer_idx == 0:
                self.sow("intermediates", "kout_raw_0", out)

            out = out.reshape(b, l, d)
            out = nn.RMSNorm(epsilon=1e-6, name="mixer_out_norm")(out).astype(x.dtype)
            return nn.Dense(d, use_bias=False, name="out_proj", dtype=jnp.bfloat16)(out * jax.nn.silu(out_gate))

    class MLP(nn.Module):
        @nn.compact
        def __call__(self, x):
            d = x.shape[-1]
            h = 4 * d
            gate = nn.Dense(h, use_bias=False, name="gate_proj", dtype=jnp.bfloat16)(x)
            up = nn.Dense(h, use_bias=False, name="up_proj", dtype=jnp.bfloat16)(x)
            act = jax.nn.silu(gate) * up
            return nn.Dense(d, use_bias=False, name="down_proj", dtype=jnp.bfloat16)(act)

    class Block(nn.Module):
        layer_idx: int

        @nn.compact
        def __call__(self, x):
            h = Mixer(layer_idx=self.layer_idx, name="mixer")(nn.RMSNorm(epsilon=1e-6, name="mixer_norm")(x))
            x = jnp.nan_to_num(jnp.clip(x + h, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)
            m = MLP(name="mlp")(nn.RMSNorm(epsilon=1e-6, name="mlp_norm")(x))
            x = jnp.nan_to_num(jnp.clip(x + m, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)
            return x

    class BisectionLM(nn.Module):
        @nn.compact
        def __call__(self, input_ids):
            embed = nn.Embed(num_embeddings=256, features=d_model, name="embed", dtype=jnp.bfloat16)
            x = embed(input_ids)
            for i in range(num_layers):
                x = Block(layer_idx=i, name=f"block_{i}")(x)
            x = nn.RMSNorm(epsilon=1e-6, name="final_norm")(x).astype(x.dtype)
            return embed.attend(x)

    return BisectionLM()


def _find_and_unwrap(intermediates, target_key):
    """Рекурсивно ищет target_key во вложенном дереве mutated['intermediates']
    (flax кладёт sow-значения под путём модуля -- глубина вложенности
    зависит от того, сколько submodule-скоупов между корнем и self.sow)
    и распаковывает tuple/list-обёртку (sow копит в tuple длиной 1 за
    один apply-вызов)."""
    def _walk(node):
        if isinstance(node, dict):
            if target_key in node:
                return node[target_key]
            for v in node.values():
                found = _walk(v)
                if found is not None:
                    return found
        return None

    found = _walk(intermediates)
    if found is None:
        raise KeyError(f"sow key {target_key!r} не найден в дереве intermediates "
                        f"(top-level keys: {list(intermediates.keys())})")
    while isinstance(found, (list, tuple)):
        found = found[0]
    return found


def _extract_kernel_io(inter):
    names = ("q", "k", "v", "w", "b", "g")
    vals = tuple(_find_and_unwrap(inter, f"kin_{n}_0") for n in names)
    kout = _find_and_unwrap(inter, "kout_raw_0")
    return vals, kout


# ==========================================================================
# BS1: replay реальных значений в ИЗОЛИРОВАННОМ jit
# ==========================================================================
def test_bs1_replay_isolated_jit(cfg):
    print("\n" + "=" * 78)
    print("BS1: replay реальных layer-0 входов кернела в ИЗОЛИРОВАННОМ @jax.jit")
    print("=" * 78)

    config = KAGGLE_MEDIUM
    L, B = cfg["seq_len"], cfg["bsz"]
    num_layers = cfg["bisect_num_layers"]
    T_list = [cfg["bisect_T"], *cfg["extra_T_probe"]]

    model = _build_bisection_model(config, num_layers=num_layers)
    init_rng = jax.random.PRNGKey(12345)
    dummy = jnp.zeros((B, L), dtype=jnp.int32)
    params = model.init(init_rng, dummy)["params"]

    @jax.jit
    def forward_with_intermediates(p, ids):
        _logits, mutated = model.apply({"params": p}, ids, mutable=["intermediates"])
        return mutated["intermediates"]

    @jax.jit
    def bare_fwd(q_, k_, v_, w_, b_, g_):
        o, _hf = gdn2_pallas_forward_trainable(q_, k_, v_, w_, b_, g_, scale=1.0, config=config)
        return o

    x_a = jax.random.randint(jax.random.PRNGKey(777), (B, L), 0, 256, dtype=jnp.int32)
    inter_a = forward_with_intermediates(params, x_a)
    jax.block_until_ready(inter_a)
    print(f"  [debug] top-level intermediates keys: {sorted(inter_a.keys())}")
    (q_a, k_a, v_a, w_a, b_a, g_a), kout_a = _extract_kernel_io(inter_a)

    results = {}
    for T in T_list:
        x_b = x_a.at[:, T].set((x_a[:, T] + 1) % 256)
        inter_b = forward_with_intermediates(params, x_b)
        jax.block_until_ready(inter_b)
        (q_b, k_b, v_b, w_b, b_b, g_b), kout_b = _extract_kernel_io(inter_b)

        input_diff_before_T = max(
            _max_abs_diff(pa[:, :T], pb[:, :T])
            for pa, pb in zip((q_a, k_a, v_a, w_a, b_a, g_a), (q_b, k_b, v_b, w_b, b_b, g_b))
        ) if T > 0 else 0.0

        diff_inmodel = _max_abs_diff(kout_a[:, :T], kout_b[:, :T]) if T > 0 else 0.0

        o_bare_a = bare_fwd(q_a, k_a, v_a, w_a, b_a, g_a)
        o_bare_b = bare_fwd(q_b, k_b, v_b, w_b, b_b, g_b)
        jax.block_until_ready((o_bare_a, o_bare_b))
        diff_isolated = _max_abs_diff(o_bare_a[:, :T], o_bare_b[:, :T]) if T > 0 else 0.0

        tol = cfg["causal_tol"]
        if input_diff_before_T >= tol:
            verdict = "ВХОДЫ УЖЕ РАЗНЫЕ ДО T -- утечка выше кернела (Dense/embed), не в кернеле"
        elif diff_isolated < tol <= diff_inmodel:
            verdict = "INFRA: контекст компиляции (leak исчезает в изолированном jit)"
        elif diff_isolated >= tol:
            verdict = "MATH/VALUES: leak воспроизводится и на изолированном replay"
        else:
            verdict = "нет утечки на этом T ни там, ни там"

        results[T] = dict(
            input_diff_before_T=input_diff_before_T,
            diff_inmodel_kernel_output=diff_inmodel,
            diff_isolated_bare_replay=diff_isolated,
            verdict=verdict,
        )
        print(f"  T={T:4d}  input_diff<T={input_diff_before_T:.3e}  "
              f"in-model={diff_inmodel:.3e}  isolated-replay={diff_isolated:.3e}")
        print(f"           => {verdict}")

    _RESULTS["bs1"] = results
    _dump("bs1_replay_isolated_jit")

    del model, params, inter_a
    gc.collect()
    jax.clear_caches()
    return results


# ==========================================================================
# BS2: тот же replay, БЕЗ внешнего @jax.jit (eager dispatch)
# ==========================================================================
def test_bs2_replay_eager(cfg):
    print("\n" + "=" * 78)
    print("BS2: тот же replay БЕЗ внешнего @jax.jit (eager per-op dispatch)")
    print("=" * 78)

    config = KAGGLE_MEDIUM
    L, B = cfg["seq_len"], cfg["bsz"]
    num_layers = cfg["bisect_num_layers"]
    T_list = [cfg["bisect_T"], *cfg["extra_T_probe"]]

    model = _build_bisection_model(config, num_layers=num_layers)
    init_rng = jax.random.PRNGKey(12345)
    dummy = jnp.zeros((B, L), dtype=jnp.int32)
    params = model.init(init_rng, dummy)["params"]

    @jax.jit
    def forward_with_intermediates(p, ids):
        _logits, mutated = model.apply({"params": p}, ids, mutable=["intermediates"])
        return mutated["intermediates"]

    x_a = jax.random.randint(jax.random.PRNGKey(777), (B, L), 0, 256, dtype=jnp.int32)
    inter_a = forward_with_intermediates(params, x_a)
    jax.block_until_ready(inter_a)
    (q_a, k_a, v_a, w_a, b_a, g_a), _kout_a = _extract_kernel_io(inter_a)

    results = {}
    for T in T_list:
        x_b = x_a.at[:, T].set((x_a[:, T] + 1) % 256)
        inter_b = forward_with_intermediates(params, x_b)
        jax.block_until_ready(inter_b)
        (q_b, k_b, v_b, w_b, b_b, g_b), _kout_b = _extract_kernel_io(inter_b)

        # НЕТ @jax.jit -- прямой eager-вызов
        o_bare_a, _ = gdn2_pallas_forward_trainable(q_a, k_a, v_a, w_a, b_a, g_a, scale=1.0, config=config)
        o_bare_b, _ = gdn2_pallas_forward_trainable(q_b, k_b, v_b, w_b, b_b, g_b, scale=1.0, config=config)
        jax.block_until_ready((o_bare_a, o_bare_b))
        diff_eager = _max_abs_diff(o_bare_a[:, :T], o_bare_b[:, :T]) if T > 0 else 0.0

        results[T] = dict(diff_eager_replay=diff_eager)
        print(f"  T={T:4d}  eager-replay diff={diff_eager:.3e}")

    _RESULTS["bs2"] = results
    _dump("bs2_replay_eager")

    del model, params, inter_a
    gc.collect()
    jax.clear_caches()
    return results


# ==========================================================================
# BS3: decay_scale sweep на голом кернеле (закрывает дыру покрытия D4)
# ==========================================================================
def test_bs3_decay_scale_sweep(cfg):
    print("\n" + "=" * 78)
    print("BS3: decay_scale sweep на голом кернеле (D4 держал decay_scale=0.1")
    print("     фиксированным; модель даёт decay ~-0.018 при init -- на порядок меньше)")
    print("=" * 78)

    bsz, H, D = cfg["bsz"], cfg["H"], cfg["D"]
    L = cfg["seq_len"]
    results = {}

    for label, config in (("centered", KAGGLE_MEDIUM), ("nocenter", KAGGLE_MEDIUM_NOCENTER)):
        n_chunks = L // config.bt
        for decay_scale in cfg["decay_scale_sweep"]:
            key = jax.random.PRNGKey(31337)
            q, k, v, w, b, g, h0 = _make_bare_inputs(key, bsz, n_chunks, config.bt, H, D, decay_scale)

            @jax.jit
            def fwd(q_, k_, v_, w_, b_, g_):
                o, _hf = gdn2_pallas_forward_trainable(q_, k_, v_, w_, b_, g_, scale=1.0, h0=h0, config=config)
                return o

            o_a = fwd(q, k, v, w, b, g)
            jax.block_until_ready(o_a)

            boundaries = _boundary_positions(config, L)
            worst = 0.0
            n_fail = 0
            for T in boundaries:
                q_b = q.at[:, T].add(0.3)
                k_b = k.at[:, T].add(0.3)
                v_b = v.at[:, T].add(0.3)
                w_b = jnp.clip(w.at[:, T].add(0.1), 0.01, 1.0)
                b_b = jnp.clip(b.at[:, T].add(0.1), 0.01, 1.0)
                g_b = g.at[:, T].add(-0.05)
                o_b = fwd(q_b, k_b, v_b, w_b, b_b, g_b)
                jax.block_until_ready(o_b)
                diff = _max_abs_diff(o_a[:, :T], o_b[:, :T])
                worst = max(worst, diff)
                if diff >= cfg["causal_tol"]:
                    n_fail += 1

            tag = f"{label}/decay_scale={decay_scale}"
            results[tag] = dict(worst_diff=worst, n_failed=n_fail, n_total=len(boundaries))
            print(f"  {tag:32s} worst_diff={worst:.3e}  n_failed={n_fail}/{len(boundaries)}")

    _RESULTS["bs3"] = results
    _dump("bs3_decay_scale_sweep")

    print("\n  Интерпретация:")
    any_leak = any(r["n_failed"] > 0 for r in results.values())
    if any_leak:
        print("  => Утечка воспроизводится на ГОЛОМ кернеле при определённом decay_scale --")
        print("     гипотеза (B) подтверждена напрямую, без всякого replay. Смотрите, при")
        print("     каком именно decay_scale появляется n_failed>0 -- это порог, ниже")
        print("     которого centering-математика ломается.")
    else:
        print("  => Даже с decay_scale=0.001-0.018 голый кернель НЕ течёт. Гипотеза (B) в")
        print("     этой форме (просто малый decay) НЕ подтверждена -- ищите специфику")
        print("     именно в РЕАЛЬНЫХ значениях модели (не только decay), см. BS1/BS2.")
    return results


# ==========================================================================
# Entrypoint
# ==========================================================================
RUN_CONFIG = dict(
    bsz=2,
    H=6,
    D=128,
    seq_len=1024,
    causal_tol=1e-5,
    bisect_num_layers=1,          # H3 уже показал leak на N=1 -- минимальная,
                                   # самая чистая конфигурация для bisection
    bisect_T=127,                 # первая bc-граница -- та же точка, где D1
                                   # впервые зафиксировал leak на layer 0
    extra_T_probe=[255, 383, 1023],
    decay_scale_sweep=[0.1, 0.05, 0.018, 0.005, 0.001],
)


def main(cfg=RUN_CONFIG):
    print(f"JAX version: {jax.__version__}")
    print(f"Devices: {jax.devices()}")
    t0 = time.time()

    bs1 = test_bs1_replay_isolated_jit(cfg)
    bs2 = test_bs2_replay_eager(cfg)
    bs3 = test_bs3_decay_scale_sweep(cfg)

    elapsed = time.time() - t0
    _dump("FINAL_bisection_results")

    print("\n" + "=" * 78)
    print(f"BISECTION COMPLETE. Elapsed: {elapsed:.1f}s")
    print("=" * 78)
    print("\nСводный вердикт по T из BS1:")
    for T, r in bs1.items():
        print(f"  T={T:4d}: {r['verdict']}")

    print("\nПрочитайте BS2 (eager) рядом с BS1 (jit) для тех же T:")
    print("  Если BS1.isolated и BS2.eager СОВПАДАЮТ (оба ~0 либо оба текут) --")
    print("  jit/eager сам по себе не при чём, дело в ГРАНИЦЕ ГРАФА (одна операция")
    print("  vs узел внутри большого графа), а не в режиме компиляции как таковом.")

    print("\nBS3 закрывает или подтверждает decay-magnitude гипотезу независимо от replay.")
    print(f"\nВсе результаты: {OUT_DIR}/")


if __name__ == "__main__":
    main()

JAX version: 0.11.1
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]

BS1: replay реальных layer-0 входов кернела в ИЗОЛИРОВАННОМ @jax.jit
  [debug] top-level intermediates keys: ['block_0']
  T= 127  input_diff<T=0.000e+00  in-model=1.967e-06  isolated-replay=1.967e-06
           => нет утечки на этом T ни там, ни там
  T= 255  input_diff<T=0.000e+00  in-model=1.848e-06  isolated-replay=1.848e-06
           => нет утечки на этом T ни там, ни там
  T= 383  input_diff<T=0.000e+00  in-model=2.176e-06  isolat

In [19]:
"""
gdn2_bs3_extended_verify.py

Продолжение bisection-серии (gdn2_causal_leak_bisection_real_values.py).

ЗАЧЕМ ЭТОТ ФАЙЛ:
Оригинальный BS3 (decay_scale sweep на голом кернеле, decay_scale in
{0.1, 0.05, 0.018, 0.005, 0.001}) дал n_failed=0/7 на ВСЕХ 10
комбинациях (centered/nocenter x 5 decay_scale) -- т.е. вообще не
воспроизвёл утечку, которую H3/H4 видели прямо в модели (diff=0.0163
на num_layers=1, worst=0.088 на 8 слоях).

Это значит: либо (a) BS3 просто не покрыл нужную область входного
пространства (decay_scale -- не единственная переменная, которая
отличается между "голый синтетический кернель" и "реальная модель"),
либо (b) утечка вообще не воспроизводится на голом кернеле ни при
каких условиях, и вся причина -- в контексте (что должен бы был
показать BS1, но BS1 тоже дал 0 -- см. gdn2_bs4_replicate_h3.py).

Этот скрипт расширяет BS3 в сторону (a), закрывая конкретные дыры
покрытия оригинального grid'а:

  EXT1. GATE SATURATION. Оригинальный BS3 брал w/b (write/erase gate)
        из jax.random.uniform(0.2, 1.0) -- это ПЛОСКОЕ распределение,
        далёкое от того, что реально выдаёт sigmoid(Dense(x)) в модели
        после нескольких шагов обучения (gate'ы стремятся к 0 или 1,
        насыщаются). Здесь пробуем b/w near-saturated: {near 0, near 1,
        mixed} -- это единственная переменная, которую H3/H4 harness
        (реальная модель со sow) даёт, а синтетический BS3 -- нет.

  EXT2. ТОЧНАЯ РЕПЛИКА DECAY-ФОРМУЛЫ МОДЕЛИ. BS3 брал
        g = -abs(normal)*decay_scale -- НЕ то же самое распределение,
        что модель: g_model = -exp(a)*softplus(f), a~-4.0 (softplus
        асимметричен и НЕ обнуляется никогда, в отличие от abs(normal),
        которое может давать g~0 в отдельных позициях). Здесь строим
        g именно через exp(a)*softplus(f) с f~N(0,1) -- ближе к
        реальному forward.

  EXT3. МЕЛКИЙ GRID ВОКРУГ decay_scale=0.018 (a=-4.0 exact) + больше
        seeds (16 вместо 1) на каждую точку -- оригинальный BS3 гонял
        РОВНО один seed на комбинацию; если утечка seed-зависима (что
        WY-solve с плохо обусловленным Akk вполне может давать -- см.
        userMemories: "WY-solve conditioning vulnerability"), один seed
        мог просто промахнуться.

  EXT4. НЕНУЛЕВОЙ h0 БОЛЬШЕЙ МАГНИТУДЫ + non-trivial batch: оригинальный
        h0_nonzero=True давал h0 ~ N(0,1)*0.1 -- маленький. Пробуем
        {0.1, 1.0, 5.0} -- ближе к тому, что накопленный state может
        достигать после нескольких chunks в длинной последовательности.

Если EXT1-EXT4 ВСЕ ЕЩЁ дают n_failed=0 -- это сильно усиливает вывод,
что причина НЕ в значениях/математике голого кернеля, и вся тяжесть
доказательства переходит на gdn2_bs4_replicate_h3.py (контекст +
состояние весов после обучения).

Kaggle TPU v5e-8, notebook-only: без argparse, top-level константы, main().
"""
from __future__ import annotations

import gc
import json
import os
import time

import jax
import jax.numpy as jnp

from Atomic_ops.configs import KernelConfig, KAGGLE_MEDIUM, KAGGLE_MEDIUM_NOCENTER
from Atomic_ops.gdn2_pipeline import gdn2_pallas_forward_trainable

_RESULTS = {}
OUT_DIR = "./gdn2_bs3_extended_results"
os.makedirs(OUT_DIR, exist_ok=True)


def _dump(tag):
    path = os.path.join(OUT_DIR, f"{tag}.json")
    with open(path, "w") as f:
        json.dump(_RESULTS, f, indent=2, default=str)
    print(f"    [checkpoint written: {path}]")


def _max_abs_diff(a, b):
    return float(jnp.max(jnp.abs(jnp.asarray(a, jnp.float32) - jnp.asarray(b, jnp.float32))))


def _boundary_positions(config: KernelConfig, seq_len: int):
    pts = set()
    for base in range(0, seq_len, config.bt):
        if 0 < base < seq_len:
            pts.add(base)
    for base in range(0, seq_len, config.bc):
        if 0 < base < seq_len:
            pts.add(base)
    return sorted(pts)


def _make_inputs_ext(key, bsz, n_chunks, bt, H, D, decay_a, gate_mode, h0_scale):
    """Строит входы ближе к реальному forward модели, чем оригинальный
    BS3's _make_bare_inputs:
      - g через exp(a)*softplus(f), не abs(normal)*scale (EXT2)
      - w/b (gates) через режим насыщения, не плоский uniform (EXT1)
      - h0 масштабируемый (EXT4)
    """
    L = n_chunks * bt
    k1, k2, k3, k4, k5, k6 = jax.random.split(key, 6)
    shape = (bsz, L, H, D)

    q = jax.random.normal(k1, shape)
    k = jax.random.normal(k2, shape)
    q = q / (jnp.linalg.norm(q, axis=-1, keepdims=True) + 1e-6)
    k = k / (jnp.linalg.norm(k, axis=-1, keepdims=True) + 1e-6)
    v = jax.random.normal(k3, shape) * 0.5

    # EXT1: gate saturation modes
    if gate_mode == "uniform":
        w = jax.random.uniform(k4, shape, minval=0.2, maxval=1.0)
        b = jax.random.uniform(jax.random.fold_in(k4, 1), shape, minval=0.2, maxval=1.0)
    elif gate_mode == "near_zero":
        w = jax.nn.sigmoid(jax.random.normal(k4, shape) * 0.5 - 4.0)
        b = jax.nn.sigmoid(jax.random.normal(jax.random.fold_in(k4, 1), shape) * 0.5 - 4.0)
    elif gate_mode == "near_one":
        w = jax.nn.sigmoid(jax.random.normal(k4, shape) * 0.5 + 4.0)
        b = jax.nn.sigmoid(jax.random.normal(jax.random.fold_in(k4, 1), shape) * 0.5 + 4.0)
    elif gate_mode == "mixed_saturated":
        # половина позиций near-0, половина near-1 -- имитирует то, что
        # разные головы/позиции насыщаются в разные стороны после обучения
        raw = jax.random.normal(k4, shape)
        sign = jnp.sign(jax.random.normal(jax.random.fold_in(k4, 7), shape))
        w = jax.nn.sigmoid(raw * 0.5 + sign * 4.0)
        raw2 = jax.random.normal(jax.random.fold_in(k4, 1), shape)
        sign2 = jnp.sign(jax.random.normal(jax.random.fold_in(k4, 8), shape))
        b = jax.nn.sigmoid(raw2 * 0.5 + sign2 * 4.0)
    else:
        raise ValueError(gate_mode)

    # EXT2: decay точно как в модели -- g = -exp(a) * softplus(f)
    f = jax.random.normal(k5, shape)
    g = -jnp.exp(jnp.clip(decay_a, -20.0, 20.0)) * jax.nn.softplus(f)
    g = jnp.nan_to_num(g, nan=0.0, posinf=0.0, neginf=-20.0).astype(jnp.float32)

    # EXT4: scaled h0
    h0 = jax.random.normal(k6, (bsz, H, D, D)) * h0_scale

    return q, k, v, w, b, g, h0


def test_ext_sweep(cfg):
    print("\n" + "=" * 78)
    print("EXT: расширенный decay/gate/h0 sweep на голом кернеле")
    print("=" * 78)

    bsz, H, D = cfg["bsz"], cfg["H"], cfg["D"]
    L = cfg["seq_len"]
    results = {}

    configs = (("centered", KAGGLE_MEDIUM), ("nocenter", KAGGLE_MEDIUM_NOCENTER))

    for label, config in configs:
        n_chunks = L // config.bt
        for decay_a in cfg["decay_a_grid"]:
            for gate_mode in cfg["gate_modes"]:
                for h0_scale in cfg["h0_scales"]:
                    worst_over_seeds = 0.0
                    n_fail_total = 0
                    n_total = 0

                    for seed in range(cfg["n_seeds"]):
                        key = jax.random.PRNGKey(1000 + seed)
                        q, k, v, w, b, g, h0 = _make_inputs_ext(
                            key, bsz, n_chunks, config.bt, H, D, decay_a, gate_mode, h0_scale
                        )

                        @jax.jit
                        def fwd(q_, k_, v_, w_, b_, g_, h0_):
                            o, _hf = gdn2_pallas_forward_trainable(
                                q_, k_, v_, w_, b_, g_, scale=1.0, h0=h0_, config=config
                            )
                            return o

                        o_a = fwd(q, k, v, w, b, g, h0)
                        jax.block_until_ready(o_a)

                        boundaries = _boundary_positions(config, L)
                        for T in boundaries:
                            q_b = q.at[:, T].add(0.3)
                            k_b = k.at[:, T].add(0.3)
                            v_b = v.at[:, T].add(0.3)
                            w_b = jnp.clip(w.at[:, T].add(0.1), 1e-4, 1.0 - 1e-4)
                            b_b = jnp.clip(b.at[:, T].add(0.1), 1e-4, 1.0 - 1e-4)
                            g_b = g.at[:, T].add(-0.05)
                            o_b = fwd(q_b, k_b, v_b, w_b, b_b, g_b, h0)
                            jax.block_until_ready(o_b)
                            diff = _max_abs_diff(o_a[:, :T], o_b[:, :T])
                            worst_over_seeds = max(worst_over_seeds, diff)
                            n_total += 1
                            if diff >= cfg["causal_tol"]:
                                n_fail_total += 1

                    tag = f"{label}/a={decay_a}/gate={gate_mode}/h0={h0_scale}"
                    results[tag] = dict(
                        worst_diff=worst_over_seeds, n_failed=n_fail_total, n_total=n_total
                    )
                    flag = "  <<< LEAK" if n_fail_total > 0 else ""
                    print(f"  {tag:48s} worst={worst_over_seeds:.3e}  "
                          f"failed={n_fail_total}/{n_total}{flag}")

                    del q, k, v, w, b, g, h0, o_a
                    gc.collect()

    _RESULTS["ext_sweep"] = results
    _dump("ext_sweep_full")

    any_leak = any(r["n_failed"] > 0 for r in results.values())
    print("\n  Итог:")
    if any_leak:
        print("  => Утечка ВОСПРОИЗВЕДЕНА на голом кернеле в расширенном gate/decay/h0")
        print("     grid'е. Смотрите, при каком именно gate_mode/decay_a/h0_scale она")
        print("     появляется -- это даёт конкретную гипотезу для проверки формулы")
        print("     centering (см. _kernel_a_body / _kernel_b4_body в gdn2_fwd.py/gdn2_bwd.py).")
    else:
        print("  => Даже с насыщенными gate'ами, точной decay-формулой модели и большим h0")
        print("     голый кернель НЕ течёт. Гипотеза (B) отклонена во всех проверенных")
        print("     формах -- вся тяжесть доказательства переходит на BS4")
        print("     (gdn2_bs4_replicate_h3.py): контекст компиляции ИЛИ состояние весов")
        print("     после обучения (насыщение через ОБУЧЕНИЕ, а не искусственно).")
    return results


RUN_CONFIG = dict(
    bsz=2,
    H=6,
    D=128,
    seq_len=1024,
    causal_tol=1e-5,
    n_seeds=16,                                   # EXT3
    decay_a_grid=[-4.0, -3.0, -2.0, -1.0],         # EXT3: вокруг реального init a=-4.0
    gate_modes=["uniform", "near_zero", "near_one", "mixed_saturated"],  # EXT1
    h0_scales=[0.1, 1.0, 5.0],                     # EXT4
)


def main(cfg=RUN_CONFIG):
    print(f"JAX version: {jax.__version__}")
    print(f"Devices: {jax.devices()}")
    t0 = time.time()

    test_ext_sweep(cfg)

    elapsed = time.time() - t0
    _dump("FINAL_ext_results")
    print(f"\nEXT SWEEP COMPLETE. Elapsed: {elapsed:.1f}s")
    print(f"Все результаты: {OUT_DIR}/")


if __name__ == "__main__":
    main()

JAX version: 0.11.1
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]

EXT: расширенный decay/gate/h0 sweep на голом кернеле
  centered/a=-4.0/gate=uniform/h0=0.1              worst=3.085e-05  failed=110/112  <<< LEAK
  centered/a=-4.0/gate=uniform/h0=1.0              worst=3.073e-05  failed=110/112  <<< LEAK


KeyboardInterrupt: 

In [23]:
"""
gdn2_bs4_replicate_h3.py

Продолжение bisection-серии. Читать после
gdn2_causal_leak_bisection_real_values.py (BS1-BS3) и его результатов:

  BS1 (isolated jit replay реальных layer-0 входов модели): NO LEAK
      на T in {127, 255, 383, 1023}, diff ~ 1e-6 везде.
  BS2 (тот же replay, eager): совпадает с BS1 -- NO LEAK.
  BS3 (decay_scale sweep {0.1..0.001} на голом кернеле): NO LEAK,
      0/70 отказов.

Это ПРОТИВОРЕЧИТ H3/H4, которые на, по-видимому, той же архитектуре
видели diff=0.0163 на num_layers=1 и worst=0.088 на 8 слоях, строго
специфично к use_centering=True.

ВЫВОД ИЗ ЭТОГО ПРОТИВОРЕЧИЯ (см. PROJECT_STATE-style рассуждение):
раз BS1-BS3 не воспроизвели даже BASELINE утечку из H3/H4, значит
_build_bisection_model (fresh init, jax.random.PRNGKey(12345)) НЕ
эквивалентен тому, на чём H3/H4 реально наблюдали leak. Нельзя делать
никаких выводов "leak исчезает в изолированном jit" или "decay_scale
не виноват", пока сам harness не воспроизводит явление, которое
изначально пытались изолировать. Это чисто методологическая дыра, и
закрывать её нужно ПЕРЕД тем, как городить новые гипотезы поверх
неподтверждённого противоречия.

Этот скрипт делает две вещи по отдельности:

  BS4a. ТОЧНАЯ РЕПЛИКА H3. Прогоняет ИДЕНТИЧНЫЙ (по числу слоёв,
        perturbation-методу через input_ids, tolerance) сетап на fresh
        init и ищет: воспроизводится ли diff~0.016 хоть на КАКОМ-то
        seed/T, если расширить sweep за пределы {127,255,383,1023}
        (H3 мог упасть на другом T или другом seed инициализации,
        которые BS1 не пробовал -- BS1 использовал только
        PRNGKey(12345) для init и PRNGKey(777) для входа).

  BS4b. ОБУЧЕННЫЕ ВЕСА. Если BS4a тоже даёт 0 -- прогоняет N шагов
        простого обучения (MSE к случайным таргетам, тот же паттерн,
        что дефолтный training loop) на bisection-модели, ЗАТЕМ
        повторяет BS1-style isolated-vs-in-model replay на весах ПОСЛЕ
        обучения, на нескольких чекпоинтах шагов (0, 50, 200, 500).
        Гипотеза: leak требует НАСЫЩЕННЫХ gate'ов / смещённого decay_a,
        которые возникают только через реальный градиентный спуск
        (не через искусственное forcing значений, как в
        gdn2_bs3_extended_verify.py -- то было "какие значения могут
        воспроизвести leak на ГОЛОМ кернеле"; это "воспроизводится ли
        leak, когда веса реально обучены, ВНУТРИ модели").

Если BS4a находит leak -- дело было в недостаточном sweep (T/seed), и
дальше нужно просто расширить BS1's T-grid и повторить isolated-replay
на найденной точке (дешёвая доработка, не новый скрипт).

Если BS4a чист, а BS4b (какой-то checkpoint шагов > 0) ловит leak --
тогда H3/H4 наблюдали эффект ИМЕННО обученных активаций, и дальше
нужен BS1-style isolated replay reference НА ЭТОМ ЧЕКПОИНТЕ, чтобы
разделить "контекст компиляции" vs "значения" уже на подтверждённом
воспроизводимом случае.

Kaggle TPU v5e-8, notebook-only: без argparse, top-level константы, main().
"""
from __future__ import annotations

import gc
import json
import os
import time

import jax
import jax.numpy as jnp
import flax.linen as nn
import optax

from Atomic_ops.configs import KAGGLE_MEDIUM, KAGGLE_MEDIUM_NOCENTER
from Atomic_ops.gdn2_pipeline import gdn2_pallas_forward_trainable

_RESULTS = {}
OUT_DIR = "./gdn2_bs4_replicate_h3_results"
os.makedirs(OUT_DIR, exist_ok=True)


def _dump(tag):
    path = os.path.join(OUT_DIR, f"{tag}.json")
    with open(path, "w") as f:
        json.dump(_RESULTS, f, indent=2, default=str)
    print(f"    [checkpoint written: {path}]")


def _max_abs_diff(a, b):
    return float(jnp.max(jnp.abs(jnp.asarray(a, jnp.float32) - jnp.asarray(b, jnp.float32))))


# ==========================================================================
# Модель -- идентична gdn2_causal_leak_bisection_real_values.py's
# _build_bisection_model, с добавлением: конфигурируемый use_centering
# (через kernel_config), и сохранением params для возможности обучения.
# ==========================================================================
def _build_model(kernel_config, num_layers, d_model=768, n_heads=6):
    d_head = d_model // n_heads
    assert d_head == 128

    def _safe_normalize(t, eps=1e-6):
        return t * jax.lax.rsqrt(jnp.sum(t * t, axis=-1, keepdims=True) + eps ** 2)

    def _sanitize(t):
        return jnp.nan_to_num(jnp.clip(t, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)

    class Mixer(nn.Module):
        layer_idx: int

        @nn.compact
        def __call__(self, x):
            b, l, d = x.shape
            q_lin = nn.Dense(d, use_bias=False, name="q_proj", dtype=jnp.bfloat16)(x)
            k_lin = nn.Dense(d, use_bias=False, name="k_proj", dtype=jnp.bfloat16)(x)
            v_lin = nn.Dense(d, use_bias=False, name="v_proj", dtype=jnp.bfloat16)(x)
            q = jax.nn.silu(q_lin).reshape(b, l, n_heads, d_head).astype(jnp.float32)
            k = jax.nn.silu(k_lin).reshape(b, l, n_heads, d_head).astype(jnp.float32)
            v = jax.nn.silu(v_lin).reshape(b, l, n_heads, d_head).astype(jnp.float32)
            v = jnp.clip(v, -50.0, 50.0)
            q = _safe_normalize(q)
            k = _safe_normalize(k)

            b_gate = jax.nn.sigmoid(nn.Dense(d, name="erase_gate", dtype=jnp.bfloat16)(x)) \
                .reshape(b, l, n_heads, d_head).astype(jnp.float32)
            w_gate = jax.nn.sigmoid(nn.Dense(d, name="write_gate", dtype=jnp.bfloat16)(x)) \
                .reshape(b, l, n_heads, d_head).astype(jnp.float32)

            a_param = self.param("decay_a", nn.initializers.constant(-4.0), (n_heads,)).astype(jnp.float32)
            f_proj = nn.Dense(d, name="decay_proj", dtype=jnp.bfloat16)(x).reshape(b, l, n_heads, d_head)
            a_safe = jnp.clip(a_param, -20.0, 20.0)
            g = -jnp.exp(a_safe)[None, None, :, None] * jax.nn.softplus(f_proj.astype(jnp.float32))
            g = jnp.nan_to_num(g, nan=0.0, posinf=0.0, neginf=-20.0)

            out_gate = jnp.clip(nn.Dense(d, use_bias=False, name="out_gate", dtype=jnp.bfloat16)(x), -1e2, 1e2)

            q, k, v, w_gate, b_gate, g = map(_sanitize, (q, k, v, w_gate, b_gate, g))

            if self.layer_idx == 0:
                self.sow("intermediates", "kin_q_0", q)
                self.sow("intermediates", "kin_k_0", k)
                self.sow("intermediates", "kin_v_0", v)
                self.sow("intermediates", "kin_w_0", w_gate)
                self.sow("intermediates", "kin_b_0", b_gate)
                self.sow("intermediates", "kin_g_0", g)
                self.sow("intermediates", "gate_w_stats", (jnp.mean(w_gate), jnp.std(w_gate)))
                self.sow("intermediates", "gate_b_stats", (jnp.mean(b_gate), jnp.std(b_gate)))
                self.sow("intermediates", "decay_a_val", a_param)

            out, _hf = gdn2_pallas_forward_trainable(
                q, k, v, w_gate, b_gate, g, scale=1.0, config=kernel_config
            )

            if self.layer_idx == 0:
                self.sow("intermediates", "kout_raw_0", out)

            out = out.reshape(b, l, d)
            out = nn.RMSNorm(epsilon=1e-6, name="mixer_out_norm")(out).astype(x.dtype)
            return nn.Dense(d, use_bias=False, name="out_proj", dtype=jnp.bfloat16)(out * jax.nn.silu(out_gate))

    class MLP(nn.Module):
        @nn.compact
        def __call__(self, x):
            d = x.shape[-1]
            h = 4 * d
            gate = nn.Dense(h, use_bias=False, name="gate_proj", dtype=jnp.bfloat16)(x)
            up = nn.Dense(h, use_bias=False, name="up_proj", dtype=jnp.bfloat16)(x)
            act = jax.nn.silu(gate) * up
            return nn.Dense(d, use_bias=False, name="down_proj", dtype=jnp.bfloat16)(act)

    class Block(nn.Module):
        layer_idx: int

        @nn.compact
        def __call__(self, x):
            h = Mixer(layer_idx=self.layer_idx, name="mixer")(nn.RMSNorm(epsilon=1e-6, name="mixer_norm")(x))
            x = jnp.nan_to_num(jnp.clip(x + h, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)
            m = MLP(name="mlp")(nn.RMSNorm(epsilon=1e-6, name="mlp_norm")(x))
            x = jnp.nan_to_num(jnp.clip(x + m, -1e3, 1e3), nan=0.0, posinf=1e3, neginf=-1e3)
            return x

    class LM(nn.Module):
        @nn.compact
        def __call__(self, input_ids):
            embed = nn.Embed(num_embeddings=256, features=d_model, name="embed", dtype=jnp.bfloat16)
            x = embed(input_ids)
            for i in range(num_layers):
                x = Block(layer_idx=i, name=f"block_{i}")(x)
            x = nn.RMSNorm(epsilon=1e-6, name="final_norm")(x).astype(x.dtype)
            return embed.attend(x)

    return LM()


def _find_and_unwrap(intermediates, target_key):
    def _walk(node):
        if isinstance(node, dict):
            if target_key in node:
                return node[target_key]
            for v in node.values():
                found = _walk(v)
                if found is not None:
                    return found
        return None

    found = _walk(intermediates)
    if found is None:
        raise KeyError(f"sow key {target_key!r} не найден (top-level keys: {list(intermediates.keys())})")
    while isinstance(found, (list, tuple)):
        found = found[0]
    return found


def _extract_kernel_io(inter):
    names = ("q", "k", "v", "w", "b", "g")
    vals = tuple(_find_and_unwrap(inter, f"kin_{n}_0") for n in names)
    kout = _find_and_unwrap(inter, "kout_raw_0")
    return vals, kout


def _extract_gate_stats(inter):
    def _find_raw(intermediates, target_key):
        def _walk(node):
            if isinstance(node, dict):
                if target_key in node:
                    return node[target_key]
                for v in node.values():
                    found = _walk(v)
                    if found is not None:
                        return found
            return None
        found = _walk(intermediates)
        if found is None:
            raise KeyError(f"sow key {target_key!r} не найден")
        # разматываем ТОЛЬКО flax-обёртку (list длины 1), НЕ tuple
        while isinstance(found, list) and len(found) == 1:
            found = found[0]
        return found

    w_stats = _find_raw(inter, "gate_w_stats")
    b_stats = _find_raw(inter, "gate_b_stats")
    wm, ws = w_stats
    bm, bs = b_stats
    a_val = _find_and_unwrap(inter, "decay_a_val")
    return dict(
        w_mean=float(wm), w_std=float(ws),
        b_mean=float(bm), b_std=float(bs),
        decay_a=[float(x) for x in a_val],
    )

def _boundary_positions_L(seq_len, bt, bc):
    pts = set()
    for base in range(0, seq_len, bt):
        if 0 < base < seq_len:
            pts.add(base)
    for base in range(0, seq_len, bc):
        if 0 < base < seq_len:
            pts.add(base)
    return sorted(pts)


# ==========================================================================
# BS4a: точная реплика H3, но с расширенным sweep по seed/T
# ==========================================================================
def test_bs4a_wide_replica(cfg):
    print("\n" + "=" * 78)
    print("BS4a: реплика H3 (num_layers=1, centered) -- расширенный seed/T sweep")
    print("      (H3 наблюдал diff=0.0163 -- проверяем, воспроизводится ли это")
    print("      на КАКОМ-либо seed/T, раз BS1 (один seed, 4 T) дал 0)")
    print("=" * 78)

    config = KAGGLE_MEDIUM
    L, B = cfg["seq_len"], cfg["bsz"]
    num_layers = 1
    T_list = _boundary_positions_L(L, config.bt, config.bc)

    model = _build_model(config, num_layers=num_layers)

    @jax.jit
    def forward_with_intermediates(p, ids):
        _logits, mutated = model.apply({"params": p}, ids, mutable=["intermediates"])
        return mutated["intermediates"]

    results = {}
    worst_seen = 0.0
    worst_tag = None

    for init_seed in cfg["init_seeds"]:
        dummy = jnp.zeros((B, L), dtype=jnp.int32)
        params = model.init(jax.random.PRNGKey(init_seed), dummy)["params"]

        for input_seed in cfg["input_seeds"]:
            x_a = jax.random.randint(jax.random.PRNGKey(input_seed), (B, L), 0, 256, dtype=jnp.int32)
           
            inter_a = forward_with_intermediates(params, x_a)
            jax.block_until_ready(inter_a)
            _, kout_a = _extract_kernel_io(inter_a)

            for T in T_list:
                # --- null control: no-op update ---
                x_null = x_a.at[:, T].set(x_a[:, T])
                inter_null = forward_with_intermediates(params, x_null)
                jax.block_until_ready(inter_null)
                kout_null = _extract_kernel_io(inter_null)[1]
                diff_null = _max_abs_diff(kout_a[:, :T], kout_null[:, :T]) if T > 0 else 0.0
                if diff_null > 0:
                    print(f"  [ARTIFACT] T={T}: no-op set() → diff={diff_null:.3e} на [:T]")

                # --- perturbation ---
                x_b = x_a.at[:, T].set((x_a[:, T] + 1) % 256)
                inter_b = forward_with_intermediates(params, x_b)
                jax.block_until_ready(inter_b)
                kout_b = _extract_kernel_io(inter_b)[1]

                # --- positive control: должно меняться СТРОГО ПОСЛЕ T ---
                if T + 1 < L:
                    diff_after = _max_abs_diff(kout_a[:, T+1:], kout_b[:, T+1:])
                    if diff_after == 0.0:
                        print(f"  [BROKEN] T={T}: perturbation не влияет на [T+1:] — harness мёртв")

                # --- собственно тест: не должно меняться ДО T ---
                diff_before = _max_abs_diff(kout_a[:, :T], kout_b[:, :T]) if T > 0 else 0.0
                tag = f"init={init_seed}/input={input_seed}/T={T}"
                if diff_before >= cfg["causal_tol"]:
                    results[tag] = diff_before
                    print(f"  LEAK  {tag:40s} diff={diff_before:.3e}")
                if diff_before > worst_seen:
                    worst_seen = diff_before
                    worst_tag = tag

            del inter_a
        del params
        gc.collect()

    print(f"\n  Worst overall: {worst_tag} diff={worst_seen:.3e}")
    _RESULTS["bs4a"] = dict(leaks=results, worst_seen=worst_seen, worst_tag=worst_tag)
    _dump("bs4a_wide_replica")

    if not results:
        print("  => НИ ОДНА комбинация init_seed x input_seed x T не дала leak >= tol.")
        print("     BS4a НЕ воспроизвёл H3's baseline -- переходим к BS4b (обученные веса).")
    else:
        print("  => Leak воспроизведён при определённых seed/T -- H3 был чувствителен к")
        print("     инициализации, не к структурному отличию 'модель vs голый кернель'.")
        print("     Возьмите worst_tag и прогоните BS1-style isolated replay ИМЕННО на нём.")
    return results


# ==========================================================================
# BS4b: обучить веса, затем повторить BS1-style replay на нескольких
# чекпоинтах шагов обучения
# ==========================================================================
def test_bs4b_trained_weights(cfg):
    print("\n" + "=" * 78)
    print("BS4b: обучаем bisection-модель N шагов, ищем leak на каждом checkpoint'e")
    print("      (гипотеза: leak требует насыщенных gate/decay, возникающих через")
    print("      реальное обучение -- не воспроизводимых искусственным forcing)")
    print("=" * 78)

    config = KAGGLE_MEDIUM
    L, B = cfg["seq_len"], cfg["bsz"]
    num_layers = cfg["train_num_layers"]
    T_list = [cfg["bisect_T"], *cfg["extra_T_probe"]]

    model = _build_model(config, num_layers=num_layers)
    dummy = jnp.zeros((B, L), dtype=jnp.int32)
    params = model.init(jax.random.PRNGKey(12345), dummy)["params"]

    tx = optax.adamw(learning_rate=cfg["lr"])
    opt_state = tx.init(params)

    rng = jax.random.PRNGKey(0)

    @jax.jit
    def train_step(params, opt_state, rng):
        rng, sub_x, sub_y = jax.random.split(rng, 3)
        ids = jax.random.randint(sub_x, (B, L), 0, 256, dtype=jnp.int32)
        targets = jax.random.randint(sub_y, (B, L), 0, 256, dtype=jnp.int32)

        def loss_fn(p):
            logits = model.apply({"params": p}, ids)
            logp = jax.nn.log_softmax(logits.astype(jnp.float32), axis=-1)
            onehot = jax.nn.one_hot(targets, 256)
            loss = -jnp.mean(jnp.sum(onehot * logp, axis=-1))
            return loss

        loss, grads = jax.value_and_grad(loss_fn)(params)
        updates, opt_state = tx.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state, rng, loss

    @jax.jit
    def forward_with_intermediates(p, ids):
        _logits, mutated = model.apply({"params": p}, ids, mutable=["intermediates"])
        return mutated["intermediates"]

    @jax.jit
    def bare_fwd(q_, k_, v_, w_, b_, g_):
        o, _hf = gdn2_pallas_forward_trainable(q_, k_, v_, w_, b_, g_, scale=1.0, config=config)
        return o

    def _run_replay_at_checkpoint(step_tag, params):
        x_a = jax.random.randint(jax.random.PRNGKey(777), (B, L), 0, 256, dtype=jnp.int32)
        inter_a = forward_with_intermediates(params, x_a)
        jax.block_until_ready(inter_a)
        gate_stats = _extract_gate_stats(inter_a)
        print(f"    gate/decay stats @ {step_tag}: {gate_stats}")

        (q_a, k_a, v_a, w_a, b_a, g_a), kout_a = _extract_kernel_io(inter_a)

        step_results = {"gate_stats": gate_stats, "per_T": {}}
        for T in T_list:
            x_b = x_a.at[:, T].set((x_a[:, T] + 1) % 256)
            inter_b = forward_with_intermediates(params, x_b)
            jax.block_until_ready(inter_b)
            (q_b, k_b, v_b, w_b, b_b, g_b), kout_b = _extract_kernel_io(inter_b)

            diff_inmodel = _max_abs_diff(kout_a[:, :T], kout_b[:, :T]) if T > 0 else 0.0

            o_bare_a = bare_fwd(q_a, k_a, v_a, w_a, b_a, g_a)
            o_bare_b = bare_fwd(q_b, k_b, v_b, w_b, b_b, g_b)
            jax.block_until_ready((o_bare_a, o_bare_b))
            diff_isolated = _max_abs_diff(o_bare_a[:, :T], o_bare_b[:, :T]) if T > 0 else 0.0

            tol = cfg["causal_tol"]
            if diff_isolated < tol <= diff_inmodel:
                verdict = "INFRA: контекст компиляции"
            elif diff_isolated >= tol:
                verdict = "MATH/VALUES: воспроизводится и изолированно"
            else:
                verdict = "нет утечки"

            step_results["per_T"][T] = dict(
                diff_inmodel=diff_inmodel, diff_isolated=diff_isolated, verdict=verdict
            )
            flag = "  <<<" if diff_inmodel >= tol or diff_isolated >= tol else ""
            print(f"    T={T:4d}  in-model={diff_inmodel:.3e}  isolated={diff_isolated:.3e}  "
                  f"{verdict}{flag}")

        del inter_a
        return step_results

    all_step_results = {}
    step = 0
    for target_step in cfg["checkpoint_steps"]:
        while step < target_step:
            params, opt_state, rng, loss = train_step(params, opt_state, rng)
            step += 1
        jax.block_until_ready(params)
        print(f"\n  --- checkpoint @ step={step} (loss={float(loss) if step > 0 else float('nan'):.4f}) ---")
        all_step_results[step] = _run_replay_at_checkpoint(f"step={step}", params)
        gc.collect()

    _RESULTS["bs4b"] = all_step_results
    _dump("bs4b_trained_weights")

    any_leak = any(
        r["verdict"] != "нет утечки"
        for step_res in all_step_results.values()
        for r in step_res["per_T"].values()
    )
    print("\n  Итог BS4b:")
    if any_leak:
        print("  => Leak появляется на КАКОМ-ТО checkpoint'е обучения. Смотрите gate_stats")
        print("     на этом шаге -- это даёт конкретное числовое условие (насыщение gate,")
        print("     смещение decay_a) для воспроизведения на синтетике / для фикса formulas.")
    else:
        print("  => Даже после обучения leak не появляется в этом сетапе. Тогда H3/H4,")
        print("     видимо, использовали другую архитектуру/конфиг (не эту bisection-модель) --")
        print("     нужно точно свериться с кодом H3/H4 (какая модель, сколько шагов,")
        print("     какой d_model/n_heads/num_layers) и воспроизвести ИМЕННО его, а не")
        print("     предполагаемую копию.")
    return all_step_results


RUN_CONFIG = dict(
    bsz=2,
    H=6,
    D=128,
    seq_len=1024,
    causal_tol=1e-5,
    # BS4a
    init_seeds=[12345, 1, 2, 3],
    input_seeds=[777, 42, 1000],
    # BS4b
    train_num_layers=1,
    lr=3e-4,
    checkpoint_steps=[0, 50, 200, 500],
    bisect_T=127,
    extra_T_probe=[255, 383, 1023],
)


def main(cfg=RUN_CONFIG):
    print(f"JAX version: {jax.__version__}")
    print(f"Devices: {jax.devices()}")
    t0 = time.time()

    bs4a_leaks = test_bs4a_wide_replica(cfg)
    # Только если BS4a не нашёл ничего -- есть смысл платить за BS4b
    # (обучение дороже по времени). Если хотите BS4b безусловно, просто
    # закомментируйте условие ниже.
    if not bs4a_leaks:
        test_bs4b_trained_weights(cfg)
    else:
        print("\n  [BS4b пропущен: BS4a уже нашёл воспроизводимый leak на fresh init --")
        print("   разберитесь с ним в изоляции прежде, чем платить за обучение.]")

    elapsed = time.time() - t0
    _dump("FINAL_bs4_results")
    print(f"\nBS4 COMPLETE. Elapsed: {elapsed:.1f}s")
    print(f"Все результаты: {OUT_DIR}/")


if __name__ == "__main__":
    main()

JAX version: 0.11.1
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]

BS4a: реплика H3 (num_layers=1, centered) -- расширенный seed/T sweep
      (H3 наблюдал diff=0.0163 -- проверяем, воспроизводится ли это
      на КАКОМ-либо seed/T, раз BS1 (один seed, 4 T) дал 0)

  Worst overall: None diff=0.000e+00
    [checkpoint written: ./gdn2_bs4_replicate_h3_results/bs4a_wide_replica.json]
  => НИ ОДНА комбинация init_seed x input_seed x T не дала leak >= tol.
     BS4a НЕ воспроизвёл H3's baseline -- переходим 

ValueError: not enough values to unpack (expected 2, got 1)